# FloodTwin: forecast-informed extreme-value relief-inventory allocation
### occurrence (calibrated machine learning) + severity (extreme value theory) + convex bi-objective allocation

Code accompanying: **A. Gupta and K. Dutta Choudhury, "A forecast-informed extreme-value model for stochastic relief-inventory allocation under flood risk, with special reference to a real flood event"**.

**Input data (not included in this repository).** The raw India Meteorological Department (IMD) daily gridded files for **1951-2025**, in three folders inside `BASE_DIR` (set in Step 2):

| Folder | Files |
|---|---|
| `Rainfall/` | `Rainfall_ind<year>_rfp25.grd` (0.25 deg rainfall) |
| `Max_temp/` | `Maxtemp_MaxT_<year>.GRD` (1 deg maximum temperature) |
| `Min_temp/` | `Mintemp_MinT_<year>.GRD` (1 deg minimum temperature) |

The files can be downloaded from the IMD Pune data portal (https://www.imdpune.gov.in); see `data/README.md`.

**What the notebook creates.** District daily series, models, figures, result files and the dashboard are written to `OUT_DIR` (Step 2). Step 10j copies the files needed for the repository into `OUT_DIR/final_output`. All numbers reported in the paper are taken from `OUT_DIR/saved/final_results.json` (written in Step 10d and extended in Steps 10e-10i).

**How to run.** Set `BASE_DIR` in Step 2, then *Run all* with `QUICK_TEST = False`. Step 2 checks that all 225 raw files exist before anything else runs. The notebook runs on Google Colab (Drive is mounted automatically) or in a local Jupyter environment.

**Applying the model to another region.** Edit only the `REGION` settings block in Step 2 (district bounding boxes, population and exposure, warehouses and road distances, kit composition, budget, threshold and decision days).

**Structure and link to the paper.**

| Step | Content | Paper |
|---|---|---|
| 1-4 | Libraries, settings, grid extraction, district daily series | Sections 5.1, 6.1 |
| 5 | Reference regressors (not used by the decision model) | -- |
| 6-7 | Occurrence classifier: purged expanding-window tuning, Platt calibration, 2021-2025 test | Sections 5.2, 6.3; Tables 5-6; Fig. 2 |
| 8 | GPD severity model and diagnostics | Sections 5.3, 6.4; Table 7; Fig. 3 |
| 9 | Decision model and solvers (exact conic, SLSQP, PSO) with numerical self-checks | Sections 4, 5.4 |
| 10 | Case study of 18 June 2022: inputs, exact Pareto frontier, PSO benchmark | Sections 6.5-6.6, 6.10; Tables 8-10; Fig. 4 |
| 10b | Lead times L = 1-5 and re-issued forecast trajectory | Section 7.3; Table 16; Fig. 5 |
| 10c | Robustness: costs, floors, EVT threshold, bootstrap, one-at-a-time | Sections 7.1-7.2; Tables 14-15 |
| 10d | Reactive decision of 19 June 2022, frozen results | Section 6.9; Table 13 |
| 10e | Regime map and value of the severity layer | Sections 6.7-6.8; Tables 11-12; Fig. 6 |
| 10f-10g | Demand scale and demand map | Section 7.4; Tables 18-19 |
| 10h | Controlled experiment on the tail shape | Section 7.5; Table 20; Fig. 8 |
| 10i | Demand source and retrospective evaluation | Sections 7.4, 8; Table 17 |
| 10j | Collect the final outputs for upload | -- |
| 11-13 | Decision-support dashboard | Section 9.3; Fig. 7 |


## Step 1: Install required libraries

In [ ]:
# Step 1: install the packages used by the notebook (Google Colab).
# cvxpy >= 1.5 provides the exact power cones of the conic reformulation (Proposition 7).

import warnings; warnings.filterwarnings('ignore')
!pip install -q numpy pandas scikit-learn scipy xgboost lightgbm streamlit pyngrok folium plotly streamlit-folium
!pip install -q -U cvxpy clarabel
print('Libraries installed.')

## Step 2: Set paths and the region/date configuration

In [ ]:
# =============================================================================
# Step 2: paths, region settings and method settings
# =============================================================================
import os

# ---- Paths (edit if your folders differ) ----------------------------------------
# On Google Colab, Google Drive is mounted and BASE_DIR is a folder on Drive.
# In a local Jupyter session, BASE_DIR is the folder that contains this notebook.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/ISRO_Digital_Twin'
except ImportError:
    BASE_DIR = os.path.abspath('.')

# Raw IMD daily grids, 1951-2025 (read only; see data/README.md)
RAIN_DIR    = os.path.join(BASE_DIR, 'Rainfall')
MAXTEMP_DIR = os.path.join(BASE_DIR, 'Max_temp')
MINTEMP_DIR = os.path.join(BASE_DIR, 'Min_temp')

# Everything the notebook creates is written to OUT_DIR (= APP_DIR).
# Step 10j copies the files needed for the repository into OUT_DIR/final_output.
OUT_DIR = os.path.join(BASE_DIR, 'FloodTwin_outputs')
APP_DIR = OUT_DIR
FORCE_REEXTRACT = False           # True: rebuild the district series from the raw grids even if a valid cache exists

os.makedirs(f'{APP_DIR}/data', exist_ok=True)
os.makedirs(f'{APP_DIR}/saved', exist_ok=True)

# =============================================================================
# REGION SETTINGS: the only block to edit to apply the model to another region
# =============================================================================
REGION = {
    'name': 'Barak Valley, Assam (India)',
    'currency': 'Rs',
    # 1. Districts: bounding boxes (degrees) on the 0.25-degree rainfall grid
    'districts': {
        'Cachar':      {'LAT_MIN': 24.40, 'LAT_MAX': 25.10, 'LON_MIN': 92.45, 'LON_MAX': 93.15},
        'Karimganj':   {'LAT_MIN': 24.10, 'LAT_MAX': 24.90, 'LON_MIN': 92.15, 'LON_MAX': 92.70},
        'Hailakandi':  {'LAT_MIN': 24.30, 'LAT_MAX': 24.90, 'LON_MIN': 92.30, 'LON_MAX': 92.90},
    },
    # 2. Exposure
    'population': {'Cachar': 1_736_617, 'Karimganj': 1_228_686, 'Hailakandi': 659_296},   # Census of India 2011
    'household_size': 5,                                                                   # Census of India 2011
    # Population documented as affected in 2022 (ASDMA flood memorandum, Table 2.5). Known only after
    # the event; used for comparison (Step 10i), not for the proactive decision.
    'affected_population': {'Cachar': 1434750, 'Karimganj': 308581, 'Hailakandi': 256019},
    # Baseline exposure, known before the event: flood-prone area in the NRSC/ISRO Flood Hazard Atlas of
    # Assam (inundation frequency 1998-2015). Classes: Very Low = flooded 1-4 of 18 years, Low 5-8,
    # Moderate 9-12, High 13-15, Very High 16-18.
    'demand_source': 'hazard_atlas',     # 'hazard_atlas' (ex-ante) | 'affected_population' (ex-post) | 'uniform'
    'hazard_classes_at_risk': ['Very Low', 'Low', 'Moderate', 'High', 'Very High'],
    'flood_hazard_area_ha': {
        'Cachar':     {'Very Low': 37101, 'Low': 20229, 'Moderate': 17458, 'High': 14934, 'Very High': 11419},
        'Karimganj':  {'Very Low': 23257, 'Low': 10193, 'Moderate': 12263, 'High': 10588, 'Very High': 2613},
        'Hailakandi': {'Very Low': 17893, 'Low': 6082,  'Moderate': 5239,  'High': 3337,  'Very High': 1447},
    },
    'flood_hazard_villages': {'Cachar': 628, 'Karimganj': 558, 'Hailakandi': 218},
    'district_area_ha': {'Cachar': 378600, 'Karimganj': 180900, 'Hailakandi': 132700},
    # 3. Network: warehouse capacities (tonnes), road distances district -> warehouse (km, in the order of
    #    'warehouses'), unusable routes and road freight rate (Rs per tonne-km)
    'warehouses': [{'name': 'WH1_Silchar_Cachar', 'capacity_t': 5905},     # Assam State Warehousing Corporation
                   {'name': 'WH2_Karimganj',      'capacity_t': 2450}],
    'distance_km': {'Cachar': [5, 50], 'Karimganj': [50, 5], 'Hailakandi': [40, 50]},
    'disallowed_routes': [('Hailakandi', 'WH2_Karimganj')],   # Hailakandi has no warehouse and is served from Silchar
    'freight_rate_per_tonne_km': 3.78,                         # national road freight rate
    # 4. Relief kit per household: clothing and utensils allowances, and gratuitous relief for two adults
    #    at the MGNREGA daily wage for seven days (SDRF norms)
    'kit': {'weight_kg': 50, 'clothing': 2500, 'utensils': 2500,
            'food_daily_wage': 256, 'adults_covered': 2, 'relief_days': 7},
    # 5. Budget: annual State Disaster Response Fund and the population it serves (Assam, Census 2011);
    #    the case-study budget is the valley's population share of this fund
    'budget': {'annual_fund_crore': 920, 'fund_population': 31_205_576},
    'holding_rate_per_month': 0.015,     # share of kit value per month
    'shelf_life_days': 180,              # shelf life of the food items
    # 6. Hazard definition and decision days
    'hazard_threshold_mm': 64.5,         # IMD "heavy rain"
    'hazard_consecutive_days': 2,        # alert = two consecutive days at or above the threshold
    'decision_date': '2022-06-18',       # proactive decision (Scenario A)
    'reaction_date': '2022-06-19',       # reactive decision (Scenario B)
    'trajectory_dates': ('2022-06-14', '2022-06-18'),   # decision days for which the forecast is re-issued
}

# QUICK_TEST = True runs a short period (2015-2025) to check the pipeline end to end.
QUICK_TEST = False
YEARS = list(range(2015, 2026)) if QUICK_TEST else list(range(1951, 2026))

HEAVY_RAIN_THRESHOLD_MM = REGION['hazard_threshold_mm']
FLASH_FLOOD_CONSEC_DAYS = REGION['hazard_consecutive_days']

LEAD_DAYS = 2                          # baseline lead time L (days)

LEAD_DAYS_LIST = [1, 2, 3, 4, 5]       # lead times solved in Step 10b

MANUAL_AS_OF_DATE = REGION['decision_date']

WALK_FORWARD_CASE_STUDY = True         # refit the classifier on all data up to the decision day

DISTRICTS = REGION['districts']
print(f"Years: {YEARS[0]}-{YEARS[-1]} ({len(YEARS)} years) | QUICK_TEST={QUICK_TEST} | Districts: {list(DISTRICTS.keys())}")

import numpy as np
import pandas as pd

# =============================================================================
# METHOD SETTINGS
# =============================================================================
# ---- Occurrence classifier: purged, expanding-window validation ----
# Each block is predicted by a model trained only on rows whose L-day label was observable
# before the block starts.
TUNE_VAL_BLOCKS = [(1996, 2000), (2001, 2005), (2006, 2010), (2011, 2015)]   # hyperparameter selection
OOF_BLOCKS = [(1981, 1985), (1986, 1990), (1991, 1995), (1996, 2000), (2001, 2005),
              (2006, 2010), (2011, 2015), (2016, 2020), (2021, 2025)]       # out-of-sample scores
TEST_START_YEAR = 2021          # 2021-2025: held-out test, never used for tuning or calibration
CLF_PSO_PARTICLES = 10
CLF_PSO_ITERATIONS = 10
TUNE_ALGOS = ['RandomForest', 'XGBoost', 'LightGBM']
RETUNE_PER_LEAD = False         # hyperparameters tuned once at LEAD_DAYS and reused for the other lead times

# ---- Probability calibration ----
# 'platt': two-parameter logistic map, chosen in advance; 'isotonic' is reported for comparison.
CALIBRATION_METHOD = 'platt'

# ---- Severity model (GPD) ----
EVT_THRESHOLD_GRID = np.arange(40.0, 100.1, 2.5)              # threshold diagnostics
EVT_U_SENSITIVITY = [55.0, 60.0, 64.5, 70.0, 75.0, 80.0]      # thresholds refitted in Step 10c
EVT_GOF_BOOT = 300              # parametric-bootstrap replicates for AD / CvM p-values
EVT_PARAM_BOOT = 200            # year-block bootstrap replicates for (xi, sigma_i)
EVT_SEED = 2025

# ---- Decision model and experiments ----
MIN_SERVICE_FRAC = 0.0                              # baseline alpha = 0 (no minimum-service floor)
ALPHA_GRID = [0.0, 0.10, 0.15, 0.20, 0.30]          # floors tested in Step 10c
LAMBDA_GRID = np.round(np.linspace(0.0, 1.0, 21), 2)
N_EPS_POINTS = 25               # epsilon-constraint problems
N_RESTARTS_FRONTIER = 8         # PSO restarts per lambda (benchmark)
PSO_PARTICLES, PSO_ITERATIONS = 40, 100
FEAS_TOL = 1e-8                 # reported feasibility tolerance (relative)
SENS_LAMBDAS = [0.02, 0.05]     # trade-off weights of the sensitivity runs (checked to lie below lambda*)
COST_MULTIPLIERS = {'h': [0.5, 1.0, 1.5, 2.0], 's': [0.5, 1.0, 2.0, 5.0],
                    'o': [0.5, 1.0, 2.0], 'B': [0.5, 0.75, 1.0, 1.25, 1.5]}

# ---- Exposure source ----
AFFECTED_POP_2022 = REGION['affected_population']
USE_EMPIRICAL_AT_RISK = True
AT_RISK_FRACTION_FALLBACK = 0.5     # used only if demand_source = 'uniform'

DEMAND_SOURCE = REGION.get('demand_source', 'affected_population') if USE_EMPIRICAL_AT_RISK else 'uniform'


def at_risk_from_source(source, classes=None):
    """Per-district at-risk fraction and a provenance label.
    'hazard_atlas'        : flood-prone share of district AREA in the hazard atlas (EX-ANTE),
                            counting the hazard classes in `classes` (default REGION['hazard_classes_at_risk']);
                            the area share is used as a proxy for the population share.
    'affected_population' : documented affected population / census population (EX-POST for 2022).
    'uniform'             : AT_RISK_FRACTION_FALLBACK everywhere (ASSUMED)."""
    dn = list(REGION['districts'])
    if source == 'hazard_atlas':
        cls = classes or REGION['hazard_classes_at_risk']
        fr = {n: sum(REGION['flood_hazard_area_ha'][n][c] for c in cls) / REGION['district_area_ha'][n] for n in dn}
        return fr, f"EX-ANTE (NRSC flood hazard atlas 1998-2015, classes {'+'.join(cls)}: share of district area)"
    if source == 'affected_population':
        return ({n: REGION['affected_population'][n] / REGION['population'][n] for n in dn},
                'EX-POST (ASDMA 2022 memorandum, Table 2.5 / 2011 Census)')
    return {n: AT_RISK_FRACTION_FALLBACK for n in dn}, f'ASSUMED uniform {AT_RISK_FRACTION_FALLBACK}'

CASE_STUDY_DECISION_DATES = pd.date_range(*REGION['trajectory_dates'])

# ---- Figure style (same fonts, colours and symbols as the manuscript) ----
import matplotlib.pyplot as plt
plt.rcParams.update({
    'font.family': 'STIXGeneral', 'mathtext.fontset': 'stix',
    'font.size': 9, 'axes.titlesize': 9, 'axes.titlelocation': 'left', 'axes.labelsize': 9,
    'legend.fontsize': 8, 'legend.frameon': False, 'xtick.labelsize': 8, 'ytick.labelsize': 8,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.edgecolor': '#52514e', 'xtick.color': '#52514e', 'ytick.color': '#52514e',
    'figure.dpi': 110, 'savefig.dpi': 300, 'savefig.bbox': 'tight',
})
DIST_COLORS = {'Cachar': '#2a78d6', 'Karimganj': '#eb6834', 'Hailakandi': '#1baf7a'}


def dcolor(name, i=0):
    "Colour of a district (fixed across figures); other regions fall back to the default cycle."
    return DIST_COLORS.get(name, plt.rcParams['axes.prop_cycle'].by_key()['color'][i % 10])


def light_grid(ax):
    ax.yaxis.grid(True, color='#e6e5e0', lw=0.6); ax.set_axisbelow(True)
print(f"Config: L in {LEAD_DAYS_LIST}, tuning blocks {TUNE_VAL_BLOCKS[0][0]}-{TUNE_VAL_BLOCKS[-1][1]}, "
      f"test block >= {TEST_START_YEAR}, calibration={CALIBRATION_METHOD}, alpha(baseline)={MIN_SERVICE_FRAC}")


# ---- Check that every raw input file exists before any long computation ----
def _raw_files(year):
    return [f'{RAIN_DIR}/Rainfall_ind{year}_rfp25.grd',
            f'{MAXTEMP_DIR}/Maxtemp_MaxT_{year}.GRD',
            f'{MINTEMP_DIR}/Mintemp_MinT_{year}.GRD']

_missing = [p for y in YEARS for p in _raw_files(y) if not os.path.exists(p)]
if _missing:
    print(f"{len(_missing)} raw input file(s) missing, e.g.:")
    for p in _missing[:10]:
        print("   ", p)
    raise FileNotFoundError("Raw IMD files missing -- fix RAIN_DIR / MAXTEMP_DIR / MINTEMP_DIR or upload the files.")
print(f"All {3 * len(YEARS)} raw input files found ({YEARS[0]}-{YEARS[-1]}). Outputs go to {OUT_DIR}")

## Step 3: Grid extraction + feature-engineering functions

In [ ]:
# Step 3: read the IMD binary grids and build district-mean daily series and the classifier features.
# All features use only information available at the end of each day (lags and moving sums look backward).

import numpy as np
import pandas as pd

RAIN_NLAT, RAIN_NLON = 129, 135
RAIN_LAT = np.arange(6.5, 38.75, 0.25)
RAIN_LON = np.arange(66.5, 100.25, 0.25)
TEMP_NLAT, TEMP_NLON = 31, 31
TEMP_LAT = np.arange(7.5, 38.5, 1.0)
TEMP_LON = np.arange(67.5, 98.5, 1.0)


def extract_all_regions(regions, years, heavy_rain_threshold, flash_flood_consec_days,
                         rain_dir, maxtemp_dir, mintemp_dir):
    """
    regions: dict of {label: {'LAT_MIN':.., 'LAT_MAX':.., 'LON_MIN':.., 'LON_MAX':..}}
    Returns: dict of {label: dataframe}, with Flash_Flood_Alert already computed.
    Each year's file is opened exactly ONCE regardless of how many regions requested.
    """
    masks = {}
    for label, box in regions.items():
        AR_LAT = (RAIN_LAT >= box['LAT_MIN']) & (RAIN_LAT <= box['LAT_MAX'])
        AR_LON = (RAIN_LON >= box['LON_MIN']) & (RAIN_LON <= box['LON_MAX'])
        AT_LAT = (TEMP_LAT >= box['LAT_MIN']) & (TEMP_LAT <= box['LAT_MAX'])
        AT_LON = (TEMP_LON >= box['LON_MIN']) & (TEMP_LON <= box['LON_MAX'])
        n_rain = int(AR_LAT.sum()) * int(AR_LON.sum())
        n_temp = int(AT_LAT.sum()) * int(AT_LON.sum())
        if n_rain == 0:
            raise ValueError(f"{label}: zero rainfall grid cells -- widen the bounding box.")
        masks[label] = {'AR_LAT': AR_LAT, 'AR_LON': AR_LON, 'AT_LAT': AT_LAT, 'AT_LON': AT_LON,
                         'n_rain': n_rain, 'n_temp': n_temp}
        print(f"  [{label}] rainfall cells={n_rain}  temp cells={n_temp}")

    rain_rows = {label: [] for label in regions}
    maxt_rows = {label: [] for label in regions}
    mint_rows = {label: [] for label in regions}

    for year in years:
        path = f'{rain_dir}/Rainfall_ind{year}_rfp25.grd'
        data = np.frombuffer(open(path, 'rb').read(), dtype=np.float32)
        days = len(data) // (RAIN_NLAT * RAIN_NLON)
        data = data.reshape(days, RAIN_NLAT, RAIN_NLON)
        dates = pd.date_range(f'{year}-01-01', periods=days, freq='D')
        for label, mk in masks.items():
            region = data[:, mk['AR_LAT'], :][:, :, mk['AR_LON']]
            region = np.where(region < -900, np.nan, region)
            daily_mean = np.nanmean(region, axis=(1, 2))
            rain_rows[label].append(pd.DataFrame({'Date': dates, 'Rainfall': daily_mean}))
        del data

        for var, d, rows in [('MaxTemp', maxtemp_dir, maxt_rows), ('MinTemp', mintemp_dir, mint_rows)]:
            fname = f'Maxtemp_MaxT_{year}.GRD' if var == 'MaxTemp' else f'Mintemp_MinT_{year}.GRD'
            tdata = np.frombuffer(open(f'{d}/{fname}', 'rb').read(), dtype=np.float32)
            tdays = len(tdata) // (TEMP_NLAT * TEMP_NLON)
            tdata = tdata.reshape(tdays, TEMP_NLAT, TEMP_NLON)
            tdates = pd.date_range(f'{year}-01-01', periods=tdays, freq='D')
            for label, mk in masks.items():
                if mk['n_temp'] > 0:
                    region = tdata[:, mk['AT_LAT'], :][:, :, mk['AT_LON']]
                    region = np.where((region > 99) | (region < -90), np.nan, region)
                    daily_mean = np.nanmean(region, axis=(1, 2))
                    rows[label].append(pd.DataFrame({'Date': tdates, var: daily_mean}))
            del tdata

    results = {}
    for label, mk in masks.items():
        df_rain = pd.concat(rain_rows[label], ignore_index=True)
        if mk['n_temp'] > 0:
            df_maxT = pd.concat(maxt_rows[label], ignore_index=True)
            df_minT = pd.concat(mint_rows[label], ignore_index=True)
        else:
            df_maxT = pd.DataFrame({'Date': df_rain['Date'], 'MaxTemp': np.nan})
            df_minT = pd.DataFrame({'Date': df_rain['Date'], 'MinTemp': np.nan})

        df = df_rain.merge(df_maxT, on='Date').merge(df_minT, on='Date')
        df = df.sort_values('Date').reset_index(drop=True)
        df['Rainfall'] = df['Rainfall'].clip(lower=0)
        df['Heavy_Rain_Flag'] = (df['Rainfall'] >= heavy_rain_threshold).astype(int)
        df['Consec_Heavy_Days'] = df['Heavy_Rain_Flag'].rolling(flash_flood_consec_days).sum().fillna(0).astype(int)
        df['Flash_Flood_Alert'] = (df['Consec_Heavy_Days'] >= flash_flood_consec_days).astype(int)
        results[label] = df

    return results


def engineer_features(df):
    df = df.copy()
    df['DayOfYear'] = df['Date'].dt.dayofyear
    df['Month'] = df['Date'].dt.month
    df['Year'] = df['Date'].dt.year
    for col in ['Rainfall','MaxTemp','MinTemp']:
        df[f'{col}_lag1'] = df[col].shift(1)
        df[f'{col}_lag3'] = df[col].shift(3)
        df[f'{col}_lag7'] = df[col].shift(7)
        df[f'{col}_MA7']  = df[col].rolling(7, min_periods=1).mean()
        df[f'{col}_MA30'] = df[col].rolling(30, min_periods=1).mean()
    df['Rainfall_MA3'] = df['Rainfall'].rolling(3, min_periods=1).sum()
    return df.bfill().ffill()


FEAT_FLOOD = ['DayOfYear','Month','Year','Rainfall','Rainfall_lag1','Rainfall_lag3','Rainfall_lag7',
              'Rainfall_MA7','Rainfall_MA3','MaxTemp_MA7','MinTemp_MA7',
              'Heavy_Rain_Flag','Consec_Heavy_Days']

LEAD_TARGET_COL = 'Flash_Flood_Alert_Lead'

def add_lead_target(df, lead_days, source_col='Flash_Flood_Alert'):
    """Reframes the classifier's target from a same-day NOWCAST ('is heavy
    rain happening today') to an N-day-ahead EARLY WARNING ('will a
    flood-alert day occur within the next N days'), while every FEAT_FLOOD
    feature stays strictly backward-looking (lags/moving averages of PAST
    days only) -- so there is no leakage: the model only ever sees data up
    to and including 'today' to predict something about 'today+1..today+N'.
    Drops the last `lead_days` rows of each district (no future data exists
    to know their label)."""
    df = df.copy()
    future_cols = [df[source_col].shift(-k) for k in range(1, lead_days + 1)]
    df[f'{source_col}_Lead'] = pd.concat(future_cols, axis=1).max(axis=1)
    return df.dropna(subset=[f'{source_col}_Lead']).reset_index(drop=True)

FEAT_RAIN = ['DayOfYear','Month','Year','Rainfall_lag1','Rainfall_lag3','Rainfall_lag7',
             'Rainfall_MA7','Rainfall_MA30']
FEAT_TEMP = ['DayOfYear','Month','Year','MaxTemp_lag1','MaxTemp_lag3','MaxTemp_lag7',
             'MaxTemp_MA7','MaxTemp_MA30','MinTemp_lag1','MinTemp_lag3','MinTemp_lag7',
             'MinTemp_MA7','MinTemp_MA30']
print("Extraction and feature-engineering functions ready.")

## Step 4: Extract per-district data (or load cached CSVs from a previous run)

In [ ]:
# Step 4: district daily series 1951-2025. The cache in APP_DIR/data is reused only if it was built with the
# same years, district boxes, threshold and features; otherwise the series are re-extracted from the raw grids.

import json
def _csv_path(name):
    return f"{APP_DIR}/data/district_{name.replace(' ','_')}.csv"

_CACHE_INFO = f"{APP_DIR}/data/cache_info.json"
_cache_signature = {
    'years': [int(YEARS[0]), int(YEARS[-1])],
    'districts': {k: dict(v) for k, v in DISTRICTS.items()},
    'heavy_rain_threshold_mm': float(HEAVY_RAIN_THRESHOLD_MM),
    'consecutive_days': int(FLASH_FLOOD_CONSEC_DAYS),
    'features': list(FEAT_FLOOD),
    'raw_dirs': [RAIN_DIR, MAXTEMP_DIR, MINTEMP_DIR],
}


def _cache_is_valid():
    if not os.path.exists(_CACHE_INFO) or not all(os.path.exists(_csv_path(n)) for n in DISTRICTS):
        return False, 'no cache made by this notebook'
    try:
        with open(_CACHE_INFO) as f:
            info = json.load(f)
    except Exception:
        return False, 'cache_info.json unreadable'
    if info != _cache_signature:
        return False, 'cache was made with different settings'
    return True, 'cache matches current settings'


_ok, _why = _cache_is_valid()
district_raw = {}
if _ok and not FORCE_REEXTRACT:
    print(f"Loading cached per-district CSVs made by this notebook ({_why}).")
    for name in DISTRICTS:
        district_raw[name] = pd.read_csv(_csv_path(name), parse_dates=['Date'])
        print(f"  loaded {name}: {len(district_raw[name])} rows")
else:
    print(f"Extracting {YEARS[0]}-{YEARS[-1]} from the raw IMD grid files ({'forced' if FORCE_REEXTRACT else _why}). "
          "This is the slow, Drive-I/O-bound step (a few minutes).")
    extracted = extract_all_regions(DISTRICTS, YEARS, HEAVY_RAIN_THRESHOLD_MM, FLASH_FLOOD_CONSEC_DAYS,
                                    RAIN_DIR, MAXTEMP_DIR, MINTEMP_DIR)
    for name in DISTRICTS:
        df_d_eng = engineer_features(extracted[name]).dropna()
        district_raw[name] = df_d_eng
        df_d_eng.to_csv(_csv_path(name), index=False)
    with open(_CACHE_INFO, 'w') as f:
        json.dump(_cache_signature, f, indent=1)
    print("Extraction complete. Cached in", f"{APP_DIR}/data")

for name, d in district_raw.items():
    missing = [c for c in FEAT_FLOOD + ['Date', 'Rainfall', 'Flash_Flood_Alert'] if c not in d.columns]
    if missing:
        raise ValueError(f"{name}: columns missing {missing}. Set FORCE_REEXTRACT = True in Step 2 and rerun Step 4.")
    y0, y1 = d['Date'].min().year, d['Date'].max().year
    if (y0, y1) != (YEARS[0], YEARS[-1]):
        raise ValueError(f"{name}: data cover {y0}-{y1}, expected {YEARS[0]}-{YEARS[-1]}. "
                         "Set FORCE_REEXTRACT = True in Step 2 and rerun Step 4.")

district_raw_base = {name: df.copy() for name, df in district_raw.items()}

for name in list(district_raw.keys()):
    district_raw[name] = add_lead_target(district_raw[name], LEAD_DAYS)

print()
for name, d in district_raw.items():
    print(f"{name:<14} {len(d)} days  ({d['Date'].min().date()} to {d['Date'].max().date()})  "
          f"same-day alert days: {int(d['Flash_Flood_Alert'].sum())}  "
          f"{LEAD_DAYS}-day-ahead alert days: {int(d[LEAD_TARGET_COL].sum())}")


## Step 5: Rainfall/MaxTemp/MinTemp -- untuned baseline only (no PSO tuning)

For reference only -- these models don't feed the decision model. Skip this cell if you don't need the comparison numbers; nothing downstream depends on it.

In [ ]:
# Step 5: untuned reference regressors for rainfall and temperature. For comparison only; they are not used
# by the decision model.

import xgboost as xgb
import lightgbm as lgb
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

REG_TARGETS = {'Rainfall': FEAT_RAIN, 'MaxTemp': FEAT_TEMP, 'MinTemp': FEAT_TEMP}

def train_eval_reg(build_fn, Xtr, ytr, Xte, yte):
    m = build_fn().fit(Xtr, ytr)
    pred = m.predict(Xte)
    rmse = float(np.sqrt(mean_squared_error(yte, pred)))
    r2 = float(r2_score(yte, pred))
    return m, rmse, r2

REG_BUILDERS = {
    'RandomForest': lambda: RandomForestRegressor(n_estimators=200, max_depth=10, n_jobs=-1, random_state=42),
    'XGBoost':      lambda: xgb.XGBRegressor(n_estimators=200, max_depth=6, learning_rate=0.1, random_state=42, verbosity=0),
    'LightGBM':     lambda: lgb.LGBMRegressor(n_estimators=200, num_leaves=31, learning_rate=0.1, random_state=42, verbose=-1),
}

baseline_regressors = {}
print(f"{'District':<14}{'Target':<10}{'Algorithm':<15}{'RMSE':>10}{'R2':>10}")
print("-"*60)
for name, df_eng_d in district_raw.items():
    split_d = int(len(df_eng_d)*0.8)
    train_d, test_d = df_eng_d.iloc[:split_d], df_eng_d.iloc[split_d:]
    baseline_regressors[name] = {}
    for target, feats in REG_TARGETS.items():
        Xtr, ytr = train_d[feats], train_d[target]
        Xte, yte = test_d[feats], test_d[target]
        for algo, build_fn in REG_BUILDERS.items():
            model, rmse, r2 = train_eval_reg(build_fn, Xtr, ytr, Xte, yte)
            baseline_regressors[name][(target, algo)] = model
            print(f"{name:<14}{target:<10}{algo:<15}{rmse:>10.3f}{r2:>10.3f}")

print("\nBaseline regressor models trained (untuned). Not used by Steps 7-9 -- "
      "kept for your own comparison only.")


## Step 6: Classifier hyperparameters by purged expanding-window validation

PSO over RandomForest/XGBoost/LightGBM. Each candidate is scored by mean PR-AUC over the temporally ordered validation blocks in `TUNE_VAL_BLOCKS`, and every block is predicted by a model trained only on earlier, label-observable rows. There is no random split anywhere.

In [ ]:
# Step 6: classifier hyperparameters by purged, expanding-window validation (paper, Section 5.2).
# PSO searches RandomForest, XGBoost and LightGBM. Each candidate is scored by its mean PR-AUC over
# TUNE_VAL_BLOCKS, each block being predicted by a model trained only on earlier, label-observable rows.

import time
import xgboost as xgb
import lightgbm as lgb
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (f1_score, precision_score, recall_score, roc_auc_score,
                             average_precision_score, brier_score_loss, log_loss)


def pso_optimize(objective_fn, bounds, n_particles=15, n_iterations=15, w=0.7, c1=1.5, c2=1.5,
                  seed=42, verbose=True, metric_name='metric'):
    """PSO for classifier hyperparameters. objective_fn(p) -> loss or (loss, metric)."""
    rng = np.random.default_rng(seed)
    n_dims = len(bounds)
    lo = np.array([b[0] for b in bounds]); hi = np.array([b[1] for b in bounds])
    pos = lo + rng.random((n_particles, n_dims)) * (hi - lo)
    vel = rng.normal(0, 1, (n_particles, n_dims)) * (hi - lo) * 0.1

    def eval_all(P):
        losses = np.empty(len(P)); metrics = np.full(len(P), np.nan)
        for i, p in enumerate(P):
            out = objective_fn(p)
            if isinstance(out, tuple):
                losses[i], metrics[i] = float(out[0]), float(out[1])
            else:
                losses[i] = float(out)
        return losses, metrics

    pbest_pos = pos.copy()
    pbest_val, pbest_metric = eval_all(pos)
    g = int(np.argmin(pbest_val))
    gbest_pos, gbest_val, gbest_metric = pbest_pos[g].copy(), pbest_val[g], pbest_metric[g]
    history = [{'iteration': 0, 'best_loss': gbest_val, 'best_metric': gbest_metric}]
    if verbose:
        print(f"    iter  0/{n_iterations}: best loss = {gbest_val:.4f}   best {metric_name} = {gbest_metric:.4f}")
    for it in range(n_iterations):
        r1 = rng.random((n_particles, n_dims)); r2 = rng.random((n_particles, n_dims))
        vel = w * vel + c1 * r1 * (pbest_pos - pos) + c2 * r2 * (gbest_pos - pos)
        pos = np.clip(pos + vel, lo, hi)
        vals, metrics = eval_all(pos)
        imp = vals < pbest_val
        pbest_pos[imp] = pos[imp]; pbest_val[imp] = vals[imp]; pbest_metric[imp] = metrics[imp]
        b = int(np.argmin(pbest_val))
        if pbest_val[b] < gbest_val:
            gbest_val, gbest_pos, gbest_metric = pbest_val[b], pbest_pos[b].copy(), pbest_metric[b]
        history.append({'iteration': it + 1, 'best_loss': gbest_val, 'best_metric': gbest_metric})
        if verbose:
            print(f"    iter {it+1:2d}/{n_iterations}: best loss = {gbest_val:.4f}   best {metric_name} = {gbest_metric:.4f}")
    return gbest_pos, gbest_val, history


def build_rf_clf(p, y):
    return RandomForestClassifier(
        n_estimators=int(round(p[0])), max_depth=int(round(p[1])),
        min_samples_split=int(round(p[2])), min_samples_leaf=int(round(p[3])),
        class_weight='balanced', n_jobs=-1, random_state=42)

def build_xgb_clf(p, y):
    n_pos = max(int(np.sum(y)), 1)
    return xgb.XGBClassifier(
        n_estimators=int(round(p[0])), max_depth=int(round(p[1])), learning_rate=float(p[2]),
        subsample=float(p[3]), colsample_bytree=float(p[4]), min_child_weight=float(p[5]),
        scale_pos_weight=(len(y) - n_pos) / n_pos, tree_method='hist', n_jobs=-1,
        random_state=42, verbosity=0)

def build_lgbm_clf(p, y):
    return lgb.LGBMClassifier(
        n_estimators=int(round(p[0])), num_leaves=int(round(p[1])), learning_rate=float(p[2]),
        subsample=float(p[3]), subsample_freq=1, min_child_samples=int(round(p[4])),
        is_unbalance=True, random_state=42, verbose=-1, n_jobs=-1)

CLF_SEARCH_SPACES = {
    'RandomForest': {'bounds': [(150, 400), (3, 8), (2, 20), (2, 10)], 'build': build_rf_clf},
    'XGBoost':      {'bounds': [(150, 400), (3, 6), (0.01, 0.15), (0.7, 1.0), (0.7, 1.0), (1, 50)], 'build': build_xgb_clf},
    'LightGBM':     {'bounds': [(150, 400), (15, 60), (0.01, 0.15), (0.7, 1.0), (10, 50)], 'build': build_lgbm_clf},
}


def make_pool(district_raw_dict):
    """Stack the 3 districts into one frame (a District column is kept for
    reporting; it is NOT a model feature)."""
    parts = [df.assign(District=n) for n, df in district_raw_dict.items()]
    return pd.concat(parts, ignore_index=True).sort_values(['Date', 'District']).reset_index(drop=True)


def train_cutoff(issue_date, lead):
    """Last row date whose LEAD-day-ahead label is observable at the end of issue_date."""
    return pd.Timestamp(issue_date) - pd.Timedelta(days=int(lead))


def fit_clf(build_fn, params, X, y):
    y = np.asarray(y).astype(int)
    if y.sum() < 2 or y.sum() == len(y):
        return None
    return build_fn(params, y).fit(X, y)


def rolling_origin_scores(pool, build_fn, params, blocks, lead, end_date=None, verbose=False):
    """Out-of-sample raw scores. For block (y0, y1) the model is issued at the end
    of (y0-01-01 - 1 day) and trained on rows with Date <= issue - LEAD (purged)."""
    out = []
    for (y0, y1) in blocks:
        start = pd.Timestamp(f'{y0}-01-01'); end = pd.Timestamp(f'{y1}-12-31')
        if end_date is not None:
            end = min(end, pd.Timestamp(end_date))
        if start > end:
            continue
        issue = start - pd.Timedelta(days=1)
        tr = pool[pool['Date'] <= train_cutoff(issue, lead)]
        va = pool[(pool['Date'] >= start) & (pool['Date'] <= end)]
        if len(va) == 0:
            continue
        clf = fit_clf(build_fn, params, tr[FEAT_FLOOD], tr[LEAD_TARGET_COL])
        if clf is None:
            continue
        raw = clf.predict_proba(va[FEAT_FLOOD])[:, 1]
        out.append(pd.DataFrame({'Date': va['Date'].values, 'District': va['District'].values,
                                 'y': va[LEAD_TARGET_COL].astype(int).values, 'raw': raw,
                                 'block': f'{y0}-{y1}', 'train_max_date': tr['Date'].max()}))
        if verbose:
            print(f"    block {y0}-{y1}: train rows <= {tr['Date'].max().date()} "
                  f"({int(tr[LEAD_TARGET_COL].sum())} pos) -> {len(va)} val rows ({int(va[LEAD_TARGET_COL].sum())} pos)")
    return pd.concat(out, ignore_index=True) if out else pd.DataFrame(
        columns=['Date', 'District', 'y', 'raw', 'block', 'train_max_date'])


def blockwise_ap(oof):
    aps = []
    for _, g in oof.groupby('block'):
        if 0 < g['y'].sum() < len(g):
            aps.append(average_precision_score(g['y'], g['raw']))
    return float(np.mean(aps)) if aps else 0.0


def tune_classifier_temporal(district_raw_dict, lead, verbose=True):
    pool = make_pool(district_raw_dict)
    print(f"Temporal tuning (L={lead}): {len(pool)} pooled rows, {int(pool[LEAD_TARGET_COL].sum())} positives; "
          f"validation blocks {TUNE_VAL_BLOCKS} (purge gap = {lead} days)")
    results, histories, t0 = {}, {}, time.time()
    for algo in TUNE_ALGOS:
        spec = CLF_SEARCH_SPACES[algo]
        def obj(p, spec=spec):
            oof = rolling_origin_scores(pool, spec['build'], p, TUNE_VAL_BLOCKS, lead)
            ap = blockwise_ap(oof)
            return 1.0 - ap, ap
        print(f"\n--- PSO tuning {algo} (objective: mean block PR-AUC) ---")
        best_p, best_loss, hist = pso_optimize(obj, spec['bounds'], n_particles=CLF_PSO_PARTICLES,
                                               n_iterations=CLF_PSO_ITERATIONS, seed=42,
                                               verbose=verbose, metric_name='PR-AUC')
        results[algo] = (best_p, 1.0 - best_loss); histories[algo] = hist
        print(f"{algo:<13} best mean block PR-AUC = {1.0 - best_loss:.4f}")
    best_algo = max(results, key=lambda a: results[a][1])
    params = results[best_algo][0]
    print(f"\nSelected (temporal validation, data <= {TUNE_VAL_BLOCKS[-1][1]} only): {best_algo}  params={np.round(params, 4)}"
          f"   [{time.time() - t0:.0f}s]")
    print("Per-block PR-AUC of the selected model (base rate in brackets):")
    oof = rolling_origin_scores(pool, CLF_SEARCH_SPACES[best_algo]['build'], params, TUNE_VAL_BLOCKS, lead)
    for b, g in oof.groupby('block'):
        ap = average_precision_score(g['y'], g['raw']) if 0 < g['y'].sum() < len(g) else float('nan')
        print(f"  {b}: PR-AUC={ap:.3f}  [{g['y'].mean():.4f}, {int(g['y'].sum())} pos]")
    return dict(algo=best_algo, params=params, build_fn=CLF_SEARCH_SPACES[best_algo]['build'],
                all_results={a: (list(np.round(v[0], 5)), v[1]) for a, v in results.items()},
                histories=histories, lead=lead)

_clf6 = tune_classifier_temporal(district_raw, LEAD_DAYS)
tuned_clf_algo_name = _clf6['algo']; tuned_clf_params = _clf6['params']; tuned_clf_build_fn = _clf6['build_fn']


## Step 7: Out-of-sample scores, probability calibration, held-out test

Rolling-origin scores → Platt calibration map fitted on earlier blocks only → evaluation on the 2021–2025 test block (Brier, BSS, log-loss, ECE, reliability, ROC/PR-AUC). `forecast_as_of()` is the only function that produces a decision-date $p_i$, and it asserts that no information after the decision date is used.

In [ ]:
# Step 7: out-of-sample scores, probability calibration and the 2021-2025 test (Sections 5.2, 6.3; Table 5; Fig. 2).
# The Platt map is fitted only on earlier out-of-sample scores; isotonic regression is reported for comparison.
# forecast_as_of() issues the probability for a decision day using only information available on that day.

import matplotlib.pyplot as plt


_CALIB_SRC = r'''
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression

def _logit_clip(s, eps=1e-6):
    s = np.clip(np.asarray(s, dtype=float), eps, 1 - eps)
    return np.log(s / (1 - s))

def fit_calibrator(raw, y, method='platt'):
    """Fit a calibration map g: raw score -> probability. 'platt' = logistic
    regression on logit(raw) (Platt 1999); 'isotonic' = monotone step map
    (Zadrozny & Elkan 2002)."""
    raw = np.asarray(raw, dtype=float); y = np.asarray(y).astype(int)
    if method == 'platt':
        cal = LogisticRegression(C=1e4, solver='lbfgs', max_iter=1000)
        cal.fit(_logit_clip(raw).reshape(-1, 1), y)
    elif method == 'isotonic':
        cal = IsotonicRegression(y_min=0.0, y_max=1.0, out_of_bounds='clip')
        cal.fit(raw, y)
    else:
        raise ValueError(method)
    return cal

def apply_calibration(cal, method, raw):
    raw = np.asarray(raw, dtype=float)
    if cal is None:
        return raw
    if method == 'platt':
        return cal.predict_proba(_logit_clip(raw).reshape(-1, 1))[:, 1]
    return np.clip(cal.predict(raw), 0.0, 1.0)

'''
exec(_CALIB_SRC)


def expected_calibration_error(y, p, n_bins=10):
    """ECE with equal-count (quantile) bins -- equal-width bins are useless when
    99% of predictions sit near zero."""
    y = np.asarray(y); p = np.asarray(p)
    order = np.argsort(p); bins = np.array_split(order, n_bins)
    return float(sum(len(b) / len(p) * abs(y[b].mean() - p[b].mean()) for b in bins if len(b)))


def reliability_points(y, p, n_bins=10):
    y = np.asarray(y); p = np.asarray(p)
    order = np.argsort(p); bins = np.array_split(order, n_bins)
    return np.array([p[b].mean() for b in bins if len(b)]), np.array([y[b].mean() for b in bins if len(b)])


def prob_metrics(y, p, base_rate):
    y = np.asarray(y).astype(int); p = np.clip(np.asarray(p, dtype=float), 1e-9, 1 - 1e-9)
    two = 0 < y.sum() < len(y)
    bs = brier_score_loss(y, p); bs_clim = brier_score_loss(y, np.full(len(y), base_rate))
    return {'n': len(y), 'pos': int(y.sum()), 'rate': float(y.mean()),
            'ROC_AUC': roc_auc_score(y, p) if two else np.nan,
            'PR_AUC': average_precision_score(y, p) if two else np.nan,
            'Brier': bs, 'Brier_clim': bs_clim, 'BSS': 1 - bs / bs_clim if (bs_clim > 0 and y.sum() > 0) else np.nan,
            'LogLoss': log_loss(y, p, labels=[0, 1]), 'ECE': expected_calibration_error(y, p),
            'mean_p': float(p.mean())}


def build_classifier_pipeline(district_raw_dict, lead, build_fn, params, algo, verbose=True, plot=True):
    pool = make_pool(district_raw_dict)
    oof = rolling_origin_scores(pool, build_fn, params, OOF_BLOCKS, lead, verbose=verbose)
    test_start = pd.Timestamp(f'{TEST_START_YEAR}-01-01')
    cal_part = oof[oof['Date'] < test_start]; test_part = oof[oof['Date'] >= test_start].copy()
    base_rate = float(cal_part['y'].mean())
    cals = {m: fit_calibrator(cal_part['raw'], cal_part['y'], m) for m in ('platt', 'isotonic')}
    for m in cals:
        test_part[f'p_{m}'] = apply_calibration(cals[m], m, test_part['raw'])
        oof[f'p_{m}'] = apply_calibration(cals[m], m, oof['raw'])
    grid = np.unique(np.quantile(oof.loc[oof['Date'] < test_start, f'p_{CALIBRATION_METHOD}'], np.linspace(0.90, 0.9999, 60)))
    f1s = [f1_score(cal_part['y'], (oof.loc[oof['Date'] < test_start, f'p_{CALIBRATION_METHOD}'] >= t).astype(int), zero_division=0) for t in grid]
    thr = float(grid[int(np.argmax(f1s))]) if len(grid) else 0.5

    rows = []
    for who, g in [('POOLED', test_part)] + [(n, test_part[test_part['District'] == n]) for n in district_raw_dict]:
        for label, col in [('raw', 'raw'), ('platt', 'p_platt'), ('isotonic', 'p_isotonic')]:
            r = prob_metrics(g['y'], g[col], base_rate); r.update(district=who, scores=label); rows.append(r)
        pred = (g[f'p_{CALIBRATION_METHOD}'] >= thr).astype(int)
        rows[-1].update(dict(threshold=thr, precision=precision_score(g['y'], pred, zero_division=0),
                             recall=recall_score(g['y'], pred, zero_division=0), F1=f1_score(g['y'], pred, zero_division=0)))
    test_metrics = pd.DataFrame(rows)
    if verbose:
        print(f"\nHELD-OUT TEST ({TEST_START_YEAR}-{test_part['Date'].max().year}, L={lead}, {algo}); "
              f"calibrator fitted on {len(cal_part)} earlier out-of-sample rows ({int(cal_part['y'].sum())} pos); "
              f"climatological base rate = {base_rate:.5f}")
        cols = ['district', 'scores', 'n', 'pos', 'ROC_AUC', 'PR_AUC', 'Brier', 'Brier_clim', 'BSS', 'LogLoss', 'ECE', 'mean_p']
        with pd.option_context('display.width', 200, 'display.max_columns', 30):
            print(test_metrics[cols].round(5).to_string(index=False))
            print("\nThreshold metrics (reporting only; the optimiser never thresholds):")
            print(test_metrics.dropna(subset=['F1'])[['district', 'threshold', 'precision', 'recall', 'F1', 'pos']].round(4).to_string(index=False))
    if plot:
        fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.2))
        _end = test_part['Date'].max().year
        for ax, (title, g) in zip(axes, [(f'(a) Calibration blocks, before {TEST_START_YEAR} ($L={lead}$)', oof[oof['Date'] < test_start]),
                                          (f'(b) Test block, {TEST_START_YEAR}\u2013{_end} ($L={lead}$)', test_part)]):
            for label, col, mk, c in [('raw scores', 'raw', 'x', '#8a8983'), ('Platt', 'p_platt', 'o', '#2a78d6'),
                                      ('isotonic', 'p_isotonic', 's', '#eb6834')]:
                if g['y'].sum() > 0:
                    px, py = reliability_points(g['y'], g[col], n_bins=10)
                    ax.plot(px, py, marker=mk, ms=4, lw=1.2, color=c, label=label)
            ax.plot([0, 1], [0, 1], 'k--', lw=0.8, label='perfect calibration')
            ax.set_xscale('symlog', linthresh=1e-3); ax.set_yscale('symlog', linthresh=1e-3)
            ax.set_xlabel('mean predicted probability $p_i$ (quantile bins)'); ax.set_ylabel('observed alert frequency')
            ax.set_title(title); ax.legend(loc='upper left')
        fig.tight_layout()
        fig.savefig(f'{APP_DIR}/reliability_L{lead}.png'); plt.show()

    tr = pool[pool['Date'] <= train_cutoff(test_start - pd.Timedelta(days=1), lead)]
    test_model = fit_clf(build_fn, params, tr[FEAT_FLOOD], tr[LEAD_TARGET_COL])
    return dict(lead=lead, algo=algo, build_fn=build_fn, params=params, pool=pool, oof=oof,
                calibrators=cals, base_rate=base_rate, test_metrics=test_metrics, threshold=thr,
                test_model=test_model, district_raw=district_raw_dict)


def forecast_as_of(pipe, T, method=None, verbose=True):
    """Strictly out-of-sample forecast issued at the END of day T.
    - model trained on rows with Date <= T - L   (labels observable at T)
    - calibrator fitted on rolling-origin out-of-sample scores with Date <= T - L
    Returns calibrated p_i, raw scores, and an information-set audit."""
    method = method or CALIBRATION_METHOD
    T = pd.Timestamp(T); L = pipe['lead']; pool = pipe['pool']; cut = train_cutoff(T, L)
    tr = pool[pool['Date'] <= cut]
    clf = fit_clf(pipe['build_fn'], pipe['params'], tr[FEAT_FLOOD], tr[LEAD_TARGET_COL])
    oof_T = pipe['oof'][pipe['oof']['Date'] <= cut]
    cal = fit_calibrator(oof_T['raw'], oof_T['y'], method)
    raw, p = {}, {}
    for n in pipe['district_raw']:
        row = pool[(pool['District'] == n) & (pool['Date'] == T)]
        if len(row) == 0:
            raise ValueError(f"No feature row for {n} on {T.date()}")
        raw[n] = float(clf.predict_proba(row[FEAT_FLOOD])[0, 1])
        p[n] = float(apply_calibration(cal, method, [raw[n]])[0])
    audit = {'decision_date': str(T.date()), 'lead': L,
             'train_max_row_date': str(tr['Date'].max().date()),
             'train_max_label_date': str((tr['Date'].max() + pd.Timedelta(days=L)).date()),
             'train_rows': int(len(tr)), 'train_pos': int(tr[LEAD_TARGET_COL].sum()),
             'calib_max_row_date': str(oof_T['Date'].max().date()),
             'calib_models_max_train_date': str(pd.Timestamp(oof_T['train_max_date'].max()).date()),
             'calib_rows': int(len(oof_T)), 'calib_pos': int(oof_T['y'].sum()), 'method': method}
    assert tr['Date'].max() + pd.Timedelta(days=L) <= T, "training label window extends past decision date"
    assert oof_T['Date'].max() + pd.Timedelta(days=L) <= T, "calibration label window extends past decision date"
    assert pd.Timestamp(oof_T['train_max_date'].max()) < T
    if verbose:
        print(f"  [OOS audit] decision {T.date()} (end of day), L={L}: training rows <= {audit['train_max_row_date']} "
              f"(labels observed up to {audit['train_max_label_date']}), {audit['train_rows']} rows/{audit['train_pos']} pos; "
              f"calibration rows <= {audit['calib_max_row_date']} ({audit['calib_rows']} rows/{audit['calib_pos']} pos). "
              f"No information after {T.date()} used.")
    return p, raw, audit, clf, cal


pipelines = {}
pipelines[LEAD_DAYS] = build_classifier_pipeline(district_raw, LEAD_DAYS, tuned_clf_build_fn, tuned_clf_params,
                                                 tuned_clf_algo_name)
_pipe = pipelines[LEAD_DAYS]

district_models = {n: {'df_eng': district_raw_base[n], 'rf_clf': _pipe['test_model'], 'threshold': _pipe['threshold'],
                       'calibrator': _pipe['calibrators'][CALIBRATION_METHOD]} for n in DISTRICTS}


## Step 8: GPD severity model: common shape, district scales, diagnostics

Joint MLE of $(\xi,\sigma_1,\dots,\sigma_n)$, a likelihood-ratio test of the common shape, mean residual life plot, parameter-stability plot, QQ/PP plots, AD/CvM tests with parametric-bootstrap p-values, a declustered refit, and year-block bootstrap SEs.

The main fit (8a) runs first and immediately creates everything Steps 9-10 need; the diagnostics (8b-8e) follow, each timed and protected, so a slow or failing diagnostic can never block the decision model. A progress line is printed for every sub-step.


In [ ]:
# Step 8: common-shape GPD severity model (Sections 5.3, 6.4; Table 7; Fig. 3).
# 8a joint maximum-likelihood fit and likelihood-ratio test of the common shape; 8b year-block bootstrap;
# 8c threshold diagnostics; 8d QQ/PP plots and AD/CvM tests; 8e declustered refit.

import time, json
import matplotlib.pyplot as plt
from scipy.stats import genpareto, chi2
from scipy.optimize import minimize


THRESHOLD = HEAVY_RAIN_THRESHOLD_MM
_evt_rng = np.random.default_rng(EVT_SEED)
_t_step8 = time.time()


def _log(msg):
    print(f"[Step 8 | {time.time() - _t_step8:6.1f}s] {msg}", flush=True)


def gpd_negloglik(xi, sigma, y):
    y = np.asarray(y, dtype=float)
    if sigma <= 0 or not np.isfinite(sigma):
        return np.inf
    if abs(xi) < 1e-8:
        return len(y) * np.log(sigma) + y.sum() / sigma
    z = 1 + xi * y / sigma
    if np.any(z <= 0):
        return np.inf
    return len(y) * np.log(sigma) + (1 + 1 / xi) * np.log(z).sum()


def _num_hessian(f, x, h=1e-4):
    x = np.asarray(x, dtype=float); k = len(x); H = np.zeros((k, k))
    for a in range(k):
        for b in range(a, k):
            ea = np.zeros(k); eb = np.zeros(k); ea[a] = h; eb[b] = h
            H[a, b] = H[b, a] = (f(x + ea + eb) - f(x + ea - eb) - f(x - ea + eb) + f(x - ea - eb)) / (4 * h * h)
    return H


def fit_gpd_pooled(exc_list, se=True, x0=None, fast=False):
    """Joint MLE of (xi, sigma_1..sigma_n) with a COMMON xi.
    x0   : optional warm start [xi, sigma_1..sigma_n] (used by the bootstraps -> much faster).
    fast : looser tolerances, no BFGS polish (bootstrap replicates only; the main fit is never fast)."""
    exc_list = [np.asarray(e, dtype=float) for e in exc_list]

    def nll(theta):
        xi = theta[0]; s = np.exp(theta[1:])
        v = sum(gpd_negloglik(xi, s[i], e) for i, e in enumerate(exc_list))
        return v if np.isfinite(v) else 1e100

    if x0 is None:
        th0 = np.r_[0.1, [np.log(max(e.mean() * 0.9, 1e-3)) for e in exc_list]]
    else:
        th0 = np.r_[x0[0], np.log(np.maximum(np.asarray(x0[1:], dtype=float), 1e-3))]
    tol = 1e-6 if fast else 1e-8
    res = minimize(nll, th0, method='Nelder-Mead',
                   options={'xatol': tol, 'fatol': tol, 'maxiter': 4000 if fast else 20000, 'maxfev': 8000 if fast else 40000})
    if not fast:
        res2 = minimize(nll, res.x, method='BFGS', options={'maxiter': 500})
        if np.isfinite(res2.fun) and res2.fun <= res.fun:
            res = res2
    xi = float(res.x[0]); sig = np.exp(res.x[1:])
    out = {'xi': xi, 'sigma': sig, 'nll': float(res.fun), 'n_exc': [len(e) for e in exc_list]}
    if se:
        try:
            cov = np.linalg.inv(_num_hessian(nll, res.x))
            out['se_xi'] = float(np.sqrt(max(cov[0, 0], 0)))
            out['se_sigma'] = sig * np.sqrt(np.clip(np.diag(cov)[1:], 0, None))
        except np.linalg.LinAlgError:
            out['se_xi'] = np.nan; out['se_sigma'] = np.full(len(sig), np.nan)
    return out


def fit_gpd_single(e):
    r = fit_gpd_pooled([e], se=True)
    return {'xi': r['xi'], 'sigma': float(r['sigma'][0]), 'nll': r['nll'], 'se_xi': r.get('se_xi', np.nan)}


def get_exceedances(df, u, cutoff=None, decluster=False):
    d = df if cutoff is None else df[df['Date'] <= pd.Timestamp(cutoff)]
    r = d['Rainfall'].values
    r = r[np.isfinite(r)]
    if not decluster:
        return r[r > u] - u
    above = r > u; peaks, cur = [], None
    for val, a in zip(r, above):
        if a:
            cur = val if cur is None else max(cur, val)
        elif cur is not None:
            peaks.append(cur); cur = None
    if cur is not None:
        peaks.append(cur)
    return np.array(peaks) - u


def fit_evt(u, cutoff=None, decluster=False, se=True):
    exc = [get_exceedances(district_raw_base[n], u, cutoff, decluster) for n in DISTRICTS]
    return fit_gpd_pooled(exc, se=se), exc


def year_block_bootstrap(u, cutoff=None, B=200, rng=None, verbose=False):
    """Resample whole YEARS with replacement (same years for all districts, so
    within-season clustering and cross-district dependence are preserved).
    Replicates are warm-started from the full-sample fit (fast and stable)."""
    rng = rng or np.random.default_rng(EVT_SEED)
    per_year = {}
    for n in DISTRICTS:
        d = district_raw_base[n] if cutoff is None else district_raw_base[n][district_raw_base[n]['Date'] <= pd.Timestamp(cutoff)]
        yrs = d['Date'].dt.year.values; r = d['Rainfall'].values
        per_year[n] = {y: r[(yrs == y) & (r > u)] - u for y in np.unique(yrs)}
    years = np.array(sorted(per_year[list(DISTRICTS)[0]].keys()))
    full = fit_gpd_pooled([np.concatenate([per_year[n][y] for y in years]) for n in DISTRICTS], se=False)
    x0 = np.r_[full['xi'], full['sigma']]
    draws = []
    for b in range(B):
        ys = rng.choice(years, size=len(years), replace=True)
        exc = [np.concatenate([per_year[n][y] for y in ys]) for n in DISTRICTS]
        if min(len(e) for e in exc) < 10:
            continue
        f = fit_gpd_pooled(exc, se=False, x0=x0, fast=True)
        draws.append(np.r_[f['xi'], f['sigma']])
        if verbose and (b + 1) % 50 == 0:
            _log(f"   year-block bootstrap {b + 1}/{B}")
    return np.array(draws)


def ad_cvm(z):
    z = np.sort(np.clip(z, 1e-12, 1 - 1e-12)); n = len(z); i = np.arange(1, n + 1)
    A2 = -n - np.mean((2 * i - 1) * (np.log(z) + np.log(1 - z[::-1])))
    W2 = 1 / (12 * n) + np.sum((z - (2 * i - 1) / (2 * n)) ** 2)
    return A2, W2


names_evt = list(DISTRICTS.keys())

_log("8a main GPD fit ...")
evt_main, exc_main = fit_evt(THRESHOLD)
if min(len(e) for e in exc_main) < 30:
    raise ValueError(f"Too few exceedances of u={THRESHOLD} mm ({[len(e) for e in exc_main]}): check the rainfall data in Step 4.")
separate = [fit_gpd_single(e) for e in exc_main]
LR = 2 * (evt_main['nll'] - sum(s['nll'] for s in separate))
p_LR = float(chi2.sf(max(LR, 0), df=len(separate) - 1))
print(f"GPD above u={THRESHOLD} mm, full record {district_raw_base[names_evt[0]]['Date'].min().year}-"
      f"{district_raw_base[names_evt[0]]['Date'].max().year}")
print(f"{'District':<12}{'n_exc':>7}{'mean exc':>10}{'max exc':>9} | {'separate xi_i (SE)':>20}{'sigma_i':>9} | {'pooled sigma_i (SE)':>22}")
for i, n in enumerate(names_evt):
    e = exc_main[i]
    print(f"{n:<12}{len(e):>7}{e.mean():>10.2f}{e.max():>9.1f} | {separate[i]['xi']:>11.4f} ({separate[i]['se_xi']:.4f}){separate[i]['sigma']:>9.3f} |"
          f" {evt_main['sigma'][i]:>12.4f} ({evt_main['se_sigma'][i]:.4f})")
print(f"Pooled (common) shape: xi = {evt_main['xi']:.4f}  (observed-information SE {evt_main['se_xi']:.4f})")
print(f"LRT common-xi vs district-specific xi: LR = {LR:.3f}, df = {len(separate)-1}, p = {p_LR:.3f}  -> "
      + ("common shape NOT rejected at 5%: pooling is supported by the data." if p_LR > 0.05 else
         "common shape REJECTED at 5%: report district-specific xi_i or justify pooling differently."))
if evt_main['xi'] >= 1:
    raise ValueError("Pooled xi >= 1: infinite mean, the decision model is undefined. Check the data/threshold.")

xi_pooled = evt_main['xi']
pooled_gpd_models = {n: {'xi': xi_pooled, 'sigma': float(evt_main['sigma'][i]), 'threshold': THRESHOLD}
                     for i, n in enumerate(names_evt)}
district_exceedances = dict(zip(names_evt, exc_main))
evt_summary = {'u': THRESHOLD, 'xi': xi_pooled, 'se_xi_hessian': evt_main['se_xi'],
               'sigma': evt_main['sigma'].tolist(), 'n_exc': [len(e) for e in exc_main],
               'LR': LR, 'LR_p': p_LR, 'separate_xi': [s['xi'] for s in separate],
               'se_xi_boot': np.nan, 'se_sigma_boot': [np.nan] * len(names_evt), 'gof': [], 'declustered': None,
               'diagnostics_failed': []}
_log("8a done -- objects for Steps 9-10 are ready (pooled_gpd_models, evt_summary).")

try:
    _log(f"8b year-block bootstrap, B={EVT_PARAM_BOOT} ...")
    evt_boot = year_block_bootstrap(THRESHOLD, B=EVT_PARAM_BOOT, rng=_evt_rng, verbose=True)
    print(f"Year-block bootstrap ({len(evt_boot)} replicates):")
    print(f"  xi: SE={evt_boot[:, 0].std(ddof=1):.4f}  95% CI [{np.percentile(evt_boot[:, 0], 2.5):.4f}, {np.percentile(evt_boot[:, 0], 97.5):.4f}]")
    for i, n in enumerate(names_evt):
        print(f"  sigma_{n}: SE={evt_boot[:, i+1].std(ddof=1):.4f}  95% CI [{np.percentile(evt_boot[:, i+1], 2.5):.3f}, {np.percentile(evt_boot[:, i+1], 97.5):.3f}]")
    evt_summary['se_xi_boot'] = float(evt_boot[:, 0].std(ddof=1))
    evt_summary['se_sigma_boot'] = evt_boot[:, 1:].std(axis=0, ddof=1).tolist()
    _log("8b done.")
except Exception as _e:
    evt_summary['diagnostics_failed'].append(f'8b: {_e}'); _log(f"8b FAILED (not needed by Steps 9-10): {_e!r}")

try:
    _log(f"8c threshold diagnostics over {len(EVT_THRESHOLD_GRID)} thresholds ...")
    mrl_rows, stab_rows = [], []
    for uu in EVT_THRESHOLD_GRID:
        exc_u = [get_exceedances(district_raw_base[n], uu) for n in names_evt]
        for n, e in zip(names_evt, exc_u):
            if len(e) >= 5:
                mrl_rows.append({'u': uu, 'district': n, 'mean_excess': e.mean(), 'ci': 1.96 * e.std(ddof=1) / np.sqrt(len(e)), 'n': len(e)})
        if min(len(e) for e in exc_u) >= 15:
            f = fit_gpd_pooled(exc_u)
            row = {'u': uu, 'xi': f['xi'], 'xi_ci': 1.96 * f['se_xi'], 'n_min': min(len(e) for e in exc_u)}
            for i, n in enumerate(names_evt):
                row[f'sigma*_{n}'] = f['sigma'][i] - f['xi'] * uu
            stab_rows.append(row)
    mrl_df, stab_df = pd.DataFrame(mrl_rows), pd.DataFrame(stab_rows)

    fig, axes = plt.subplots(1, 3, figsize=(7.4, 2.7))
    for i, n in enumerate(names_evt):
        g = mrl_df[mrl_df['district'] == n]; c = dcolor(n, i)
        axes[0].plot(g['u'], g['mean_excess'], color=c, lw=1.2, label=n)
        axes[0].fill_between(g['u'], g['mean_excess'] - g['ci'], g['mean_excess'] + g['ci'], color=c, alpha=0.15, lw=0)
        axes[2].plot(stab_df['u'], stab_df[f'sigma*_{n}'], marker='.', ms=3, lw=1.2, color=c, label=n)
    axes[1].errorbar(stab_df['u'], stab_df['xi'], yerr=stab_df['xi_ci'], fmt='o-', ms=3, lw=1.2, color='#0b0b0b', ecolor='#8a8983')
    for ax in axes:
        ax.axvline(THRESHOLD, color='k', ls='--', lw=0.8); ax.set_xlabel('threshold $u$ (mm)'); light_grid(ax)
    axes[0].set_title('(a) Mean residual life'); axes[0].set_ylabel(r'$\mathrm{E}[R_{i,t}-u \mid R_{i,t}>u]$ (mm)')
    axes[1].set_title(r'(b) Common shape $\hat\xi(u)$, 95% CI'); axes[1].set_ylabel(r'$\hat\xi$')
    axes[2].set_title(r'(c) Modified scale $\hat\sigma_i-\hat\xi u$'); axes[2].set_ylabel(r'$\hat\sigma_i-\hat\xi u$ (mm)')
    axes[0].legend(loc='upper left')
    fig.tight_layout(); fig.savefig(f'{APP_DIR}/evt_threshold_diagnostics.png'); plt.show(); plt.close(fig)
    print("Parameter stability (pooled fit) at selected thresholds:")
    print(stab_df[stab_df['u'].isin([55.0, 60.0, 62.5, 65.0, 70.0, 75.0, 80.0])].round(4).to_string(index=False))
    _log("8c done.")
except Exception as _e:
    evt_summary['diagnostics_failed'].append(f'8c: {_e}'); _log(f"8c FAILED (not needed by Steps 9-10): {_e!r}")

try:
    _log(f"8d QQ/PP plots + goodness-of-fit bootstrap, B={EVT_GOF_BOOT} ...")
    fig, axes = plt.subplots(2, len(names_evt), figsize=(2.5 * len(names_evt), 5.0), squeeze=False)
    gof_obs = []
    for i, n in enumerate(names_evt):
        e = np.sort(exc_main[i]); m = len(e); pp = (np.arange(1, m + 1) - 0.5) / m
        theo_q = genpareto.ppf(pp, evt_main['xi'], scale=evt_main['sigma'][i])
        z = genpareto.cdf(e, evt_main['xi'], scale=evt_main['sigma'][i])
        c = dcolor(n, i)
        axes[0, i].scatter(theo_q, e, s=8, color=c); lim = max(theo_q.max(), e.max()); axes[0, i].plot([0, lim], [0, lim], 'k--', lw=0.8)
        axes[0, i].set_title(f'Q\u2013Q, {n}'); axes[0, i].set_xlabel(r'GPD quantile of $E_i$ (mm)'); axes[0, i].set_ylabel(r'empirical quantile of $E_i$ (mm)')
        axes[1, i].scatter(pp, z, s=8, color=c); axes[1, i].plot([0, 1], [0, 1], 'k--', lw=0.8)
        axes[1, i].set_title(f'P\u2013P, {n}'); axes[1, i].set_xlabel('empirical probability'); axes[1, i].set_ylabel('GPD probability')
        gof_obs.append(ad_cvm(z))
    fig.tight_layout(); fig.savefig(f'{APP_DIR}/evt_qq_pp.png'); plt.show(); plt.close(fig)

    _x0 = np.r_[evt_main['xi'], evt_main['sigma']]
    boot_stats = []
    for b in range(EVT_GOF_BOOT):
        sim = [genpareto.rvs(evt_main['xi'], scale=evt_main['sigma'][i], size=len(exc_main[i]), random_state=_evt_rng)
               for i in range(len(names_evt))]
        fb = fit_gpd_pooled(sim, se=False, x0=_x0, fast=True)
        boot_stats.append([ad_cvm(genpareto.cdf(sim[i], fb['xi'], scale=fb['sigma'][i])) for i in range(len(names_evt))])
        if (b + 1) % 50 == 0:
            _log(f"   goodness-of-fit bootstrap {b + 1}/{EVT_GOF_BOOT}")
    boot_stats = np.array(boot_stats)
    evt_gof = []
    print(f"Goodness of fit (common-xi model), parametric bootstrap with re-estimation, B={EVT_GOF_BOOT}:")
    for i, n in enumerate(names_evt):
        A2, W2 = gof_obs[i]
        pA = (1 + np.sum(boot_stats[:, i, 0] >= A2)) / (1 + len(boot_stats))
        pW = (1 + np.sum(boot_stats[:, i, 1] >= W2)) / (1 + len(boot_stats))
        evt_gof.append({'district': n, 'AD': A2, 'AD_p': pA, 'CvM': W2, 'CvM_p': pW})
        print(f"  {n:<12} Anderson-Darling A2={A2:.3f} (p={pA:.3f})   Cramer-von Mises W2={W2:.4f} (p={pW:.3f})")
    evt_summary['gof'] = evt_gof
    _log("8d done.")
except Exception as _e:
    evt_summary['diagnostics_failed'].append(f'8d: {_e}'); _log(f"8d FAILED (not needed by Steps 9-10): {_e!r}")

try:
    _log("8e declustered refit ...")
    evt_dc, exc_dc = fit_evt(THRESHOLD, decluster=True)
    theta = [len(a) / len(b) for a, b in zip(exc_dc, exc_main)]
    print(f"Runs-declustered (run length 1) refit: xi={evt_dc['xi']:.4f} (SE {evt_dc['se_xi']:.4f}); sigma_i="
          f"{np.round(evt_dc['sigma'], 3)}; cluster/exceedance ratio (extremal-index estimate) = {np.round(theta, 3)}")
    print("Baseline keeps ALL daily exceedances: the decision model's severity variable is the daily hazard"
          " exceedance (marginal distribution); within-cluster dependence is reflected in the year-block bootstrap SEs.")
    evt_summary['declustered'] = {'xi': evt_dc['xi'], 'sigma': evt_dc['sigma'].tolist(), 'theta': theta}
    _log("8e done.")
except Exception as _e:
    evt_summary['diagnostics_failed'].append(f'8e: {_e}'); _log(f"8e FAILED (not needed by Steps 9-10): {_e!r}")

with open(f'{APP_DIR}/saved/evt_summary.json', 'w') as _f:
    json.dump(evt_summary, _f, indent=1, default=lambda o: o.tolist() if hasattr(o, 'tolist') else str(o))
_log("STEP 8 COMPLETE" + (f" (diagnostics that failed: {evt_summary['diagnostics_failed']})" if evt_summary['diagnostics_failed'] else "")
     + f". Saved saved/evt_summary.json")


## Step 9: Decision model and solvers: exact conic, SLSQP, PSO with feasibility projection

Includes numerical self-checks: closed-form expected shortage vs Monte Carlo, analytic gradient vs finite differences, and the closed-form critical-fractile solution vs the exact solver.

In [ ]:
# Step 9: decision model and solvers (Sections 4 and 5.4).
#   solve_exact          conic reformulation (Proposition 7) in CVXPY with Clarabel: the reference solver
#   solve_slsqp          SQP with analytic gradients: an independent check
#   trace_pareto_frontier  PSO: benchmark, with Euclidean projection onto the feasible set
# The model code is also written to decision_model.py, which the dashboard imports.
# Self-checks: closed-form shortage vs Monte Carlo, analytic vs finite-difference gradient, Eq. (9) vs solver.

_DECISION_MODEL_SRC = r'''
import time
from scipy.optimize import minimize as _sp_minimize
try:
    import cvxpy as cp
    _HAS_CVXPY = True
except Exception:
    _HAS_CVXPY = False


def pso_optimize_decision(objective_fn, bounds, n_particles=30, n_iterations=60, w=0.7, c1=1.5, c2=1.5,
                           seed=42, verbose=True):
    """Standard global-best PSO minimising a scalar objective."""
    rng = np.random.default_rng(seed)
    n_dims = len(bounds)
    lo = np.array([b[0] for b in bounds]); hi = np.array([b[1] for b in bounds])
    pos = lo + rng.random((n_particles, n_dims)) * (hi - lo)
    vel = rng.normal(0, 1, (n_particles, n_dims)) * (hi - lo) * 0.1
    pbest_pos = pos.copy(); pbest_val = np.array([objective_fn(p) for p in pos])
    g = int(np.argmin(pbest_val)); gbest_pos = pbest_pos[g].copy(); gbest_val = pbest_val[g]
    for it in range(n_iterations):
        r1 = rng.random((n_particles, n_dims)); r2 = rng.random((n_particles, n_dims))
        vel = w * vel + c1 * r1 * (pbest_pos - pos) + c2 * r2 * (gbest_pos - pos)
        pos = np.clip(pos + vel, lo, hi)
        vals = np.array([objective_fn(p) for p in pos])
        imp = vals < pbest_val
        pbest_pos[imp] = pos[imp]; pbest_val[imp] = vals[imp]
        b = int(np.argmin(pbest_val))
        if pbest_val[b] < gbest_val:
            gbest_val = pbest_val[b]; gbest_pos = pbest_pos[b].copy()
    return gbest_pos, gbest_val


class PrepositioningModel:
    """min (f1, f2) over x_ij >= 0 (stock at warehouse j reserved for district i)
    f1 = sum_j h_j Y_j + sum_i p_i sum_j c_ij x_ij + sum_i (1-p_i) o_i X_i
    f2 = sum_i p_i s_i E[(D_i - X_i)^+],  D_i = d_i (u+E_i)/(u+Ebar_i),  E_i ~ GPD(xi, sigma_i)
    s.t. sum_i x_ij <= K_j,  sum_ij pi_ij x_ij <= B,  X_i >= alpha d_i (alpha = min_service_frac),
         x_ij = 0 for disallowed routes.
    xi may be a scalar (common shape -- the paper's baseline) or a length-n array."""

    def __init__(self, p, d, xi, sigma, u, h, c, s, o, K, B, pi, min_service_frac=0.0, disallowed=None):
        self.p = np.array(p, dtype=float); self.n = len(self.p)
        self.d = np.array(d, dtype=float)
        self.xi = np.broadcast_to(np.array(xi, dtype=float), (self.n,)).copy()
        self.sigma = np.array(sigma, dtype=float); self.u = float(u)
        self.h = np.array(h, dtype=float); self.m = len(self.h)
        self.c = np.array(c, dtype=float); self.s = np.broadcast_to(np.array(s, dtype=float), (self.n,)).copy()
        self.o = np.broadcast_to(np.array(o, dtype=float), (self.n,)).copy()
        self.K = np.array(K, dtype=float); self.B = float(B); self.pi = np.array(pi, dtype=float)
        self.min_service_frac = float(min_service_frac)
        self.disallowed = [tuple(t) for t in (disallowed or [])]
        if np.any(self.xi >= 1):
            raise ValueError("xi >= 1: GPD mean is infinite, expected shortage undefined")
        self.E_bar = self.sigma / (1 - self.xi)
        self.k = self.d / (self.u + self.E_bar)
        self.u_prime = self.k * self.u
        self.sigma_prime = self.k * self.sigma

    def unflatten(self, x_flat):
        return np.asarray(x_flat, dtype=float).reshape(self.n, self.m)

    def allowed_mask(self):
        M = np.ones((self.n, self.m), dtype=bool)
        for i, j in self.disallowed:
            M[i, j] = False
        return M

    def marginal_prep_cost(self):
        """m_ij = d f1 / d x_ij = h_j + p_i c_ij + (1 - p_i) o_i  (constant: f1 is linear)."""
        return self.h[None, :] + self.p[:, None] * self.c + ((1 - self.p) * self.o)[:, None]

    def survival(self, X):
        """P(D_i > X_i | hazard). Equals 1 for X_i <= u'_i (D_i >= u'_i always)."""
        X = np.asarray(X, dtype=float); out = np.ones(self.n)
        for i in range(self.n):
            y = X[i] - self.u_prime[i]; xi, sp = self.xi[i], self.sigma_prime[i]
            if y <= 0:
                continue
            if abs(xi) < 1e-8:
                out[i] = np.exp(-y / sp)
            elif xi < 0 and y >= -sp / xi:
                out[i] = 0.0
            else:
                out[i] = (1 + xi * y / sp) ** (-1 / xi)
        return out

    def expected_shortage_units(self, X):
        """E[(D_i - X_i)^+ | hazard], closed form (Proposition 1, both branches)."""
        X = np.asarray(X, dtype=float); out = np.zeros(self.n)
        for i in range(self.n):
            y = X[i] - self.u_prime[i]; xi, sp = self.xi[i], self.sigma_prime[i]
            if y <= 0:
                out[i] = self.d[i] - X[i]
            elif xi < 0 and y >= -sp / xi:
                out[i] = 0.0
            elif abs(xi) < 1e-8:
                out[i] = sp * np.exp(-y / sp)
            else:
                out[i] = (sp + xi * y) / (1 - xi) * (1 + xi * y / sp) ** (-1 / xi)
        return out

    def f1_components(self, x):
        X = x.sum(axis=1)
        holding = np.sum(self.h[None, :] * x, axis=1)
        transport = np.sum(self.p[:, None] * self.c * x, axis=1)
        waste = (1 - self.p) * self.o * X
        return holding, transport, waste

    def f1(self, x):
        return float(sum(np.sum(t) for t in self.f1_components(x)))

    def f2_per_district(self, x):
        return self.p * self.s * self.expected_shortage_units(x.sum(axis=1))

    def f2(self, x):
        return float(np.sum(self.f2_per_district(x)))

    def grad_f1(self):
        return self.marginal_prep_cost()

    def grad_f2(self, x):
        """d f2 / d x_ij = -p_i s_i P(D_i > X_i)   (the same for every warehouse j)."""
        g = -self.p * self.s * self.survival(x.sum(axis=1))
        return np.repeat(g[:, None], self.m, axis=1)

    def stocking_threshold_lambda(self):
        """lambda*_i = p_i s_i / (p_i s_i + min_j m_ij): district i receives stock at an
        interior optimum of F_lambda only if lambda < lambda*_i (no binding constraints)."""
        m = np.where(self.allowed_mask(), self.marginal_prep_cost(), np.inf).min(axis=1)
        ps = self.p * self.s
        return ps / (ps + m)

    def critical_fractile_solution(self, lam):
        """Closed-form minimiser of F_lambda when capacity/budget/floor do not bind:
        P(D_i > X_i*) = q_i := lam m_i / ((1-lam) p_i s_i)   (newsvendor critical fractile)
        X_i* = 0 if q_i >= 1;  else  u'_i + (s'_i/xi) (q_i^(-xi) - 1)."""
        M = np.where(self.allowed_mask(), self.marginal_prep_cost(), np.inf)
        j_star = M.argmin(axis=1); m = M.min(axis=1); X = np.zeros(self.n); q = np.full(self.n, np.inf)
        for i in range(self.n):
            ps = self.p[i] * self.s[i]
            if lam >= 1 or ps <= 0:
                continue
            q[i] = lam * m[i] / ((1 - lam) * ps) if lam > 0 else 0.0
            if q[i] >= 1:
                continue
            xi, sp = self.xi[i], self.sigma_prime[i]
            if q[i] == 0:
                X[i] = np.inf if xi >= 0 else self.u_prime[i] - sp / xi
            elif abs(xi) < 1e-8:
                X[i] = self.u_prime[i] - sp * np.log(q[i])
            else:
                X[i] = self.u_prime[i] + sp / xi * (q[i] ** (-xi) - 1)
        return X, j_star, q

    def feasibility_report(self, x):
        x = np.asarray(x, dtype=float); Y = x.sum(axis=0); X = x.sum(axis=1)
        cap = np.maximum(Y - self.K, 0.0); bud = max(float(np.sum(self.pi * x)) - self.B, 0.0)
        floor = np.maximum(self.min_service_frac * self.d - X, 0.0)
        neg = float(np.maximum(-x, 0).max()); dis = max([abs(x[i, j]) for i, j in self.disallowed] + [0.0])
        rel = max(float((cap / self.K).max()), bud / self.B,
                  float((floor / np.maximum(self.d, 1e-9)).max()), neg, dis)
        return {'cap_excess_units': cap.tolist(), 'budget_excess_rs': bud, 'floor_deficit_units': floor.tolist(),
                'neg': neg, 'disallowed': dis, 'max_rel_violation': rel}

    def constraint_violation(self, x):
        """Relative squared violation used as the PSO penalty."""
        Y = x.sum(axis=0)
        v = np.sum((np.maximum(Y - self.K, 0.0) / np.maximum(self.K, 1e-9)) ** 2)
        v += (max(np.sum(self.pi * x) - self.B, 0.0) / max(self.B, 1e-9)) ** 2
        if self.min_service_frac > 0:
            fl = self.min_service_frac * self.d
            v += np.sum((np.maximum(fl - x.sum(axis=1), 0.0) / np.maximum(fl, 1e-9)) ** 2)
        return v

    def F_lambda(self, x_flat, lam, penalty_weight=None):
        x = self.unflatten(x_flat)
        f1v, f2v = self.f1(x), self.f2(x)
        if penalty_weight is None:
            penalty_weight = 10.0 * (abs(f1v) + abs(f2v) + self.B + 1.0)
        return lam * f1v + (1 - lam) * f2v + penalty_weight * self.constraint_violation(x)

    def obj_scale(self):
        return max(1.0, float(np.max(self.s) * np.max(self.d)))


def _clean_solution(model, x, zero_tol=1e-4):
    """Remove interior-point round-off: entries below zero_tol units (= 0.1 kit) -> 0,
    disallowed routes -> 0, then a uniform shrink (<= 1e-6 relative) if a <= constraint
    is violated by round-off, so reported solutions satisfy every constraint."""
    x = np.where(np.asarray(x, dtype=float) < zero_tol, 0.0, x)
    for i, j in model.disallowed:
        x[i, j] = 0.0
    Y = x.sum(axis=0); spend = float(np.sum(model.pi * x))
    ratios = [model.K[j] / Y[j] for j in range(model.m) if Y[j] > model.K[j]]
    if spend > model.B:
        ratios.append(model.B / spend)
    if ratios:
        x = x * min(ratios)
    return x

XI_EXP_TOL = 2e-3


def _cvx_power(expr, p):
    """Exact power cone when this cvxpy supports it (>= 1.5, approx=False); older
    versions (e.g. Colab's preinstalled cvxpy) only have the rational SOC approximation
    (relative error ~1e-5), used as a fallback."""
    if abs(1 - p) > 500:
        return cp.exp(-(expr - 1) * (1 - p))
    try:
        return cp.power(expr, p, approx=False)
    except TypeError:
        return cp.power(expr, p)


def _cvx_problem(model, lam=None, eps_f2=None, eps_f1=None, objective='weighted'):
    n, m = model.n, model.m; S = model.obj_scale()
    x = cp.Variable((n, m), nonneg=True); z = cp.Variable(n, nonneg=True)
    X = cp.sum(x, axis=1)
    f1 = (cp.sum(cp.multiply(model.h[None, :] / S * np.ones((n, 1)), x))
          + cp.sum(cp.multiply((model.p[:, None] * model.c) / S, x))
          + cp.sum(cp.multiply((1 - model.p) * model.o / S, X)))
    terms = []
    for i in range(n):
        xi, sp = model.xi[i], model.sigma_prime[i]
        if abs(xi) < XI_EXP_TOL:
            T = sp * cp.exp(-z[i] / sp)
        elif 0 < xi < 1:
            T = sp / (1 - xi) * _cvx_power(1 + xi * z[i] / sp, 1 - 1 / xi)
        elif xi < 0:
            zmax = -sp / xi
            base = cp.pos(1 + xi * cp.minimum(z[i], zmax) / sp)
            T = sp / (1 - xi) * _cvx_power(base, 1 - 1 / xi)
        else:
            raise ValueError("xi >= 1: infinite mean, model undefined")
        terms.append(model.p[i] * model.s[i] / S * (T + z[i] - (X[i] - model.u_prime[i])))
    f2 = cp.sum(cp.hstack(terms))
    cons = [z >= X - model.u_prime, cp.sum(x, axis=0) <= model.K,
            cp.sum(cp.multiply(model.pi / model.B, x)) <= 1.0]
    if model.min_service_frac > 0:
        cons.append(X >= model.min_service_frac * model.d)
    for i, j in model.disallowed:
        cons.append(x[i, j] == 0)
    eps_con = None
    if objective == 'weighted':
        obj = lam * f1 + (1 - lam) * f2
    elif objective == 'f1_given_f2':
        eps_con = f2 <= eps_f2 / S; cons.append(eps_con); obj = f1
    elif objective == 'f2_given_f1':
        eps_con = f1 <= eps_f1 / S; cons.append(eps_con); obj = f2
    else:
        raise ValueError(objective)
    return cp.Problem(cp.Minimize(obj), cons), x, eps_con


def solve_exact(model, lam=None, eps_f2=None, eps_f1=None, objective='weighted', solver=None):
    """Global optimum (to interior-point tolerance) of the convex program."""
    if not _HAS_CVXPY:
        return solve_slsqp(model, lam if lam is not None else 0.5)
    t0 = time.time()
    try:
        prob, xv, eps_con = _cvx_problem(model, lam, eps_f2, eps_f1, objective)
    except Exception as e:
        r = solve_slsqp(model, lam if lam is not None else 0.5, objective=objective, eps_f2=eps_f2, eps_f1=eps_f1)
        r['status'] = f"slsqp-fallback ({type(e).__name__})"
        return r
    status = None
    for sv in ([solver] if solver else ['CLARABEL', 'SCS']):
        try:
            if sv == 'CLARABEL':
                try:
                    prob.solve(solver=sv, tol_gap_abs=1e-10, tol_gap_rel=1e-10, tol_feas=1e-10)
                except TypeError:
                    prob.solve(solver=sv)
            else:
                prob.solve(solver=sv)
            status = prob.status
            if status in ('optimal', 'optimal_inaccurate'):
                break
        except Exception as e:
            status = f'error:{e}'
    if xv.value is None:
        return {'x': None, 'status': status, 'time': time.time() - t0}
    x = _clean_solution(model, np.array(xv.value))
    out = {'x': x, 'f1': model.f1(x), 'f2': model.f2(x), 'status': status, 'time': time.time() - t0, 'solver': 'exact-conic'}
    if lam is not None:
        out['F'] = lam * out['f1'] + (1 - lam) * out['f2']
    if eps_con is not None and eps_con.dual_value is not None:
        mu = float(np.atleast_1d(eps_con.dual_value)[0]); out['dual'] = mu
        out['implied_lambda'] = 1.0 / (1.0 + mu) if objective == 'f1_given_f2' else mu / (1.0 + mu)
    return out


def solve_slsqp(model, lam, n_starts=6, seed=0, objective='weighted', eps_f2=None, eps_f1=None):
    """Deterministic SQP with analytic gradients, multi-start (benchmark / fallback).
    objective: 'weighted' (lam f1 + (1-lam) f2), 'f1_given_f2' (min f1 s.t. f2 <= eps_f2),
    'f2_given_f1' (min f2 s.t. f1 <= eps_f1)."""
    t0 = time.time(); S = model.obj_scale(); n, m = model.n, model.m
    if objective == 'f1_given_f2':
        lam = 1.0
    elif objective == 'f2_given_f1':
        lam = 0.0
    ub = float(max(model.K.max(), model.d.max() * 2))
    bounds = [(0.0, 0.0) if (i, j) in model.disallowed else (0.0, ub) for i in range(n) for j in range(m)]
    def fun(v):
        x = model.unflatten(v); return (lam * model.f1(x) + (1 - lam) * model.f2(x)) / S
    def jac(v):
        x = model.unflatten(v); return ((lam * model.grad_f1() + (1 - lam) * model.grad_f2(x)) / S).ravel()
    A_cap = np.zeros((m, n * m))
    for j in range(m):
        for i in range(n):
            A_cap[j, i * m + j] = 1.0
    cons = [{'type': 'ineq', 'fun': lambda v: model.K - A_cap @ v, 'jac': lambda v: -A_cap},
            {'type': 'ineq', 'fun': lambda v: np.array([1.0 - np.sum(model.pi.ravel() * v) / model.B]),
             'jac': lambda v: -(model.pi.ravel() / model.B)[None, :]}]
    if model.min_service_frac > 0:
        A_X = np.kron(np.eye(n), np.ones((1, m)))
        cons.append({'type': 'ineq', 'fun': lambda v: A_X @ v - model.min_service_frac * model.d, 'jac': lambda v: A_X})
    if objective == 'f1_given_f2':
        cons.append({'type': 'ineq', 'fun': lambda v: np.array([(eps_f2 - model.f2(model.unflatten(v))) / S]),
                     'jac': lambda v: -(model.grad_f2(model.unflatten(v)) / S).ravel()[None, :]})
    elif objective == 'f2_given_f1':
        cons.append({'type': 'ineq', 'fun': lambda v: np.array([(eps_f1 - model.f1(model.unflatten(v))) / S]),
                     'jac': lambda v: -(model.grad_f1() / S).ravel()[None, :]})
    rng = np.random.default_rng(seed); best = None
    for k in range(n_starts):
        v0 = np.array([rng.uniform(lo, hi) * 0.2 for lo, hi in bounds])
        r = _sp_minimize(fun, v0, jac=jac, bounds=bounds, constraints=cons, method='SLSQP',
                         options={'maxiter': 1000, 'ftol': 1e-12})
        x = np.clip(model.unflatten(r.x), 0, None)
        ok_eps = True
        if objective == 'f1_given_f2':
            ok_eps = model.f2(x) <= eps_f2 * (1 + 1e-6) + 1e-3
        elif objective == 'f2_given_f1':
            ok_eps = model.f1(x) <= eps_f1 * (1 + 1e-6) + 1e-3
        if ok_eps and model.feasibility_report(x)['max_rel_violation'] < 1e-6 and (best is None or r.fun < best[1]):
            best = (x, r.fun)
    if best is None:
        return {'x': None, 'status': 'infeasible/failed', 'time': time.time() - t0}
    x = _clean_solution(model, best[0])
    return {'x': x, 'f1': model.f1(x), 'f2': model.f2(x), 'F': lam * model.f1(x) + (1 - lam) * model.f2(x),
            'status': 'optimal', 'time': time.time() - t0, 'solver': 'slsqp'}


def project_feasible(model, x0):
    """Euclidean projection of x0 onto the feasible polytope.
    Written as a second-order-cone problem (minimise ||x - x0||_2) rather than a QP
    (sum_squares): some cvxpy versions (e.g. Colab's) crash in the QP coefficient
    extractor. Same minimiser, conic code path. Falls back to SLSQP if cvxpy fails."""
    x0 = np.asarray(x0, dtype=float); n, m = model.n, model.m
    if _HAS_CVXPY:
        try:
            x = cp.Variable((n, m), nonneg=True); X = cp.sum(x, axis=1)
            cons = [cp.sum(x, axis=0) <= model.K, cp.sum(cp.multiply(model.pi / model.B, x)) <= 1.0]
            if model.min_service_frac > 0:
                cons.append(X >= model.min_service_frac * model.d)
            for i, j in model.disallowed:
                cons.append(x[i, j] == 0)
            cp.Problem(cp.Minimize(cp.norm(x - x0, 'fro')), cons).solve(solver='CLARABEL')
            if x.value is not None:
                return _clean_solution(model, np.array(x.value), zero_tol=0.0)
        except Exception:
            pass
    bounds = [(0.0, 0.0) if (i, j) in model.disallowed else (0.0, None) for i in range(n) for j in range(m)]
    A_cap = np.zeros((m, n * m))
    for j in range(m):
        for i in range(n):
            A_cap[j, i * m + j] = 1.0
    cons = [{'type': 'ineq', 'fun': lambda v: model.K - A_cap @ v},
            {'type': 'ineq', 'fun': lambda v: np.array([1.0 - np.sum(model.pi.ravel() * v) / model.B])}]
    if model.min_service_frac > 0:
        A_X = np.kron(np.eye(n), np.ones((1, m)))
        cons.append({'type': 'ineq', 'fun': lambda v: A_X @ v - model.min_service_frac * model.d})
    r = _sp_minimize(lambda v: np.sum((v - x0.ravel()) ** 2), np.clip(x0.ravel(), 0, None),
                     jac=lambda v: 2 * (v - x0.ravel()), bounds=bounds, constraints=cons, method='SLSQP',
                     options={'maxiter': 500, 'ftol': 1e-14})
    return _clean_solution(model, model.unflatten(r.x), zero_tol=0.0)


def trace_pareto_frontier(model, lambdas=None, n_particles=40, n_iterations=100, seed=42, verbose=True,
                           disallowed=None, n_restarts=3, project=True):
    """PSO weighted-sum sweep (benchmark for the exact solver). For each lambda: n_restarts
    independent swarms, best kept; then (project=True) Euclidean projection onto the
    feasible set. Both the raw and the projected solution's violation are recorded."""
    if lambdas is None:
        lambdas = np.linspace(0.0, 1.0, 21)
    dis = list(disallowed or model.disallowed)
    dim = model.n * model.m
    upper = float(np.max(model.d)) * 1.5
    bounds = [(0, upper)] * dim
    for i, j in dis:
        bounds[i * model.m + j] = (0.0, 0.0)
    frontier = []
    for lam in lambdas:
        t0 = time.time(); obj = lambda xf, lam=lam: model.F_lambda(xf, lam)
        best_x, best_v, restart_F = None, np.inf, []
        for r in range(n_restarts):
            cx, cv = pso_optimize_decision(obj, bounds, n_particles=n_particles, n_iterations=n_iterations,
                                           seed=seed + r * 1009, verbose=False)
            xr = model.unflatten(cx); restart_F.append(lam * model.f1(xr) + (1 - lam) * model.f2(xr))
            if cv < best_v:
                best_x, best_v = cx, cv
        x_raw = model.unflatten(best_x)
        x = project_feasible(model, x_raw) if project else x_raw
        rec = {'lambda': float(lam), 'x': x, 'f1': model.f1(x), 'f2': model.f2(x),
               'F': lam * model.f1(x) + (1 - lam) * model.f2(x),
               'x_raw': x_raw, 'F_raw': lam * model.f1(x_raw) + (1 - lam) * model.f2(x_raw),
               'violation_raw': model.feasibility_report(x_raw)['max_rel_violation'],
               'violation': model.feasibility_report(x)['max_rel_violation'],
               'restart_F': restart_F, 'time': time.time() - t0, 'solver': 'pso'}
        frontier.append(rec)
        if verbose:
            print(f"  PSO lambda={lam:0.2f}: f1={rec['f1']:14,.0f}  f2={rec['f2']:14,.0f}  "
                  f"viol(raw)={rec['violation_raw']:.2e}  viol(proj)={rec['violation']:.2e}")
    return frontier


def trace_pareto_exact(model, lambdas=None, lexicographic=True):
    """Weighted-sum frontier by the exact solver. At lambda in {0,1} the weighted sum is
    only WEAKLY Pareto-optimal, so the other objective is minimised in a second
    (lexicographic) stage."""
    if lambdas is None:
        lambdas = np.linspace(0.0, 1.0, 21)
    out = []
    for lam in lambdas:
        r = solve_exact(model, lam=float(lam))
        if r['x'] is None:
            out.append({'lambda': float(lam), 'x': None, 'status': r['status']}); continue
        if lexicographic and lam in (0.0, 1.0):
            if lam == 0.0:
                r2 = solve_exact(model, eps_f2=r['f2'] * (1 + 1e-7) + 1e-6, objective='f1_given_f2')
            else:
                r2 = solve_exact(model, eps_f1=r['f1'] * (1 + 1e-7) + 1e-6, objective='f2_given_f1')
            if r2['x'] is not None:
                r2['F'] = lam * r2['f1'] + (1 - lam) * r2['f2']; r2.pop('implied_lambda', None); r = r2
        r['lambda'] = float(lam); out.append(r)
    return out


def trace_eps_constraint(model, n_points=25, f2_range=None):
    """epsilon-constraint frontier: min f1 s.t. f2 <= eps. Recovers the points inside the
    linear (flat) segments of the frontier that a finite lambda grid jumps over."""
    if f2_range is None:
        lo = trace_pareto_exact(model, [0.0])[0]; hi = trace_pareto_exact(model, [1.0])[0]
        f2_range = (lo['f2'], hi['f2'])
    pts = []
    for eps in np.linspace(f2_range[0], f2_range[1], n_points):
        r = solve_exact(model, eps_f2=float(eps) * (1 + 1e-9) + 1e-6, objective='f1_given_f2')
        if r['x'] is not None:
            r['eps'] = float(eps); r['lambda'] = float(r.get('implied_lambda', np.nan)); pts.append(r)
    return pts


def pareto_filter(frontier, rel_tol=1e-5):
    pts = [pt for pt in frontier if pt.get('x') is not None]
    uniq = []
    for pt in sorted(pts, key=lambda q: (q['f1'], q['f2'])):
        if not any(abs(pt['f1'] - u['f1']) <= rel_tol * max(1.0, abs(u['f1']), 1e3) and
                   abs(pt['f2'] - u['f2']) <= rel_tol * max(1.0, abs(u['f2'])) for u in uniq):
            uniq.append(pt)
    pts = uniq
    nd = []
    for a in pts:
        dominated = any((b['f1'] <= a['f1'] + 1e-9 and b['f2'] <= a['f2'] + 1e-9) and
                        (b['f1'] < a['f1'] - 1e-6 or b['f2'] < a['f2'] - 1e-6) for b in pts if b is not a)
        if not dominated:
            nd.append(a)
    return sorted(nd, key=lambda pt: pt['f1'])


# Knee points are kept for reference only and are not used in the paper: the exact frontier of the
# case study is essentially linear (Section 6.6), so budget-share operating points are reported.
def knee_point(points):
    """Normalised maximum-distance-to-chord knee of a 2-D frontier (Das 1999 style)."""
    pts = sorted([p for p in points if p.get('x') is not None], key=lambda p: p['f1'])
    if len(pts) < 3:
        return pts[len(pts) // 2]
    f1 = np.array([p['f1'] for p in pts]); f2 = np.array([p['f2'] for p in pts])
    a = (f1 - f1.min()) / max(np.ptp(f1), 1e-12); b = (f2 - f2.min()) / max(np.ptp(f2), 1e-12)
    return pts[int(np.argmax(1 - a - b))]


def summarize_solution(model, x):
    X = x.sum(axis=1); Y = x.sum(axis=0)
    return {'X': X.tolist(), 'total_stock': float(X.sum()), 'f1': model.f1(x), 'f2': model.f2(x),
            'P_short': model.survival(X).tolist(), 'E_short_units': model.expected_shortage_units(X).tolist(),
            'wh_util': (Y / model.K).tolist(), 'budget_util': float(np.sum(model.pi * x) / model.B),
            'max_rel_violation': model.feasibility_report(x)['max_rel_violation']}

'''
exec(_DECISION_MODEL_SRC)
if _HAS_CVXPY:
    print(f"cvxpy {cp.__version__}; solvers: {cp.installed_solvers()}")

with open(f'{APP_DIR}/decision_model.py', 'w') as _f:
    _f.write("import numpy as np\n" + _DECISION_MODEL_SRC + "\n" + _CALIB_SRC)
print(f"Exported decision_model.py (exact solver available: {_HAS_CVXPY}).")

_rng = np.random.default_rng(1)
_m = PrepositioningModel(p=[0.3, 0.1], d=[50, 30], xi=0.12, sigma=[20, 22], u=64.5, h=[1, 1.2],
                         c=[[0.5, 2], [2, 0.5]], s=[40, 40], o=[10, 10], K=[1e6, 1e6], B=1e9, pi=[[1, 1], [1, 1]])
_E = genpareto.rvs(0.12, scale=20, size=400000, random_state=_rng); _D = 50 * (64.5 + _E) / (64.5 + 20 / 0.88)
for _X in [20.0, _m.u_prime[0], 60.0, 90.0]:
    _mc = np.maximum(_D - _X, 0).mean(); _cf = _m.expected_shortage_units(np.array([_X, 0.0]))[0]
    print(f"  check E[(D-X)+] at X={_X:6.2f}: closed form {_cf:.4f}  vs Monte Carlo {_mc:.4f}")
_x = np.array([[30.0, 5.0], [10.0, 12.0]]); _g = (0.4 * _m.grad_f1() + 0.6 * _m.grad_f2(_x)).ravel()
_fd = np.array([(0.4*_m.f1(_m.unflatten(_x.ravel()+e))+0.6*_m.f2(_m.unflatten(_x.ravel()+e))
                 -0.4*_m.f1(_m.unflatten(_x.ravel()-e))-0.6*_m.f2(_m.unflatten(_x.ravel()-e)))/2e-5
                for e in np.eye(4)*1e-5])
print(f"  check gradient: max |analytic - finite diff| = {np.abs(_g-_fd).max():.2e}")
if _HAS_CVXPY:
    for _lam in [0.1, 0.3]:
        _Xcf, _, _ = _m.critical_fractile_solution(_lam); _r = solve_exact(_m, lam=_lam)
        print(f"  check lambda={_lam}: closed-form X*={np.round(_Xcf, 3)}  exact solver X*={np.round(_r['x'].sum(axis=1), 3)}")
print("PrepositioningModel + solvers ready.")


## Step 10: Real parameters, strictly out-of-sample case study, exact Pareto frontier, PSO benchmark

In [ ]:
# Step 10: case-study inputs (Section 6.2; Table 4) and the proactive decision of 18 June 2022
# (Sections 6.5-6.6, 6.10; Tables 8-10; Fig. 4): demand, costs, capacities and budget; exact Pareto
# frontier; PSO benchmark.

import pickle, json
names = list(DISTRICTS.keys())
_need = ['pipelines', 'forecast_as_of', 'fit_evt', 'pooled_gpd_models', 'evt_summary', 'PrepositioningModel']
_miss = [v for v in _need if v not in globals()]
if _miss:
    raise RuntimeError(f'Run Steps 4-9 in this session first (missing: {_miss}).')


CENSUS_2011_POPULATION = dict(REGION['population'])
HOUSEHOLD_SIZE = REGION['household_size']
DEMAND_SCALE = 1000

AT_RISK_BY_DISTRICT, AT_RISK_SOURCE = at_risk_from_source(DEMAND_SOURCE)
AT_RISK_FRACTION = AT_RISK_FRACTION_FALLBACK
d_real = [CENSUS_2011_POPULATION[n] / HOUSEHOLD_SIZE / DEMAND_SCALE * AT_RISK_BY_DISTRICT[n] for n in names]
print(f"At-risk fractions [{AT_RISK_SOURCE}]: " + ", ".join(f"{n} {AT_RISK_BY_DISTRICT[n]:.3f}" for n in names))
print("d_i = 2011 Census population / 5 / 1000 x at-risk fraction (thousand households):")
for n, v in zip(names, d_real):
    print(f"  {n:<14} {v:.1f}")
xi_list = [pooled_gpd_models[n]['xi'] for n in names]
sigma_list = [pooled_gpd_models[n]['sigma'] for n in names]

KIT_WEIGHT_KG = REGION['kit']['weight_kg']
KIT_WEIGHT_TONNE = KIT_WEIGHT_KG / 1000
UNIT_WEIGHT_TONNE = KIT_WEIGHT_TONNE * 1000
DPIIT_ROAD_RATE = REGION['freight_rate_per_tonne_km']

WAREHOUSE_NAMES = [w['name'] for w in REGION['warehouses']]
WAREHOUSE_CAPACITY_MT = {w['name']: w['capacity_t'] for w in REGION['warehouses']}
K_real = [WAREHOUSE_CAPACITY_MT[w] / UNIT_WEIGHT_TONNE for w in WAREHOUSE_NAMES]

DISTANCE_KM = {n: list(REGION['distance_km'][n]) for n in names}
c_real = [[DPIIT_ROAD_RATE * km * UNIT_WEIGHT_TONNE for km in DISTANCE_KM[n]] for n in names]

CLOTHING_RS_PER_FAMILY = REGION['kit']['clothing']
UTENSILS_RS_PER_FAMILY = REGION['kit']['utensils']
ASSAM_MNREGA_RATE_RS_PER_DAY = REGION['kit']['food_daily_wage']
GR_ADULTS_COVERED = REGION['kit']['adults_covered']
GR_DURATION_DAYS = REGION['kit']['relief_days']
FOOD_GR_RS_PER_FAMILY = ASSAM_MNREGA_RATE_RS_PER_DAY * GR_ADULTS_COVERED * GR_DURATION_DAYS
KIT_VALUE_RS = CLOTHING_RS_PER_FAMILY + UTENSILS_RS_PER_FAMILY + FOOD_GR_RS_PER_FAMILY
print(f"Real relief-kit value: clothing Rs{CLOTHING_RS_PER_FAMILY} + utensils Rs{UTENSILS_RS_PER_FAMILY} "
      f"+ food-GR Rs{FOOD_GR_RS_PER_FAMILY:.0f} ({GR_DURATION_DAYS} days @ Rs{ASSAM_MNREGA_RATE_RS_PER_DAY}/day "
      f"x {GR_ADULTS_COVERED} adults) = Rs{KIT_VALUE_RS:.0f} per kit")

pi_real = [[KIT_VALUE_RS * 1000, KIT_VALUE_RS * 1000] for _ in names]

s_real = [KIT_VALUE_RS * 1000] * len(names)

SHELF_LIFE_DAYS = REGION['shelf_life_days']

HOLD_RATE_PER_MONTH = REGION['holding_rate_per_month']
HOLDING_RATE_PCT_PER_MONTH = HOLD_RATE_PER_MONTH


def holding_cost_for_lead(L):
    """h_j per thousand kits for an L-day window: kit value x 1.5%/month x L/30."""
    return [KIT_VALUE_RS * 1000 * HOLD_RATE_PER_MONTH * L / 30] * len(K_real)


def unused_cost_for_lead(L, shelf_life_days=None):
    """o_i per thousand kits for an L-day window: food value x L / shelf life."""
    sl = SHELF_LIFE_DAYS if shelf_life_days is None else shelf_life_days
    return [FOOD_GR_RS_PER_FAMILY * 1000 * L / sl] * len(names)


h_real = holding_cost_for_lead(LEAD_DAYS)
o_real = unused_cost_for_lead(LEAD_DAYS)
print(f"Cost window: L={LEAD_DAYS} days -> h = Rs{h_real[0]:,.0f} and o = Rs{o_real[0]:,.0f} per thousand kits "
      f"(holding {HOLD_RATE_PER_MONTH:.1%}/month x L/30; food Rs{FOOD_GR_RS_PER_FAMILY} x L/{SHELF_LIFE_DAYS} days shelf life)")

RESULTS_JSON = 'final_results.json'

ASSAM_SDRF_RS_CRORE = REGION['budget']['annual_fund_crore']
ASSAM_TOTAL_POPULATION_2011 = REGION['budget']['fund_population']
barak_population_share = sum(CENSUS_2011_POPULATION.values()) / ASSAM_TOTAL_POPULATION_2011
B_real = ASSAM_SDRF_RS_CRORE * 1e7 * barak_population_share
print(f"Barak Valley population share of Assam: {barak_population_share:.3%} of Assam's "
      f"~Rs{ASSAM_SDRF_RS_CRORE}cr SDRF -> budget = Rs{B_real:,.0f}")

print("\nReal, unit-matched K and c:")
print("  K (capacity, thousand-kits): " + "  ".join(f"{w}={k:.1f}" for w, k in zip(WAREHOUSE_NAMES, K_real))
      + f"  (assumes {KIT_WEIGHT_KG}kg/kit)")
for n, row in zip(names, c_real):
    print(f"  c[{n:<10}] = " + "   ".join(f"WH{j+1} {v:.3f}" for j, v in enumerate(row)))

print("\nWarehouse-to-district road distance (km, real -- see DISTANCE_KM source note above):")
_dis = set(REGION['disallowed_routes'])
print(f"  {'District':<14}" + "".join(f"{'-> ' + w:>24}" for w in WAREHOUSE_NAMES))
for n in names:
    print(f"  {n:<14}" + "".join(f"{str(km) + ' km' + (' (disallowed)' if (n, w) in _dis else ''):>24}"
                                  for km, w in zip(DISTANCE_KM[n], WAREHOUSE_NAMES)))

DISALLOWED_ROUTES = [(names.index(dn), WAREHOUSE_NAMES.index(wn)) for dn, wn in REGION['disallowed_routes']]


PARAMETER_PROVENANCE = [
    ('Population (d_i basis)', 'EMPIRICAL', '2011 Census of India'),
    ('Household size = 5', 'EMPIRICAL', '2011 Census (Assam average)'),
    ('At-risk fraction per district', AT_RISK_SOURCE.split(' ')[0],
     AT_RISK_SOURCE + '; varied 0.5-1.5x in Step 10f; other sources compared in Step 10i'),
    ('Area share used as population share', 'ASSUMED' if DEMAND_SOURCE == 'hazard_atlas' else 'n/a',
     'population assumed evenly spread over the district (hazard atlas gives area, not people)'),
    ('2022 affected population', 'EX-POST', 'ASDMA 2022 memorandum Tbl 2.5 (season-cumulative); comparison only (Step 10i)'),
    ('Warehouse capacity K_j (MT)', 'EMPIRICAL', 'Assam State Warehousing Corporation list'),
    ('Road distances', 'EMPIRICAL', 'public distance lookup, rounded to 5 km'),
    ('Road freight rate Rs 3.78/t-km', 'EMPIRICAL', 'DPIIT logistics cost assessment'),
    ('Clothing / utensils Rs 2500 each', 'EMPIRICAL', 'SDRF/NDRF norms 2022-26'),
    ('MGNREGA wage Rs 256/day', 'EMPIRICAL', 'Assam FY2025-26'),
    ('SDRF allocation ~Rs 920 cr', 'EMPIRICAL', 'Assam SDRF FY22-24 average'),
    ('Budget B = population share of SDRF', 'DERIVED', 'empirical inputs, allocation rule assumed'),
    ('Kit weight 50 kg', 'ASSUMED', 'not sourced'),
    ('Relief duration 7 days', 'ASSUMED', 'not sourced'),
    ('Holding rate 1.5%/month, charged for L days (h_j)', 'ASSUMED', 'inventory rule of thumb'),
    ('Procurement pi_ij uniform', 'ASSUMED', 'no route-specific price source'),
    ('Shortage penalty s_i = kit value', 'ASSUMED', 'lower bound on social cost of a missed kit'),
    ('Unused-stock o_i = food value x L / shelf life', 'ASSUMED', 'kits kept for the next alert; clothing/utensils reusable'),
    ('Shelf life of dry rations = 180 days', 'ASSUMED', 'not sourced; varied 90-365 days and rotation loss in Step 10c (6)'),
]
print("\nParameter provenance -- all ASSUMED items are varied in Step 10c:")
for a, b, c_ in PARAMETER_PROVENANCE:
    print(f"  {b:<9} {a:<40} {c_}")

COST_BASE = dict(d=d_real, h=h_real, c=c_real, s=s_real, o=o_real, K=K_real, B=B_real, pi=pi_real)


def make_model(p, xi, sigma, u=HEAVY_RAIN_THRESHOLD_MM, alpha=None, lead=None, **over):
    """lead = lead time L of the scenario being solved (default LEAD_DAYS). h and o are
    recomputed for that L-day window; an explicit h=... or o=... in `over` still wins."""
    prm = dict(COST_BASE)
    L_ = LEAD_DAYS if lead is None else lead
    prm['h'] = holding_cost_for_lead(L_); prm['o'] = unused_cost_for_lead(L_)
    prm.update(over)
    return PrepositioningModel(p=p, d=prm['d'], xi=xi, sigma=sigma, u=u, h=prm['h'], c=prm['c'], s=prm['s'],
                               o=prm['o'], K=prm['K'], B=prm['B'], pi=prm['pi'],
                               min_service_frac=MIN_SERVICE_FRAC if alpha is None else alpha,
                               disallowed=DISALLOWED_ROUTES)


def exact_frontier(model, lambdas=LAMBDA_GRID, n_eps=N_EPS_POINTS):
    ws = trace_pareto_exact(model, lambdas)
    ok = [w for w in ws if w.get('x') is not None]
    eps = trace_eps_constraint(model, n_eps, f2_range=(ok[0]['f2'], ok[-1]['f2'])) if len(ok) >= 2 else []
    return ws, eps, pareto_filter(ws + eps)


def print_allocation(model, x, label):
    holding, transport, waste = model.f1_components(x); sf = model.f2_per_district(x)
    X = x.sum(axis=1); Ps = model.survival(X); Es = model.expected_shortage_units(X)
    print("\n" + "-" * 110 + f"\n{label}\n  f1 = Rs{model.f1(x):,.0f}   f2 = Rs{model.f2(x):,.0f}")
    print(f"  {'District':<12}" + "".join(f"{'WH' + str(j + 1):>7}" for j in range(model.m)) + f"{'X_i':>8}{'d_i':>8}{'u_i':>8}{'p_i':>8}{'P(D>X|H)':>10}{'E[short]':>10}"
          f"{'Holding':>13}{'Transport':>11}{'Waste':>13}{'Shortage':>13}")
    for i, n in enumerate(names):
        print(f"  {n:<12}" + "".join(f"{x[i, j]:>7.1f}" for j in range(model.m)) + f"{X[i]:>8.1f}{model.d[i]:>8.1f}{model.u_prime[i]:>8.1f}{model.p[i]:>8.4f}"
              f"{Ps[i]:>10.3f}{Es[i]:>10.1f}{holding[i]:>13,.0f}{transport[i]:>11,.0f}{waste[i]:>13,.0f}{sf[i]:>13,.0f}")
    Y = x.sum(axis=0); fr = model.feasibility_report(x)
    print("  Warehouses: " + ", ".join(f"WH{j+1} {Y[j]:.1f}/{model.K[j]:.1f}" for j in range(model.m)) + f"   "
          f"Budget Rs{np.sum(model.pi*x):,.0f} / Rs{model.B:,.0f}   max relative violation = {fr['max_rel_violation']:.1e}")
    people = X * 1000 * HOUSEHOLD_SIZE
    print("  People covered: " + ", ".join(f"{n} {people[i]:,.0f} ({100*people[i]/CENSUS_2011_POPULATION[n]:.1f}%)" for i, n in enumerate(names)))


def naive_proportional_allocation(model, total_budget):
    """Business-as-usual: budget split by population share d_i, ignoring p_i and severity."""
    w = model.d / model.d.sum(); want = w * total_budget / model.pi[:, 0]
    x = np.zeros((model.n, model.m)); cap = model.K.astype(float).copy()
    for i in range(model.n):
        need = want[i]
        for j in np.argsort(model.c[i]):
            if (i, j) in model.disallowed or need <= 1e-9:
                continue
            take = min(need, cap[j]); x[i, j] = take; cap[j] -= take; need -= take
    return x


def solve_scenario_A(lead, pipe, run_pso=True, verbose=True, save=True):
    AS_OF_DATE = pd.Timestamp(MANUAL_AS_OF_DATE)
    print("=" * 110 + f"\nSCENARIO A (proactive pre-positioning), L={lead}, decision at end of {AS_OF_DATE.date()}\n" + "=" * 110)
    p_cal, p_raw, audit, _, _ = forecast_as_of(pipe, AS_OF_DATE)
    p_real = [p_cal[n] for n in names]
    evt_T, _ = fit_evt(HEAVY_RAIN_THRESHOLD_MM, cutoff=AS_OF_DATE, se=True)
    audit['evt_data_max_date'] = str(min(AS_OF_DATE, district_raw_base[names[0]]['Date'].max()).date())
    xi_T, sig_T = evt_T['xi'], evt_T['sigma']
    model = make_model(p_real, xi_T, sig_T, lead=lead)
    lam_star = model.stocking_threshold_lambda()
    print(f"Cost window L={lead}: h = Rs{model.h[0]:,.0f}, o = Rs{model.o[0]:,.0f} per thousand kits; "
          f"m_i (cheapest route) = {np.round(np.where(model.allowed_mask(), model.marginal_prep_cost(), np.inf).min(axis=1), 0)}")
    print(f"\nInputs as of {AS_OF_DATE.date()} (p_i = calibrated P(alert day within {lead} days | info at decision time)):")
    print(f"{'District':<12}{'raw score':>11}{'p_i (cal.)':>12}{'d_i':>8}{'xi':>8}{'sigma_i':>9}{'k_i':>8}{'u_i':>9}{'sig_i':>9}{'lambda*_i':>11}")
    for i, n in enumerate(names):
        print(f"{n:<12}{p_raw[n]:>11.4f}{p_real[i]:>12.4f}{model.d[i]:>8.1f}{xi_T:>8.4f}{sig_T[i]:>9.3f}{model.k[i]:>8.3f}"
              f"{model.u_prime[i]:>9.2f}{model.sigma_prime[i]:>9.3f}{lam_star[i]:>11.4f}")
    print("lambda*_i = p_i s_i/(p_i s_i + min_j m_ij): with no binding constraint, district i is stocked only for"
          " lambda < lambda*_i (Corollary 1).")

    ws, eps, pf = exact_frontier(model)
    print(f"\nEXACT frontier: {len(ws)} weighted-sum points (lexicographic at lambda=0,1) + {len(eps)} epsilon-constraint points "
          f"-> {len(pf)} non-dominated points")
    print(f"{'source':>8}{'lambda':>9}{'f1 (Rs)':>18}{'f2 (Rs)':>18}   X_i")
    for pt in pf:
        src = 'eps' if 'eps' in pt else 'wsum'
        print(f"{src:>8}{pt['lambda']:>9.3f}{pt['f1']:>18,.0f}{pt['f2']:>18,.0f}   {np.round(pt['x'].sum(axis=1), 1)}")
    knee = knee_point(pf)

    bench = []
    pso = []
    if run_pso:
        print(f"\nPSO ({N_RESTARTS_FRONTIER} restarts x {PSO_PARTICLES} particles x {PSO_ITERATIONS} iterations) vs exact solver:")
        pso = trace_pareto_frontier(model, LAMBDA_GRID, n_particles=PSO_PARTICLES, n_iterations=PSO_ITERATIONS,
                                    seed=42, verbose=False, n_restarts=N_RESTARTS_FRONTIER, project=True)
        for w, q in zip(ws, pso):
            if w.get('x') is None:
                continue
            Fw = w['lambda'] * w['f1'] + (1 - w['lambda']) * w['f2']
            bench.append({'lambda': w['lambda'], 'F_exact': Fw, 'F_pso_raw': q['F_raw'], 'F_pso_proj': q['F'],
                          'gap_raw_%': 100 * (q['F_raw'] - Fw) / max(abs(Fw), 1), 'gap_proj_%': 100 * (q['F'] - Fw) / max(abs(Fw), 1),
                          'viol_raw': q['violation_raw'], 'viol_proj': q['violation'],
                          'viol_exact': model.feasibility_report(w['x'])['max_rel_violation'],
                          'restart_cv_%': 100 * np.std(q['restart_F']) / max(abs(np.mean(q['restart_F'])), 1),
                          't_exact_s': w['time'], 't_pso_s': q['time']})
        bench_df = pd.DataFrame(bench)
        with pd.option_context('display.width', 220, 'display.float_format', '{:,.4g}'.format):
            print(bench_df.to_string(index=False))
        print(f"Summary: median gap (projected PSO vs exact) = {bench_df['gap_proj_%'].median():.4f}%, "
              f"max = {bench_df['gap_proj_%'].max():.4f}%; max raw PSO violation = {bench_df['viol_raw'].max():.2e}, "
              f"after projection = {bench_df['viol_proj'].max():.2e}, exact = {bench_df['viol_exact'].max():.2e} "
              f"(tolerance {FEAS_TOL:.0e}); runtime exact {bench_df['t_exact_s'].sum():.1f}s vs PSO {bench_df['t_pso_s'].sum():.1f}s")

    lo = [w for w in ws if w.get('x') is not None][0]; hi = [w for w in ws if w.get('x') is not None][-1]
    for pt, lab in [(lo, 'MIN-SHORTAGE END (lambda=0, lexicographic)'),
                    (knee, f"Frontier point at implied lambda = {knee['lambda']:.3f} (reference only)"),
                    (hi, 'MIN-COST END (lambda=1, lexicographic)')]:
        print_allocation(model, pt['x'], lab)

    print("\n" + "=" * 110 + "\nOPTIMISED vs NAIVE population-proportional allocation (same money, same p_i and GPD)\n" + "=" * 110)
    case_rows = []
    for pt, lab in [(lo, 'lambda=0'), (knee, 'knee')]:
        spend = float(np.sum(model.pi * pt['x']))
        if spend <= 0:
            print(f"  {lab}: optimiser spends nothing here -- no comparison."); continue
        xn = naive_proportional_allocation(model, spend)
        f1n, f2n, f1o, f2o = model.f1(xn), model.f2(xn), model.f1(pt['x']), model.f2(pt['x'])
        print(f"  {lab:<9} budget Rs{spend:,.0f}: X naive={np.round(xn.sum(axis=1),1)} opt={np.round(pt['x'].sum(axis=1),1)}  "
              f"f2 naive Rs{f2n:,.0f} -> opt Rs{f2o:,.0f} ({100*(f2o-f2n)/f2n:+.1f}%);  f1 naive Rs{f1n:,.0f} -> opt Rs{f1o:,.0f} ({100*(f1o-f1n)/f1n:+.1f}%)")
        case_rows.append({'point': lab, 'budget': spend, 'X_naive': xn.sum(axis=1).tolist(), 'X_opt': pt['x'].sum(axis=1).tolist(),
                          'f1_naive': f1n, 'f2_naive': f2n, 'f1_opt': f1o, 'f2_opt': f2o})

    res = {'lead': lead, 'AS_OF_DATE': AS_OF_DATE, 'p_real': p_real, 'p_raw': [p_raw[n] for n in names], 'audit': audit,
           'xi': xi_T, 'sigma': sig_T.tolist(), 'model': model, 'weighted': ws, 'eps': eps, 'pf': pf, 'knee': knee,
           'pso': pso, 'bench': bench, 'case_rows': case_rows, 'lambda_star': lam_star.tolist(),
           'balanced': knee, 'frontier': ws}
    if save:
        with open(f'{APP_DIR}/saved/decision_results_scenario_A_lead{lead}.pkl', 'wb') as f:
            pickle.dump({'scenario_label': f"Scenario A: proactive pre-positioning (L={lead}, {AS_OF_DATE.date()})",
                         'districts': names,
                         'pareto_frontier': [{'lambda': pt['lambda'], 'x': pt['x'], 'f1': pt['f1'], 'f2': pt['f2']} for pt in pf],
                         'pso_frontier': [{k: v for k, v in q.items() if k != 'x_raw'} for q in pso],
                         'benchmark': bench, 'audit': audit,
                         'model_params': {'p': p_real, 'p_raw': [p_raw[n] for n in names], 'd': d_real, 'xi': xi_T,
                                          'sigma': sig_T.tolist(), 'u': HEAVY_RAIN_THRESHOLD_MM,
                                          'min_service_frac': MIN_SERVICE_FRAC,
                                          'h': model.h.tolist(), 'o': model.o.tolist()}}, f)
        print(f"\nSaved decision_results_scenario_A_lead{lead}.pkl")
    return res


scenario_A_results_by_lead = {LEAD_DAYS: solve_scenario_A(LEAD_DAYS, pipelines[LEAD_DAYS])}
_resA = scenario_A_results_by_lead[LEAD_DAYS]
p_real, AS_OF_DATE, model, pf = _resA['p_real'], _resA['AS_OF_DATE'], _resA['model'], _resA['pf']

fig, ax = plt.subplots(figsize=(3.6, 2.9))
ax.plot([pt['f1'] / 1e7 for pt in pf], [pt['f2'] / 1e7 for pt in pf], '-', color='#2a78d6', lw=2, label='exact frontier')
ax.scatter([w['f1'] / 1e7 for w in _resA['weighted'] if w.get('x') is not None],
           [w['f2'] / 1e7 for w in _resA['weighted'] if w.get('x') is not None], marker='o', s=16, color='#2a78d6',
           zorder=3, label=r'weighted sum ($\lambda$ grid)')
ax.scatter([e['f1'] / 1e7 for e in _resA['eps']], [e['f2'] / 1e7 for e in _resA['eps']], marker='^', s=14,
           facecolors='none', edgecolors='#0b0b0b', lw=0.8, zorder=3, label=r'$\varepsilon$-constraint')
if _resA['pso']:
    ax.scatter([q['f1'] / 1e7 for q in _resA['pso']], [q['f2'] / 1e7 for q in _resA['pso']], marker='x', s=40,
               color='#eb6834', lw=1.5, zorder=4, label='PSO (projected)')
ax.set_xlabel('$f_1$: expected preparation cost (Rs crore)'); ax.set_ylabel('$f_2$: expected shortage cost (Rs crore)')
light_grid(ax); ax.legend(loc='upper right'); fig.tight_layout()
fig.savefig(f'{APP_DIR}/pareto_frontier_L{LEAD_DAYS}.png'); plt.show()


## Step 10b: Lead-time experiment ($L=1,\dots,5$) and re-issued case-study trajectory

In [ ]:
# Step 10b: lead times L = 1-5 (Section 7.3; Table 16; Fig. 5) and the forecast re-issued for each decision
# day from 14 to 18 June 2022 (Table 6). Every forecast uses only information available on its day.

lead_summary = []
for _lead in LEAD_DAYS_LIST:
    if _lead not in pipelines:
        _dr = {n: add_lead_target(df.copy(), _lead) for n, df in district_raw_base.items()}
        if RETUNE_PER_LEAD:
            _t = tune_classifier_temporal(_dr, _lead, verbose=False)
            _bf, _pp, _al = _t['build_fn'], _t['params'], _t['algo']
        else:
            _bf, _pp, _al = tuned_clf_build_fn, tuned_clf_params, tuned_clf_algo_name
        print(f"\n{'#'*110}\nLEAD_DAYS = {_lead}  ({'retuned' if RETUNE_PER_LEAD else f'hyperparameters from L={LEAD_DAYS}'})\n{'#'*110}")
        pipelines[_lead] = build_classifier_pipeline(_dr, _lead, _bf, _pp, _al, verbose=True, plot=True)
    if _lead not in scenario_A_results_by_lead:
        scenario_A_results_by_lead[_lead] = solve_scenario_A(_lead, pipelines[_lead], run_pso=False)
    tm = pipelines[_lead]['test_metrics']
    pooled = tm[(tm['district'] == 'POOLED') & (tm['scores'] == CALIBRATION_METHOD)].iloc[0]
    rawp = tm[(tm['district'] == 'POOLED') & (tm['scores'] == 'raw')].iloc[0]
    r = scenario_A_results_by_lead[_lead]
    lead_summary.append({'L': _lead, 'test_pos': int(pooled['pos']), 'ROC_AUC': pooled['ROC_AUC'], 'PR_AUC': pooled['PR_AUC'],
                         'base_rate': pooled['rate'], 'Brier_raw': rawp['Brier'], 'Brier_cal': pooled['Brier'],
                         'Brier_clim': pooled['Brier_clim'], 'BSS_cal': pooled['BSS'], 'ECE_cal': pooled['ECE'],
                         **{f'p_{n}': v for n, v in zip(names, r['p_real'])},
                         'knee_total_stock': float(r['knee']['x'].sum()), 'knee_f1': r['knee']['f1'], 'knee_f2': r['knee']['f2']})
lead_df = pd.DataFrame(lead_summary)
print("\n" + "=" * 110 + "\nLEAD-TIME SUMMARY (test block pooled, calibrated scores; case-study p_i at MANUAL_AS_OF_DATE)\n" + "=" * 110)
with pd.option_context('display.width', 250, 'display.float_format', '{:,.4f}'.format):
    print(lead_df.to_string(index=False))
lead_df.to_csv(f'{APP_DIR}/saved/lead_time_summary.csv', index=False)

traj = []
for _lead in LEAD_DAYS_LIST:
    for T in CASE_STUDY_DECISION_DATES:
        pc, pr, au, _, _ = forecast_as_of(pipelines[_lead], T, verbose=False)
        traj.append({'L': _lead, 'decision_date': T.date(), **{n: pc[n] for n in names},
                     'train_rows_upto': au['train_max_row_date'], 'calib_rows_upto': au['calib_max_row_date']})
traj_df = pd.DataFrame(traj)
print("\nCalibrated p_i, each row RE-ISSUED with only information available at the end of its decision date:")
with pd.option_context('display.width', 250, 'display.float_format', '{:.4f}'.format):
    print(traj_df.to_string(index=False))
traj_df.to_csv(f'{APP_DIR}/saved/case_study_trajectory.csv', index=False)
obs = pd.DataFrame({n: district_raw_base[n].set_index('Date').loc['2022-06-12':'2022-06-24', 'Flash_Flood_Alert'] for n in names})
print("\nObserved same-day alert flags, 2022-06-12..24 (for interpretation only):")
print(obs.T.to_string())


lam_rows = []
for _L, _r in sorted(scenario_A_results_by_lead.items()):
    _m = _r['model']
    lam_rows.append({'L': _L, 'h': _m.h[0], 'o': _m.o[0],
                     **{f'p_{n}': _r['p_real'][i] for i, n in enumerate(names)},
                     **{f'm_{n}': float(np.where(_m.allowed_mask(), _m.marginal_prep_cost(), np.inf).min(axis=1)[i])
                        for i, n in enumerate(names)},
                     **{f'lambda*_{n}': _r['lambda_star'][i] for i, n in enumerate(names)},
                     'min_lambda*': min(_r['lambda_star']),
                     'SENS_LAMBDAS_below_lambda*': bool(max(SENS_LAMBDAS) < min(_r['lambda_star']))})
lambda_star_by_lead_df = pd.DataFrame(lam_rows)
print("\nSTOCKING THRESHOLD lambda* BY LEAD TIME (cost window: h and o charged for L days)")
with pd.option_context('display.width', 250, 'display.float_format', '{:,.4f}'.format):
    print(lambda_star_by_lead_df.to_string(index=False))
lambda_star_by_lead_df.to_csv(f'{APP_DIR}/saved/lambda_star_by_lead.csv', index=False)

_Xby = {}
for _L, _r in sorted(scenario_A_results_by_lead.items()):
    _pt = min(_r['pf'], key=lambda q: (round(q['f2'], 0), q['f1']))
    _Xby[_L] = np.asarray(_pt['x']).sum(axis=1)
_m0 = scenario_A_results_by_lead[LEAD_DAYS]['model']
fig, ax = plt.subplots(1, 2, figsize=(7.4, 2.9), gridspec_kw={'width_ratios': [1, 1.5]})
a = ax[0]; xs = np.arange(len(names)); w = 0.55; cols = [dcolor(n, i) for i, n in enumerate(names)]
a.bar(xs, _m0.d, w, color='white', edgecolor='#8a8983', lw=1, label='average-hazard demand $d_i$')
a.bar(xs, _Xby[LEAD_DAYS], w * 0.62, color=cols, edgecolor='white', lw=1, label=f'planned stock $X_i$ ($L={LEAD_DAYS}$)')
for i in range(len(names)):
    a.plot([xs[i] - w / 2, xs[i] + w / 2], [_m0.u_prime[i]] * 2, color='#0b0b0b', lw=2, ls=(0, (3, 1.5)))
    a.text(xs[i], _Xby[LEAD_DAYS][i] + 1.5, f'{_Xby[LEAD_DAYS][i]:.1f}', ha='center', va='bottom', fontsize=8)
a.plot([], [], color='#0b0b0b', lw=2, ls=(0, (3, 1.5)), label="certain hazard demand $u_i'$")
a.set_xticks(xs); a.set_xticklabels(names); a.set_ylabel('thousand households / kits'); a.set_ylim(0, 1.15 * max(_m0.d))
a.set_title(f'(a) Plan of {AS_OF_DATE.day} {AS_OF_DATE:%B %Y} ($L={LEAD_DAYS}$)'); a.legend(loc='upper right', fontsize=7); light_grid(a)
b = ax[1]; Ls = sorted(_Xby); bw = 0.8 / len(names)
for i, n in enumerate(names):
    b.bar(np.arange(len(Ls)) + (i - (len(names) - 1) / 2) * bw, [_Xby[L][i] for L in Ls], bw * 0.92, color=cols[i], edgecolor='white', lw=0.8, label=n)
b.set_xticks(np.arange(len(Ls))); b.set_xticklabels([f'$L={L}$' for L in Ls]); b.set_ylabel('planned stock $X_i$ (thousand kits)')
b.set_ylim(0, 1.15 * max(v.max() for v in _Xby.values()))
b.set_title(f'(b) Planned stock by lead time (total {sum(_Xby[LEAD_DAYS]):.1f} thousand kits)'); b.legend(ncol=3, loc='upper left'); light_grid(b)
fig.tight_layout(); fig.savefig(f'{APP_DIR}/allocation_by_lead.png'); plt.show()
if not lambda_star_by_lead_df['SENS_LAMBDAS_below_lambda*'].all():
    print(f"WARNING: SENS_LAMBDAS={SENS_LAMBDAS} is NOT below lambda* for every lead time -- robustness rows would show zero stock.")
else:
    print(f"OK: SENS_LAMBDAS={SENS_LAMBDAS} are below lambda* at every lead time.")


## Step 10c: Robustness: costs, service floor $\alpha$, EVT threshold $u$, EVT bootstrap, one-at-a-time ±20 %

In [ ]:
# Step 10c: robustness (Sections 7.1-7.2; Tables 14-15): cost multipliers, minimum-service floors, GPD
# threshold, year-block bootstrap of (xi, sigma), one-at-a-time changes and the unused-stock window.
# All problems are solved with the exact solver.

BASE_P = _resA['p_real']; BASE_XI = _resA['xi']; BASE_SIG = np.array(_resA['sigma'])


def robust_rows(label, model, group, value, lambdas=SENS_LAMBDAS, with_knee=True):
    rows = []
    pts = [(lam, solve_exact(model, lam=lam)) for lam in lambdas]
    if with_knee:
        ws = [w for w in trace_pareto_exact(model, LAMBDA_GRID) if w.get('x') is not None]
        if ws:
            k = knee_point(pareto_filter(ws)); pts.append(('knee', k))
    for lam, r in pts:
        if r.get('x') is None:
            rows.append({'group': group, 'value': value, 'label': label, 'lambda': lam, 'status': r.get('status')}); continue
        sm = summarize_solution(model, r['x'])
        row = {'group': group, 'value': value, 'label': label, 'lambda': lam if lam != 'knee' else f"knee({r['lambda']:.2f})",
               'total_stock': sm['total_stock'], 'f1': sm['f1'], 'f2': sm['f2'], 'budget_util': sm['budget_util'],
               'max_viol': sm['max_rel_violation']}
        for i, n in enumerate(names):
            row[f'X_{n}'] = sm['X'][i]; row[f'Pshort_{n}'] = sm['P_short'][i]; row[f'Eshort_{n}'] = sm['E_short_units'][i]
        for j in range(model.m):
            row[f'WHutil_{j+1}'] = sm['wh_util'][j]
        rows.append(row)
    return rows


def show(df, cols=None):
    cols = cols or (['group', 'value', 'lambda', 'total_stock'] + [f'X_{n}' for n in names] + ['f1', 'f2']
                    + [f'Pshort_{n}' for n in names] + [f'WHutil_{j+1}' for j in range(len(K_real))] + ['budget_util'])
    with pd.option_context('display.width', 260, 'display.max_columns', 40, 'display.float_format', '{:,.3f}'.format):
        print(df[[c for c in cols if c in df.columns]].to_string(index=False))

cost_rows = []
for key, mults in COST_MULTIPLIERS.items():
    for mlt in mults:
        base_val = COST_BASE[key]
        new = base_val * mlt if np.isscalar(base_val) else (np.array(base_val, dtype=float) * mlt).tolist()
        cost_rows += robust_rows(f'{key}x{mlt}', make_model(BASE_P, BASE_XI, BASE_SIG, **{key: new}), key, mlt)
cost_df = pd.DataFrame(cost_rows)
print("=" * 110 + "\n(1) COST SENSITIVITY (h, s, o, B multipliers)\n" + "=" * 110); show(cost_df)
cost_df.to_csv(f'{APP_DIR}/saved/robust_cost.csv', index=False)

alpha_rows = []
for a in ALPHA_GRID:
    alpha_rows += robust_rows(f'alpha={a}', make_model(BASE_P, BASE_XI, BASE_SIG, alpha=a), 'alpha', a)
alpha_df = pd.DataFrame(alpha_rows)
print("\n" + "=" * 110 + "\n(2) MINIMUM-SERVICE FLOOR  X_i >= alpha d_i  (economic-risk vs minimum-service allocation)\n" + "=" * 110)
show(alpha_df); alpha_df.to_csv(f'{APP_DIR}/saved/robust_alpha.csv', index=False)

u_rows = []
for uu in EVT_U_SENSITIVITY:
    fu, _ = fit_evt(uu, cutoff=AS_OF_DATE, se=True)
    rr = robust_rows(f'u={uu}', make_model(BASE_P, fu['xi'], fu['sigma'], u=uu), 'u', uu)
    for r in rr:
        r['xi'] = fu['xi']; r['se_xi'] = fu['se_xi']; r['n_exc_min'] = min(fu['n_exc'])
    u_rows += rr
u_df = pd.DataFrame(u_rows)
print("\n" + "=" * 110 + "\n(3) EVT THRESHOLD u (GPD refitted at each u; occurrence model p_i held fixed)\n" + "=" * 110)
show(u_df, ['value', 'xi', 'se_xi', 'n_exc_min', 'lambda', 'total_stock'] + [f'X_{n}' for n in names] + ['f1', 'f2'])
u_df.to_csv(f'{APP_DIR}/saved/robust_threshold.csv', index=False)

evt_boot_T = year_block_bootstrap(HEAVY_RAIN_THRESHOLD_MM, cutoff=AS_OF_DATE, B=EVT_PARAM_BOOT, rng=np.random.default_rng(EVT_SEED + 1))
boot_rows = []
for b, th in enumerate(evt_boot_T):
    for r in robust_rows(f'boot{b}', make_model(BASE_P, th[0], th[1:]), 'evt_boot', b):
        r['xi'] = th[0]; boot_rows.append(r)
boot_df = pd.DataFrame(boot_rows)
print("\n" + "=" * 110 + f"\n(4) EVT PARAMETER UNCERTAINTY: {len(evt_boot_T)} year-block bootstrap replicates, decisions re-optimised\n" + "=" * 110)
boot_df['lam_key'] = boot_df['lambda'].astype(str).str.replace(r'knee\(.*\)', 'knee', regex=True)
summ = []
for lk, g in boot_df.groupby('lam_key'):
    for col in ['total_stock'] + [f'X_{n}' for n in names] + ['f1', 'f2']:
        v = g[col].astype(float)
        summ.append({'lambda': lk, 'quantity': col, 'mean': v.mean(), 'sd': v.std(), 'p05': v.quantile(0.05), 'p95': v.quantile(0.95)})
evt_unc_df = pd.DataFrame(summ)
with pd.option_context('display.width', 200, 'display.float_format', '{:,.3f}'.format):
    print(evt_unc_df.to_string(index=False))
boot_df.to_csv(f'{APP_DIR}/saved/robust_evt_bootstrap.csv', index=False)

oat_rows = []
for key in ['p', 'd', 'xi', 'sigma', 'h', 'c', 's', 'o', 'K', 'B', 'pi']:
    for lv in [-0.2, -0.1, 0.1, 0.2]:
        kw = {}
        p_, xi_, sig_ = np.array(BASE_P), BASE_XI, BASE_SIG
        if key == 'p':
            p_ = np.clip(p_ * (1 + lv), 0, 1)
        elif key == 'xi':
            xi_ = BASE_XI * (1 + lv)
        elif key == 'sigma':
            sig_ = BASE_SIG * (1 + lv)
        else:
            base_val = COST_BASE[key]
            kw[key] = base_val * (1 + lv) if np.isscalar(base_val) else (np.array(base_val, dtype=float) * (1 + lv)).tolist()
        oat_rows += robust_rows(f'{key}{lv:+.0%}', make_model(p_, xi_, sig_, **kw), key, lv, with_knee=False)
oat_df = pd.DataFrame(oat_rows)
base_rows = pd.DataFrame(robust_rows('base', make_model(BASE_P, BASE_XI, BASE_SIG), 'base', 0.0, with_knee=False))
for lam in SENS_LAMBDAS:
    b = base_rows[base_rows['lambda'] == lam].iloc[0]; m_ = oat_df['lambda'] == lam
    oat_df.loc[m_, 'df1_%'] = 100 * (oat_df.loc[m_, 'f1'] - b['f1']) / max(b['f1'], 1)
    oat_df.loc[m_, 'df2_%'] = 100 * (oat_df.loc[m_, 'f2'] - b['f2']) / max(b['f2'], 1)
    oat_df.loc[m_, 'dstock_%'] = 100 * (oat_df.loc[m_, 'total_stock'] - b['total_stock']) / max(b['total_stock'], 1e-9)
print("\n" + "=" * 110 + "\n(5) ONE-AT-A-TIME +-10/20% (exact solver)\n" + "=" * 110)
show(oat_df, ['group', 'value', 'lambda', 'total_stock', 'dstock_%', 'f1', 'df1_%', 'f2', 'df2_%'])
oat_df.to_csv(f'{APP_DIR}/saved/robust_oat.csv', index=False)
print("\nAll robustness tables saved to saved/robust_*.csv")


def _cost_case(label, group, value, **over):
    mdl = make_model(BASE_P, BASE_XI, BASE_SIG, **over)
    lam_s = mdl.stocking_threshold_lambda()
    m_min = np.where(mdl.allowed_mask(), mdl.marginal_prep_cost(), np.inf).min(axis=1)
    rows = robust_rows(label, mdl, group, value, with_knee=False)
    for r in rows:
        r['h'] = float(mdl.h[0]); r['o'] = float(mdl.o[0])
        for i, n in enumerate(names):
            r[f'm_{n}'] = float(m_min[i]); r[f'lambda*_{n}'] = float(lam_s[i])
    return rows

ohs_rows = []
ohs_rows += _cost_case(f'baseline (shelf {SHELF_LIFE_DAYS} d)', 'baseline', SHELF_LIFE_DAYS)
for _sl in [90, 180, 365]:
    ohs_rows += _cost_case(f'shelf life {_sl} d', 'shelf_life_days', _sl, o=unused_cost_for_lead(LEAD_DAYS, _sl))
for _r in [0.02, 0.05, 0.10]:
    ohs_rows += _cost_case(f'rotation loss r={_r}', 'rotation_loss', _r, o=[FOOD_GR_RS_PER_FAMILY * 1000 * _r] * len(names))
ohs_rows += _cost_case('alternative: month holding, food written off', 'write_off_reference', 1.0,
                       o=[FOOD_GR_RS_PER_FAMILY * 1000] * len(names), h=[KIT_VALUE_RS * 1000 * HOLD_RATE_PER_MONTH] * len(K_real))
ohs_df = pd.DataFrame(ohs_rows)
print("\n" + "=" * 110 + f"\n(6) UNUSED-STOCK / HOLDING WINDOW SENSITIVITY (L={LEAD_DAYS}; plan X and f2 at lambda in {SENS_LAMBDAS})\n" + "=" * 110)
show(ohs_df, ['label', 'lambda', 'h', 'o'] + [f'm_{n}' for n in names] + [f'lambda*_{n}' for n in names]
     + [f'X_{n}' for n in names] + ['f1', 'f2'])
ohs_df.to_csv(f'{APP_DIR}/saved/robust_cost_window.csv', index=False)


## Step 10d: Scenario B (reactive), What-If export, frozen results JSON

In [ ]:
# Step 10d: reactive decision of 19 June 2022 with the observed alerts (p_i = 1) and the GPD refitted up to
# that day (Section 6.9; Table 13); export of the what-if inputs and of saved/final_results.json.

REACT_DATE = pd.Timestamp(REGION['reaction_date'])
p_react = []
for n in names:
    row = district_raw_base[n].loc[district_raw_base[n]['Date'] == REACT_DATE]
    p_react.append(float(row['Flash_Flood_Alert'].iloc[0]) if len(row) else 0.0)
print(f"SCENARIO B (reactive): confirmed same-day alert flags on {REACT_DATE.date()}: {dict(zip(names, p_react))}")
if sum(p_react) > 0:
    evt_R, _ = fit_evt(HEAVY_RAIN_THRESHOLD_MM, cutoff=REACT_DATE, se=False)
    model_react = make_model(p_react, evt_R['xi'], evt_R['sigma'])
    wsR, epsR, pfR = exact_frontier(model_react)
    kR = knee_point(pfR)
    print_allocation(model_react, kR['x'], f"Scenario B frontier point at lambda {kR['lambda']:.3f} (reference only; the paper reports the full-budget plan)")
    with open(f'{APP_DIR}/saved/decision_results_scenario_B_reactive.pkl', 'wb') as f:
        pickle.dump({'scenario_label': f"Scenario B: reactive (confirmed alert, {REACT_DATE.date()})", 'districts': names,
                     'pareto_frontier': [{'lambda': pt['lambda'], 'x': pt['x'], 'f1': pt['f1'], 'f2': pt['f2']} for pt in pfR],
                     'model_params': {'p': p_react, 'd': d_real, 'xi': evt_R['xi'], 'sigma': list(evt_R['sigma']),
                                      'u': HEAVY_RAIN_THRESHOLD_MM}}, f)

_pipe = pipelines[LEAD_DAYS]
_, _, _, clf_T, cal_T = forecast_as_of(_pipe, AS_OF_DATE, verbose=False)
whatif_rows = {}
for n in names:
    r_ = district_raw_base[n].loc[district_raw_base[n]['Date'] == AS_OF_DATE].iloc[0]
    whatif_rows[n] = {c: float(r_[c]) for c in FEAT_FLOOD}
with open(f'{APP_DIR}/saved/whatif_inputs.pkl', 'wb') as f:
    pickle.dump({'districts': names, 'feat_flood': FEAT_FLOOD, 'classifiers': {n: clf_T for n in names},
                 'calibrator': cal_T, 'calibration_method': CALIBRATION_METHOD,
                 'baseline_rows': whatif_rows, 'baseline_date': str(AS_OF_DATE.date()),
                 'p_baseline': p_real, 'd': d_real, 'xi': _resA['xi'], 'sigma': _resA['sigma'], 'u': HEAVY_RAIN_THRESHOLD_MM,
                 'h': h_real, 'c': c_real, 's': s_real, 'o': o_real, 'K': K_real, 'B': B_real, 'pi': pi_real,
                 'disallowed': DISALLOWED_ROUTES, 'min_service_frac': MIN_SERVICE_FRAC}, f)
print("Saved whatif_inputs.pkl")


def _j(o):
    if isinstance(o, (np.floating, np.integer)):
        return o.item()
    if isinstance(o, np.ndarray):
        return o.tolist()
    if isinstance(o, (pd.Timestamp,)):
        return str(o.date())
    return str(o)

FROZEN = {
    'demand_source': DEMAND_SOURCE, 'region': REGION['name'], 'decision_date': str(AS_OF_DATE.date()), 'primary_lead': LEAD_DAYS,
    'classifier': {'algo': tuned_clf_algo_name, 'params': list(np.round(tuned_clf_params, 6)),
                   'selection': f'purged expanding-window, blocks {TUNE_VAL_BLOCKS}, mean PR-AUC',
                   'calibration': CALIBRATION_METHOD,
                   'test_metrics': {L: pipelines[L]['test_metrics'].to_dict('records') for L in pipelines}},
    'evt': evt_summary,
    'scenario_A': {L: {'p_cal': r['p_real'], 'p_raw': r['p_raw'], 'audit': r['audit'], 'xi': r['xi'], 'sigma': r['sigma'],
                       'lambda_star': r['lambda_star'],
                       'knee': {'lambda': r['knee']['lambda'], 'X': r['knee']['x'].sum(axis=1).tolist(), 'f1': r['knee']['f1'], 'f2': r['knee']['f2']},
                       'frontier': [{'lambda': pt['lambda'], 'f1': pt['f1'], 'f2': pt['f2'], 'X': pt['x'].sum(axis=1).tolist()} for pt in r['pf']],
                       'benchmark': r['bench'], 'naive_vs_opt': r['case_rows']}
                   for L, r in scenario_A_results_by_lead.items()},
    'lead_summary': lead_df.to_dict('records'), 'trajectory': traj_df.to_dict('records'),
    'config': {'MIN_SERVICE_FRAC': MIN_SERVICE_FRAC, 'AT_RISK_BY_DISTRICT': AT_RISK_BY_DISTRICT, 'AT_RISK_SOURCE': AT_RISK_SOURCE, 'u': HEAVY_RAIN_THRESHOLD_MM,
               'TEST_START_YEAR': TEST_START_YEAR, 'SENS_LAMBDAS': SENS_LAMBDAS,
               'cost_window': {'HOLD_RATE_PER_MONTH': HOLD_RATE_PER_MONTH, 'SHELF_LIFE_DAYS': SHELF_LIFE_DAYS,
                               'h_by_lead': {L: holding_cost_for_lead(L)[0] for L in LEAD_DAYS_LIST},
                               'o_by_lead': {L: unused_cost_for_lead(L)[0] for L in LEAD_DAYS_LIST}},
               'demand_provenance': 'EX-POST: ASDMA 2022 memorandum, season-cumulative 6 Apr-28 Oct 2022'},
    'lambda_star_by_lead': lambda_star_by_lead_df.to_dict('records'),
    'cost_window_sensitivity': ohs_df.to_dict('records'),
}
with open(f'{APP_DIR}/saved/{RESULTS_JSON}', 'w') as f:
    json.dump(FROZEN, f, indent=1, default=_j)
print(f"FROZEN results -> {APP_DIR}/saved/{RESULTS_JSON}  (quote manuscript numbers from this file only)")


## Step 10e: Where EVT changes the decision, and what it is worth

Regime map (budget × at-risk fraction), value of EVT against exponential-tail, mean-demand and naive plans at the same money (scored under the fitted GPD, with bootstrap bands), and budget-share operating points.

In [ ]:
# Step 10e: where the severity layer changes the decision (Sections 6.7-6.8; Tables 9, 11, 12; Fig. 6):
# regime map, value against exponential, mean-demand and population-proportional plans with bootstrap
# bands, and budget-share operating points.

import time as _time

REGIME_B_MULT = [0.5, 0.75, 1.0, 1.25, 1.5, 2.0]
REGIME_AT_RISK = [0.1, 0.2, 0.3, 0.4, 0.5]
REGIME_LEADS = [l for l in [LEAD_DAYS, 1] if l in scenario_A_results_by_lead]
BUDGET_SHARES = [0.25, 0.5, 0.75, 1.0]
VALUE_BOOT = min(100, len(evt_boot_T)) if 'evt_boot_T' in globals() else 0

_census = np.array([CENSUS_2011_POPULATION[n] for n in names], dtype=float)
def d_for_fraction(frac):
    frac = np.broadcast_to(np.asarray(frac, dtype=float), (len(names),))
    return (_census / HOUSEHOLD_SIZE / DEMAND_SCALE * frac).tolist()

_exc_T = [get_exceedances(district_raw_base[n], HEAVY_RAIN_THRESHOLD_MM, cutoff=AS_OF_DATE) for n in names]
SIG_EXP = np.array([e.mean() for e in _exc_T])
_boot_T = evt_boot_T if 'evt_boot_T' in globals() else year_block_bootstrap(
    HEAVY_RAIN_THRESHOLD_MM, cutoff=AS_OF_DATE, B=EVT_PARAM_BOOT, rng=np.random.default_rng(EVT_SEED + 1))
XI_LO, XI_HI = np.percentile(_boot_T[:, 0], [5, 95])
SIG_LO, SIG_HI = np.percentile(_boot_T[:, 1:], 5, axis=0), np.percentile(_boot_T[:, 1:], 95, axis=0)
print(f"GPD (as of {AS_OF_DATE.date()}): xi={BASE_XI:.4f}, sigma={np.round(BASE_SIG, 3)} | bootstrap 5-95%: "
      f"xi [{XI_LO:.4f}, {XI_HI:.4f}] | EXP alternative sigma={np.round(SIG_EXP, 3)}")


def min_shortage_alloc(model):
    """Spend the available budget to minimise expected shortage cost f2, then (lexicographic)
    minimise f1 among f2-optimal plans. This is what an agency does with a fixed pot of money."""
    r = trace_pareto_exact(model, [0.0])[0]
    x = r.get('x')
    q = solve_slsqp(model, 0.0, n_starts=4)
    if x is None or (q.get('x') is not None and model.f2(q['x']) < model.f2(x) * (1 - 1e-6) - 1.0):
        x = q['x']
    return x


def mean_demand_alloc(model):
    """Mean-demand planning: under a hazard demand is exactly d_i. f2_MD = sum p_i s_i (d_i - X_i)^+.
    Solved exactly (LP): min f2_MD, then min f1 at that f2_MD."""
    n, m = model.n, model.m
    def _solve(stage, f2cap=None):
        x = cp.Variable((n, m), nonneg=True); X = cp.sum(x, axis=1); S = model.obj_scale()
        f2md = cp.sum(cp.multiply(model.p * model.s / S, cp.pos(model.d - X)))
        f1 = cp.sum(cp.multiply(model.marginal_prep_cost() / S, x))
        cons = [cp.sum(x, axis=0) <= model.K, cp.sum(cp.multiply(model.pi / model.B, x)) <= 1.0]
        if model.min_service_frac > 0:
            cons.append(X >= model.min_service_frac * model.d)
        for i, j in model.disallowed:
            cons.append(x[i, j] == 0)
        if stage == 1:
            cp.Problem(cp.Minimize(f2md), cons).solve()
        else:
            cons.append(f2md <= f2cap * (1 + 1e-7) + 1e-9)
            cp.Problem(cp.Minimize(f1), cons).solve()
        return _clean_solution(model, np.array(x.value)), f2md.value
    _, f2star = _solve(1)
    return _solve(2, f2star)[0]


def tail_sensitivity(p, frac, bmult, lead=None):
    """Max relative change of district stock when (xi, sigma) move to their bootstrap 5th/95th pct."""
    base = min_shortage_alloc(make_model(p, BASE_XI, BASE_SIG, d=d_for_fraction(frac), B=B_real * bmult, lead=lead)).sum(axis=1)
    worst = 0.0
    for xi_, sg_ in [(XI_LO, SIG_LO), (XI_HI, SIG_HI)]:
        alt = min_shortage_alloc(make_model(p, xi_, sg_, d=d_for_fraction(frac), B=B_real * bmult, lead=lead)).sum(axis=1)
        worst = max(worst, float(np.abs(alt - base).sum() / max(base.sum(), 1e-9)))
    return base, worst


def true_costs(x, p, frac, bmult, draws=None, lead=None):
    """(f1, f2) of plan x under the fitted GPD; plus f2 across bootstrap GPD replicates."""
    m0 = make_model(p, BASE_XI, BASE_SIG, d=d_for_fraction(frac), B=B_real * bmult, lead=lead)
    f2b = None
    if draws is not None and len(draws):
        f2b = np.array([make_model(p, th[0], th[1:], d=d_for_fraction(frac), B=B_real * bmult, lead=lead).f2(x) for th in draws])
    return m0.f1(x), m0.f2(x), f2b


t0 = _time.time()
regime_rows = []
for L in REGIME_LEADS:
    pL = scenario_A_results_by_lead[L]['p_real']
    for frac in REGIME_AT_RISK:
        for bm in REGIME_B_MULT:
            mdl = make_model(pL, BASE_XI, BASE_SIG, d=d_for_fraction(frac), B=B_real * bm, lead=L)
            X, sens = tail_sensitivity(pL, frac, bm, lead=L)
            active = X > mdl.u_prime + 1e-6
            regime_rows.append({'L': L, 'at_risk': frac, 'B_mult': bm, 'total_stock': X.sum(),
                                **{f'X_{n}': X[i] for i, n in enumerate(names)},
                                **{f'u_{n}': mdl.u_prime[i] for i, n in enumerate(names)},
                                'n_tail_active': int(active.sum()), 'tail_sensitivity_%': 100 * sens,
                                'P_short_min': float(mdl.survival(X).min())})
regime_df = pd.DataFrame(regime_rows)
print(f"\n(A) REGIME MAP  [{_time.time()-t0:.0f}s]  -- min-shortage use of the budget; tail-active = X_i > u'_i;"
      " tail sensitivity = % change of the allocation when (xi, sigma) move to bootstrap 5th/95th pct")
for L in REGIME_LEADS:
    g = regime_df[regime_df['L'] == L]
    print(f"\nL={L}: number of districts stocked beyond u'_i (0-3)")
    print(g.pivot(index='at_risk', columns='B_mult', values='n_tail_active').to_string())
    print(f"L={L}: tail sensitivity of the decision (%)")
    print(g.pivot(index='at_risk', columns='B_mult', values='tail_sensitivity_%').round(2).to_string())
regime_df.to_csv(f'{APP_DIR}/saved/evt_regime_map.csv', index=False)

from matplotlib.patches import Rectangle
_vmax = max(1e-9, regime_df['tail_sensitivity_%'].max())
fig, axes = plt.subplots(1, len(REGIME_LEADS), figsize=(3.7 * len(REGIME_LEADS), 2.9), squeeze=False, sharey=True)
for ax, L in zip(axes[0], REGIME_LEADS):
    g = regime_df[regime_df['L'] == L].pivot(index='at_risk', columns='B_mult', values='tail_sensitivity_%')
    na = regime_df[regime_df['L'] == L].pivot(index='at_risk', columns='B_mult', values='n_tail_active').values
    im = ax.imshow(g.values, cmap='Blues', origin='lower', aspect='auto', vmin=0, vmax=_vmax)
    for r_ in range(g.shape[0]):
        for c_ in range(g.shape[1]):
            ax.text(c_, r_, int(na[r_, c_]), ha='center', va='center', fontsize=8,
                    color='white' if g.values[r_, c_] > 0.6 * _vmax else '#0b0b0b')
    ax.set_xticks(range(len(g.columns))); ax.set_xticklabels(['$B$' if b == 1 else f'{b:g}$B$' for b in g.columns])
    ax.set_yticks(range(len(g.index))); ax.set_yticklabels([f'{a:g}' for a in g.index])
    jb = list(g.columns).index(1.0)
    ax.add_patch(Rectangle((jb - 0.5, -0.5), 1, len(g.index), fill=False, ec='#0b0b0b', lw=1.4, ls='--'))
    ax.set_title(f'$L={L}$'); ax.set_xlabel('budget')
    for sp in ax.spines.values(): sp.set_visible(False)
axes[0][0].set_ylabel('uniform at-risk fraction')
cb = fig.colorbar(im, ax=list(axes[0]), shrink=0.85, pad=0.02); cb.set_label('tail sensitivity of the plan (%)')
fig.savefig(f'{APP_DIR}/evt_regime_map.png'); plt.show()

t0 = _time.time()
value_rows = []
draws = _boot_T[:VALUE_BOOT] if VALUE_BOOT else None
for L in REGIME_LEADS:
    pL = scenario_A_results_by_lead[L]['p_real']
    for frac in REGIME_AT_RISK:
        for bm in REGIME_B_MULT:
            dd = d_for_fraction(frac); BB = B_real * bm
            m_gpd = make_model(pL, BASE_XI, BASE_SIG, d=dd, B=BB, lead=L)
            plans = {'GPD': min_shortage_alloc(m_gpd),
                     'EXP': min_shortage_alloc(make_model(pL, 0.0, SIG_EXP, d=dd, B=BB, lead=L)),
                     'MEAN': mean_demand_alloc(m_gpd)}
            spend = float(np.sum(m_gpd.pi * plans['GPD']))
            plans['NAIVE'] = naive_proportional_allocation(m_gpd, spend)
            res = {k: true_costs(v, pL, frac, bm, draws, lead=L) for k, v in plans.items()}
            f2g = res['GPD'][1]
            row = {'L': L, 'at_risk': frac, 'B_mult': bm, 'tail_active': int((plans['GPD'].sum(axis=1) > m_gpd.u_prime + 1e-6).sum()),
                   'f2_GPD': f2g}
            for k in ['EXP', 'MEAN', 'NAIVE']:
                row[f'f2_{k}'] = res[k][1]
                row[f'value_vs_{k}_%'] = 100 * (res[k][1] - f2g) / max(f2g, 1.0)
                row[f'value_vs_{k}_Rs'] = res[k][1] - f2g
                if draws is not None:
                    dv = res[k][2] - res['GPD'][2]
                    row[f'value_vs_{k}_Rs_p05'], row[f'value_vs_{k}_Rs_p95'] = np.percentile(dv, [5, 95])
                    tol = 1e-6 * max(f2g, 1.0)
                    row[f'P(GPD better than {k})'] = float(np.mean(dv > tol)) if np.any(np.abs(dv) > tol) else np.nan
                row[f'X_{k}'] = np.round(plans[k].sum(axis=1), 2).tolist()
            row['X_GPD'] = np.round(plans['GPD'].sum(axis=1), 2).tolist()
            value_rows.append(row)
value_df = pd.DataFrame(value_rows)
print(f"\n(B) VALUE OF EVT  [{_time.time()-t0:.0f}s]  -- same money; every plan scored under the fitted GPD; "
      f"bootstrap bands over {VALUE_BOOT} replicates")
show_cols = ['at_risk', 'B_mult', 'tail_active', 'f2_GPD', 'value_vs_EXP_%', 'value_vs_MEAN_%', 'value_vs_NAIVE_%']
if draws is not None:
    show_cols += ['P(GPD better than EXP)', 'P(GPD better than MEAN)']
for L in REGIME_LEADS:
    print(f"\nL={L}:")
    with pd.option_context('display.width', 250, 'display.float_format', '{:,.3f}'.format):
        print(value_df[value_df['L'] == L][show_cols].to_string(index=False))
value_df.to_csv(f'{APP_DIR}/saved/evt_value.csv', index=False)

fig, axes = plt.subplots(1, len(REGIME_LEADS), figsize=(3.7 * len(REGIME_LEADS), 2.9), squeeze=False, sharey=True)
_colors = {'EXP': '#2a78d6', 'MEAN': '#eb6834', 'NAIVE': '#1baf7a'}
_vs = {'EXP': r'exponential tail ($\xi=0$)', 'MEAN': 'mean demand ($D_i=d_i$)', 'NAIVE': 'population-proportional'}
for ax, L in zip(axes[0], REGIME_LEADS):
    g = value_df[(value_df['L'] == L) & (value_df['B_mult'] == 1.0)].sort_values('at_risk')
    for k in ['EXP', 'MEAN', 'NAIVE']:
        ax.plot(g['at_risk'], g[f'value_vs_{k}_%'], marker='o', ms=4, lw=1.5, color=_colors[k], label=f'vs {_vs[k]}')
        if draws is not None:
            base = g['f2_GPD'].clip(lower=1.0)
            ax.fill_between(g['at_risk'], 100 * g[f'value_vs_{k}_Rs_p05'] / base, 100 * g[f'value_vs_{k}_Rs_p95'] / base,
                            color=_colors[k], alpha=0.12, lw=0)
    ax.axhline(0, color='#888', lw=0.8)
    ax.set_xlabel('uniform at-risk fraction'); ax.set_title(f'$L={L}$, budget $B$'); light_grid(ax)
    ax.legend()
axes[0][0].set_ylabel('extra $f_2$ of the alternative plan\nover the GPD plan (%)')
fig.tight_layout(); fig.savefig(f'{APP_DIR}/evt_value.png'); plt.show()

emp_rows = []
if True:
    frac_emp = np.array([AT_RISK_BY_DISTRICT[n] for n in names])
    print(f"\nBASELINE at-risk fractions [{AT_RISK_SOURCE}]: {dict(zip(names, np.round(frac_emp, 3)))}")
    for L in REGIME_LEADS:
        pL = scenario_A_results_by_lead[L]['p_real']
        m_e = make_model(pL, BASE_XI, BASE_SIG, d=d_for_fraction(frac_emp), lead=L)
        xg = min_shortage_alloc(m_e); xe = min_shortage_alloc(make_model(pL, 0.0, SIG_EXP, d=d_for_fraction(frac_emp), lead=L))
        xm = mean_demand_alloc(m_e); xn = naive_proportional_allocation(m_e, float(np.sum(m_e.pi * xg)))
        print(f"  L={L}: u'_i={np.round(m_e.u_prime,1)}  X_GPD={np.round(xg.sum(1),1)}  tail-active={int((xg.sum(1)>m_e.u_prime+1e-6).sum())}/3")
        for k, xx in [('EXP', xe), ('MEAN', xm), ('NAIVE', xn)]:
            line = (f"     vs {k}: X={np.round(xx.sum(1),1)}  f2 {m_e.f2(xx):,.0f} vs GPD {m_e.f2(xg):,.0f}  "
                    f"({100*(m_e.f2(xx)-m_e.f2(xg))/max(m_e.f2(xg),1):+.2f}%, Rs{m_e.f2(xx)-m_e.f2(xg):,.0f})")
            if draws is not None and len(draws):
                dv = np.array([make_model(pL, th[0], th[1:], d=d_for_fraction(frac_emp), lead=L).f2(xx)
                               - make_model(pL, th[0], th[1:], d=d_for_fraction(frac_emp), lead=L).f2(xg) for th in draws])
                base_b = np.array([make_model(pL, th[0], th[1:], d=d_for_fraction(frac_emp), lead=L).f2(xg) for th in draws])
                pct = 100 * dv / np.maximum(base_b, 1.0); tol = 1e-6
                line += (f"  | bootstrap 5-95%: [{np.percentile(pct,5):+.2f}%, {np.percentile(pct,95):+.2f}%], "
                         f"P(GPD better)={np.mean(dv > tol*base_b):.2f}")
                emp_rows.append({'L': L, 'alt': k, 'value_pct': 100*(m_e.f2(xx)-m_e.f2(xg))/max(m_e.f2(xg),1),
                                 'value_Rs': m_e.f2(xx)-m_e.f2(xg), 'pct_p05': np.percentile(pct,5),
                                 'pct_p95': np.percentile(pct,95), 'P_GPD_better': float(np.mean(dv > tol*base_b)),
                                 'X_alt': xx.sum(1).tolist(), 'X_GPD': xg.sum(1).tolist(), 'u_prime': m_e.u_prime.tolist()})
            print(line)
else:
    print("\nAFFECTED_POP_2022 not filled in -- the at-risk fraction remains an assumption; report the full map.")

op_rows = []
for L in REGIME_LEADS:
    pL = scenario_A_results_by_lead[L]['p_real']
    _fr_list = [('empirical', np.array([AT_RISK_BY_DISTRICT[n] for n in names]))] + [(f, f) for f in (0.2, 0.5)]
    for lab_, frac in _fr_list:
        for sh in BUDGET_SHARES:
            mdl = make_model(pL, BASE_XI, BASE_SIG, d=d_for_fraction(frac), B=B_real * sh, lead=L)
            x = min_shortage_alloc(mdl); sm = summarize_solution(mdl, x)
            op_rows.append({'L': L, 'at_risk': lab_, 'budget_share': sh, 'total_stock': sm['total_stock'],
                            **{f'X_{n}': sm['X'][i] for i, n in enumerate(names)}, 'f1': sm['f1'], 'f2': sm['f2'],
                            **{f'Pshort_{n}': sm['P_short'][i] for i, n in enumerate(names)}})
op_df = pd.DataFrame(op_rows)
print("\n(C) BUDGET-SHARE OPERATING POINTS (min-shortage use of 25/50/75/100% of B)")
with pd.option_context('display.width', 250, 'display.float_format', '{:,.3f}'.format):
    print(op_df.to_string(index=False))
op_df.to_csv(f'{APP_DIR}/saved/operating_points.csv', index=False)

try:
    with open(f'{APP_DIR}/saved/{RESULTS_JSON}') as f:
        _fz = json.load(f)
    _fz['evt_regime'] = {'regime_map': regime_df.to_dict('records'), 'value_of_evt': value_df.to_dict('records'),
                         'empirical_2022': emp_rows, 'baseline_row_source': AT_RISK_SOURCE, 'at_risk_by_district': AT_RISK_BY_DISTRICT,
                         'operating_points': op_df.to_dict('records'),
                         'xi_boot_5_95': [float(XI_LO), float(XI_HI)], 'sigma_exp': SIG_EXP.tolist()}
    with open(f'{APP_DIR}/saved/{RESULTS_JSON}', 'w') as f:
        json.dump(_fz, f, indent=1, default=_j)
    print(f"Appended Step 10e to {RESULTS_JSON}")
except Exception as e:
    print(f"Could not append to {RESULTS_JSON}:", e)


## Step 10f: Sensitivity to the demand scale $d_i$

$d_i$ is derived from the flood-prone share of each district in the NRSC/ISRO Flood Hazard Atlas, which is used as a proxy for the share of households at risk. This step scales $d_i$ by 0.5-1.5x and re-solves the $\lambda=0$ (lexicographic) plan, as in Step 10e (paper: Table 18 reports 0.7-1.5x). Placed after 10e because it uses `min_shortage_alloc`, `mean_demand_alloc` and `true_costs`.

In [ ]:
# Step 10f: sensitivity to the demand scale d_i (Section 7.4; Table 18).

D_MULTS = [0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.1, 1.2, 1.3, 1.4, 1.5]
_frac_emp = np.array([AT_RISK_BY_DISTRICT[n] for n in names])


def mean_demand_tie_band(model, n_random=24, seed=0):
    """The mean-demand LP is indifferent between districts with equal p_i s_i / pi_i (it sees no
    spread), so it can have many optimal plans; mean_demand_alloc breaks the tie by f1.
    Returns (min, max) of the TRUE expected shortage cost f2 over that whole optimal set:
    the max is attained at a vertex (convex f2), found with linear objectives in many directions;
    the min is solved exactly as a convex conic problem."""
    n, m = model.n, model.m; S = model.obj_scale()
    def _f2md(xv):
        return cp.sum(cp.multiply(model.p * model.s / S, cp.pos(model.d - cp.sum(xv, axis=1))))
    def _cons(xv):
        c = [cp.sum(xv, axis=0) <= model.K, cp.sum(cp.multiply(model.pi / model.B, xv)) <= 1.0]
        if model.min_service_frac > 0:
            c.append(cp.sum(xv, axis=1) >= model.min_service_frac * model.d)
        return c + [xv[i, j] == 0 for i, j in model.disallowed]
    xv = cp.Variable((n, m), nonneg=True)
    cp.Problem(cp.Minimize(_f2md(xv)), _cons(xv)).solve()
    cap = float(_f2md(xv).value) * (1 + 1e-7) + 1e-9
    rng = np.random.default_rng(seed)
    dirs = [s_ * np.eye(n * m)[k] for k in range(n * m) for s_ in (1.0, -1.0)] + list(rng.normal(size=(n_random, n * m)))
    vals = []
    for w in dirs:
        xv = cp.Variable((n, m), nonneg=True)
        cp.Problem(cp.Minimize(cp.sum(cp.multiply(w.reshape(n, m), xv))), _cons(xv) + [_f2md(xv) <= cap]).solve()
        if xv.value is not None:
            vals.append(model.f2(_clean_solution(model, np.array(xv.value), zero_tol=0.0)))
    try:
        prob, xc, _ = _cvx_problem(model, lam=0.0)
        p2 = cp.Problem(prob.objective, prob.constraints + [_f2md(xc) <= cap])
        try:
            p2.solve(solver='CLARABEL')
        except Exception:
            p2.solve()
        if xc.value is not None:
            vals.append(model.f2(_clean_solution(model, np.array(xc.value), zero_tol=0.0)))
    except Exception as e:
        print('  (tie band: exact minimum skipped:', type(e).__name__, ')')
    return (min(vals), max(vals)) if vals else (np.nan, np.nan)


dsens_rows = []
for mult in D_MULTS:
    d_m = [mult * di for di in d_real]
    model_m = make_model(BASE_P, BASE_XI, BASE_SIG, d=d_m)
    x = min_shortage_alloc(model_m)
    xm = mean_demand_alloc(model_m)
    _, f2_gpd, _ = true_costs(x, BASE_P, _frac_emp * mult, 1.0)
    _, f2_mean, _ = true_costs(xm, BASE_P, _frac_emp * mult, 1.0)
    band_lo, band_hi = mean_demand_tie_band(model_m)
    X = x.sum(axis=1); Y = x.sum(axis=0); Xm = xm.sum(axis=1)
    f2_0 = model_m.f2(np.zeros_like(x)); lam_s = model_m.stocking_threshold_lambda(); Ps = model_m.survival(X)
    dsens_rows.append({
        'd_mult': mult,
        **{f'd_{n}': model_m.d[i] for i, n in enumerate(names)},
        **{f'u_{n}': model_m.u_prime[i] for i, n in enumerate(names)},
        **{f'X_{n}': X[i] for i, n in enumerate(names)},
        **{f'Y_WH{j+1}': Y[j] for j in range(model_m.m)},
        'f1': model_m.f1(x), 'f2': f2_gpd, 'f2_no_stock': f2_0, 'f2_cut_%': 100 * (1 - f2_gpd / f2_0),
        'budget_util': float(np.sum(model_m.pi * x) / model_m.B),
        'n_tail_active': int((X > model_m.u_prime + 1e-6).sum()),
        **{f'Pshort_{n}': Ps[i] for i, n in enumerate(names)},
        **{f'lambda*_{n}': lam_s[i] for i, n in enumerate(names)},
        **{f'Xmean_{n}': Xm[i] for i, n in enumerate(names)},
        'value_vs_MEAN_%': 100 * (f2_mean / f2_gpd - 1),
        'value_vs_MEAN_tieband_lo_%': 100 * (band_lo / f2_gpd - 1),
        'value_vs_MEAN_tieband_hi_%': 100 * (band_hi / f2_gpd - 1)})
dsens_df = pd.DataFrame(dsens_rows)
print("=" * 110 + f"\nSTEP 10f: DEMAND-SCALE SENSITIVITY (d_i x mult; L={LEAD_DAYS}; lambda = 0 lexicographic; GPD as of {AS_OF_DATE.date()})\n" + "=" * 110)
_cols = (['d_mult'] + [f'u_{n}' for n in names] + [f'X_{n}' for n in names] + [f'Y_WH{j+1}' for j in range(len(K_real))] + ['f1', 'f2', 'f2_no_stock',
         'f2_cut_%', 'n_tail_active'] + [f'Pshort_{n}' for n in names] + [f'lambda*_{n}' for n in names]
         + ['value_vs_MEAN_%', 'value_vs_MEAN_tieband_lo_%', 'value_vs_MEAN_tieband_hi_%'])
with pd.option_context('display.width', 300, 'display.max_columns', 60, 'display.float_format', '{:,.3f}'.format):
    print(dsens_df[_cols].to_string(index=False))
print("n_tail_active counts X_i > u'_i + 1e-6; a district just above u'_i with Pshort ~0.996 is held there only by a "
      "4th-decimal difference in p_i (see Pshort columns).\n"
      "value_vs_MEAN_tieband: range of the mean-demand plan's extra f2 over ALL its tied optimal plans "
      "(the tie-broken value value_vs_MEAN_% lies inside it).")
dsens_df.to_csv(f'{APP_DIR}/saved/d_sensitivity.csv', index=False)
try:
    with open(f'{APP_DIR}/saved/{RESULTS_JSON}') as f:
        _fz = json.load(f)
    _fz['d_sensitivity'] = dsens_df.to_dict('records')
    with open(f'{APP_DIR}/saved/{RESULTS_JSON}', 'w') as f:
        json.dump(_fz, f, indent=1, default=_j)
    print(f"Appended 'd_sensitivity' to {RESULTS_JSON}")
except Exception as e:
    print(f"Could not append to {RESULTS_JSON}:", e)


## Step 10g: The demand map is an assumption -- sensitivity analysis

The map from rainfall excess to households in need, $D=d\,(u+E)/(u+\bar E)$, is **assumed**. Two families of alternatives with the same mean demand are tested: an elasticity family $D=d\,(a+E)/(a+\bar E)$, $a=u/\beta$ (exact closed form), and a non-linear power family $D=d\,g(E)/\mathrm{E}[g(E)]$, $g=[(u+E)/(u+\bar E)]^{\gamma}$ (numerical integration). For each, the step reports the cost of having planned with the baseline map when the alternative is true (regret), and whether the paper's plan still beats mean-demand planning.

In [ ]:
# Step 10g: alternative demand maps (Section 7.4; Table 19): elasticity family (closed form) and power
# family (numerical integration); regret of the baseline plan and value over mean-demand planning.

from scipy.integrate import quad as _quad
from scipy.stats import genpareto as _gpd

DEMAND_BETAS = [0.25, 0.5, 1.0, 2.0, 4.0, 8.0]
DEMAND_GAMMAS = [0.5, 1.0, 2.0, 3.0]

base_model_g = make_model(BASE_P, BASE_XI, BASE_SIG)
X_BASE_PLAN = min_shortage_alloc(base_model_g)
X_MEAN_PLAN = mean_demand_alloc(base_model_g)


class PowerDemandModel(PrepositioningModel):
    """Same decision model with the power demand map D_i = d_i g(E_i)/E[g(E_i)],
    g(E) = ((u+E)/(u+Ebar_i))**gamma. Shortage and its gradient by numerical integration:
    E[(D-X)^+] = (Dmin - X)^+ + integral_{max(X,Dmin)}^inf P(D > t) dt."""

    def __init__(self, *args, gamma=1.0, **kw):
        super().__init__(*args, **kw)
        self.gamma = float(gamma)
        self.Mg = np.array([_quad(lambda e, i=i: ((self.u + e) / (self.u + self.E_bar[i])) ** self.gamma
                                  * _gpd.pdf(e, self.xi[i], scale=self.sigma[i]), 0, np.inf, limit=200)[0]
                            for i in range(self.n)])
        self.Dmin = self.d * (self.u / (self.u + self.E_bar)) ** self.gamma / self.Mg

    def _sf(self, i, t):
        if t <= self.Dmin[i]:
            return 1.0
        e = (t * self.Mg[i] / self.d[i]) ** (1 / self.gamma) * (self.u + self.E_bar[i]) - self.u
        return float(_gpd.sf(e, self.xi[i], scale=self.sigma[i]))

    def survival(self, X):
        return np.array([self._sf(i, v) for i, v in enumerate(np.asarray(X, dtype=float))])

    def expected_shortage_units(self, X):
        out = np.zeros(self.n)
        for i, v in enumerate(np.asarray(X, dtype=float)):
            lo = max(v, self.Dmin[i])
            out[i] = _quad(lambda t: self._sf(i, t), lo, np.inf, limit=200)[0] + max(self.Dmin[i] - v, 0.0)
        return out


def power_model(gamma):
    m0 = base_model_g
    return PowerDemandModel(p=m0.p, d=m0.d, xi=m0.xi, sigma=m0.sigma, u=m0.u, h=m0.h, c=m0.c, s=m0.s, o=m0.o,
                            K=m0.K, B=m0.B, pi=m0.pi, min_service_frac=m0.min_service_frac,
                            disallowed=m0.disallowed, gamma=gamma)


dmap_rows = []
for beta in DEMAND_BETAS:
    mb = make_model(BASE_P, BASE_XI, BASE_SIG, u=HEAVY_RAIN_THRESHOLD_MM / beta)
    xb = min_shortage_alloc(mb)
    f_opt, f_base, f_mean = mb.f2(xb), mb.f2(X_BASE_PLAN), mb.f2(X_MEAN_PLAN)
    Xb = xb.sum(axis=1)
    dmap_rows.append({'family': 'elasticity', 'parameter': beta, 'baseline_map': beta == 1.0,
                      **{f'certain_share_{n}': mb.u_prime[i] / mb.d[i] for i, n in enumerate(names)},
                      **{f'u_{n}': mb.u_prime[i] for i, n in enumerate(names)},
                      **{f'X_opt_{n}': Xb[i] for i, n in enumerate(names)},
                      'n_tail_active': int((Xb > mb.u_prime + 1e-6).sum()),
                      'f2_opt': f_opt, 'f2_baseline_plan': f_base, 'f2_mean_plan': f_mean,
                      'regret_baseline_plan_%': 100 * (f_base / f_opt - 1),
                      'value_vs_MEAN_%': 100 * (f_mean / f_base - 1)})

for gamma in DEMAND_GAMMAS:
    t0 = _time.time()
    mg = power_model(gamma)
    r = solve_slsqp(mg, 0.0, n_starts=6)
    xg = r['x'] if r.get('x') is not None else X_BASE_PLAN
    if mg.f2(X_BASE_PLAN) < mg.f2(xg):
        xg = X_BASE_PLAN
    f_opt, f_base, f_mean = mg.f2(xg), mg.f2(X_BASE_PLAN), mg.f2(X_MEAN_PLAN)
    Xg = xg.sum(axis=1)
    dmap_rows.append({'family': 'power', 'parameter': gamma, 'baseline_map': gamma == 1.0,
                      **{f'certain_share_{n}': mg.Dmin[i] / mg.d[i] for i, n in enumerate(names)},
                      **{f'u_{n}': mg.Dmin[i] for i, n in enumerate(names)},
                      **{f'X_opt_{n}': Xg[i] for i, n in enumerate(names)},
                      'n_tail_active': int((Xg > mg.Dmin + 1e-6).sum()),
                      'f2_opt': f_opt, 'f2_baseline_plan': f_base, 'f2_mean_plan': f_mean,
                      'regret_baseline_plan_%': 100 * (f_base / f_opt - 1),
                      'value_vs_MEAN_%': 100 * (f_mean / f_base - 1)})
    print(f"  power map gamma={gamma}: solved in {_time.time()-t0:.0f}s")

dmap_df = pd.DataFrame(dmap_rows)
_chk = dmap_df[(dmap_df.family == 'power') & (dmap_df.parameter == 1.0)]
if len(_chk):
    print(f"Check: power map with gamma = 1 reproduces the closed form: f2 = {_chk['f2_baseline_plan'].iloc[0]:,.0f} "
          f"vs {base_model_g.f2(X_BASE_PLAN):,.0f}")
print("=" * 110 + "\nSTEP 10g: DEMAND-MAP SENSITIVITY (lambda = 0; same budget; GPD as of "
      f"{AS_OF_DATE.date()}; L={LEAD_DAYS})\n" + "=" * 110)
_cols = (['family', 'parameter'] + [f'certain_share_{n}' for n in names] + [f'X_opt_{n}' for n in names]
         + ['n_tail_active', 'f2_opt', 'regret_baseline_plan_%', 'value_vs_MEAN_%'])
with pd.option_context('display.width', 300, 'display.max_columns', 40, 'display.float_format', '{:,.3f}'.format):
    print(dmap_df[_cols].to_string(index=False))
print("regret_baseline_plan_% = extra f2 of the paper's plan if the alternative map is true;\n"
      "value_vs_MEAN_% = extra f2 of mean-demand planning over the paper's plan under that map "
      "(positive = the paper's plan is still better).")
dmap_df.to_csv(f'{APP_DIR}/saved/demand_map_sensitivity.csv', index=False)
try:
    with open(f'{APP_DIR}/saved/{RESULTS_JSON}') as f:
        _fz = json.load(f)
    _fz['demand_map_sensitivity'] = dmap_df.to_dict('records')
    with open(f'{APP_DIR}/saved/{RESULTS_JSON}', 'w') as f:
        json.dump(_fz, f, indent=1, default=_j)
    print(f"Appended 'demand_map_sensitivity' to {RESULTS_JSON}")
except Exception as e:
    print(f"Could not append to {RESULTS_JSON}:", e)


## Step 10h: When does the extreme-value layer change the decision?

Same network, costs and mean demand as the case study; the **true** tail shape $\xi$ is varied with the mean excess held fixed. (A) Stock level with a non-binding budget, total expected cost ($\lambda=0.5$), varying the critical fractile through the alert probability and the shortage penalty. (B) Split of a binding budget when districts have different tail shapes. Explains why GPD and exponential plans coincide in 2022 and shows where they diverge.

In [ ]:
# Step 10h: controlled experiment on the tail shape (Section 7.5; Table 20; Fig. 8). The mean excess is held
# fixed and the true xi is varied; GPD, exponential and mean-demand planners are compared.

SYNTH_XI = [0.0, 0.1, 0.2, 0.3, 0.4]
SYNTH_S_MULT = [1, 5, 20]
SYNTH_B_MULT = 50
SYNTH_LAMBDA = 0.5
SYNTH_HET_XI = [(0.1, 0.1, 0.1), (0.0, 0.4, 0.4), (0.4, 0.0, 0.0), (0.4, 0.1, 0.1), (0.1, 0.3, 0.0)]
SYNTH_HET_BMULT = [1, 2, 3]
EBAR_T = np.array(BASE_SIG) / (1 - BASE_XI)

_p_cases = [(f'L={LEAD_DAYS} forecast', list(BASE_P), LEAD_DAYS)]
if 1 in scenario_A_results_by_lead and LEAD_DAYS != 1:
    _p_cases.append(('L=1 forecast', list(scenario_A_results_by_lead[1]['p_real']), 1))
_p_cases.append(('confirmed alert (p=1)', [1.0] * len(names), 1))


def mean_demand_alloc_lam(model, lam):
    """Mean-demand planner minimising lam f1 + (1-lam) sum p s (d - X)^+ (LP)."""
    if lam == 0:
        return mean_demand_alloc(model)
    n, m = model.n, model.m; S = model.obj_scale()
    x = cp.Variable((n, m), nonneg=True); X = cp.sum(x, axis=1)
    f2md = cp.sum(cp.multiply(model.p * model.s / S, cp.pos(model.d - X)))
    f1 = cp.sum(cp.multiply(model.marginal_prep_cost() / S, x))
    cons = [cp.sum(x, axis=0) <= model.K, cp.sum(cp.multiply(model.pi / model.B, x)) <= 1.0]
    cons += [x[i, j] == 0 for i, j in model.disallowed]
    cp.Problem(cp.Minimize(lam * f1 + (1 - lam) * f2md), cons).solve()
    return _clean_solution(model, np.array(x.value))


def _plan(model, lam):
    """Exact conic solve, cross-checked with SQP; the better plan (lower F_lambda) is kept."""
    if lam == 0:
        return min_shortage_alloc(model)
    Xcf, j_star, _ = model.critical_fractile_solution(lam)
    if np.all(np.isfinite(Xcf)):
        x = np.zeros((model.n, model.m)); x[np.arange(model.n), j_star] = Xcf
        if model.feasibility_report(x)['max_rel_violation'] < 1e-9:
            return x
    cands = []
    for solve in (lambda: solve_exact(model, lam=lam), lambda: solve_slsqp(model, lam, n_starts=4)):
        try:
            r = solve()
            if r.get('x') is not None and model.feasibility_report(r['x'])['max_rel_violation'] < 1e-6:
                cands.append(r['x'])
        except Exception:
            pass
    return min(cands, key=lambda x: lam * model.f1(x) + (1 - lam) * model.f2(x))


t0 = _time.time()
synthA_rows = []
for pname, pv, L in _p_cases:
    for sm in SYNTH_S_MULT:
        for xi in SYNTH_XI:
            over = dict(B=B_real * SYNTH_B_MULT, K=[k * SYNTH_B_MULT for k in K_real], s=[v * sm for v in s_real])
            mt = make_model(pv, xi, EBAR_T * (1 - xi), lead=L, **over)
            me = make_model(pv, 0.0, EBAR_T, lead=L, **over)
            lam = SYNTH_LAMBDA
            xg, xe, xm = _plan(mt, lam), _plan(me, lam), mean_demand_alloc_lam(mt, lam)
            F = lambda x: lam * mt.f1(x) + (1 - lam) * mt.f2(x)
            q = mt.critical_fractile_solution(lam)[2]
            synthA_rows.append({'p_case': pname, 'L': L, 'p_mean': float(np.mean(pv)), 's_mult': sm, 'xi_true': xi,
                                'q_mean': float(np.mean(q)), 'stock_GPD': xg.sum(), 'stock_EXP': xe.sum(),
                                'stock_MEAN': xm.sum(), 'budget_util_GPD': float(np.sum(mt.pi * xg) / mt.B),
                                'value_vs_EXP_%': 100 * (F(xe) / F(xg) - 1),
                                'value_vs_MEAN_%': 100 * (F(xm) / F(xg) - 1)})
synthA_df = pd.DataFrame(synthA_rows)

synthB_rows = []
for xis in SYNTH_HET_XI:
    xis = np.array(xis, dtype=float)
    for bm in SYNTH_HET_BMULT:
        over = dict(B=B_real * bm, K=[k * bm for k in K_real])
        mt = make_model(BASE_P, xis, EBAR_T * (1 - xis), **over)
        me = make_model(BASE_P, 0.0, EBAR_T, **over)
        xc = float(xis.mean()); mc = make_model(BASE_P, xc, EBAR_T * (1 - xc), **over)
        xg, xe, xcc, xm = min_shortage_alloc(mt), min_shortage_alloc(me), min_shortage_alloc(mc), mean_demand_alloc(mt)
        Xg, Xe = xg.sum(axis=1), xe.sum(axis=1)
        synthB_rows.append({'xi_true': str(tuple(float(v) for v in xis)), 'B_mult': bm,
                            **{f'X_GPD_{n}': Xg[i] for i, n in enumerate(names)},
                            **{f'X_EXP_{n}': Xe[i] for i, n in enumerate(names)},
                            'n_tail_active': int((Xg > mt.u_prime + 1e-6).sum()),
                            'value_vs_EXP_%': 100 * (mt.f2(xe) / mt.f2(xg) - 1),
                            'value_vs_pooled_xi_%': 100 * (mt.f2(xcc) / mt.f2(xg) - 1),
                            'value_vs_MEAN_%': 100 * (mt.f2(xm) / mt.f2(xg) - 1)})
synthB_df = pd.DataFrame(synthB_rows)

print("=" * 110 + f"\nSTEP 10h: WHEN DOES THE GPD LAYER MATTER?  [{_time.time()-t0:.0f}s]\n" + "=" * 110)
print(f"(A) Stock level, budget/capacity x{SYNTH_B_MULT} (not binding), lambda = {SYNTH_LAMBDA}; mean excess fixed, "
      "tail shape xi varied.\n    value_vs_EXP_% = extra total expected cost of the exponential planner (true model = GPD)")
for pname in synthA_df['p_case'].unique():
    g = synthA_df[synthA_df['p_case'] == pname]
    print(f"\n  {pname}: value vs EXP (%)  [rows xi, columns shortage-penalty multiplier]")
    print(g.pivot(index='xi_true', columns='s_mult', values='value_vs_EXP_%').round(2).to_string())
    print(f"  {pname}: total stock GPD / EXP (thousand kits)")
    print(g.pivot(index='xi_true', columns='s_mult', values='stock_GPD').round(1).to_string())
    print(g.pivot(index='xi_true', columns='s_mult', values='stock_EXP').round(1).iloc[[0]].to_string())
    print(f"  {pname}: budget utilisation of the GPD plan (1.00 = capped by the x{SYNTH_B_MULT} budget)")
    print(g.pivot(index='xi_true', columns='s_mult', values='budget_util_GPD').round(2).to_string())
    print(f"  critical fractile q by s_mult: " + ", ".join(f"x{s}: {v:.4f}" for s, v in
                                                          g.groupby('s_mult')['q_mean'].first().items()))
print("\n(B) Binding budget, district-specific tails (lambda = 0)")
with pd.option_context('display.width', 300, 'display.max_columns', 40, 'display.float_format', '{:,.2f}'.format):
    print(synthB_df.to_string(index=False))

def _ptitle(k, pname):
    "Panel title in the manuscript's notation, e.g. '(a) $L=2$ forecast', '(c) confirmed alert ($p_i=1$)'."
    t = pname.replace('confirmed alert (p=1)', 'confirmed alert ($p_i=1$)')
    t = t.replace(f'L={LEAD_DAYS} forecast', f'$L={LEAD_DAYS}$ forecast').replace('L=1 forecast', '$L=1$ forecast')
    return f'({chr(97 + k)}) {t}'


_scol = ['#2a78d6', '#eb6834', '#1baf7a', '#8a5a9e', '#b23a48']
fig, axes = plt.subplots(1, len(_p_cases), figsize=(2.6 * len(_p_cases), 2.7), squeeze=False, sharey=False)
for k, (ax, (pname, _, _)) in enumerate(zip(axes[0], _p_cases)):
    g = synthA_df[synthA_df['p_case'] == pname]
    for j, sm in enumerate(SYNTH_S_MULT):
        gg = g[g['s_mult'] == sm]
        ax.plot(gg['xi_true'], gg['value_vs_EXP_%'], marker='o', ms=4, lw=1.5, color=_scol[j % 5], label=f'$s\\times{sm}$')
    ax.axvline(BASE_XI, ls=':', c='grey', lw=1)
    ax.set_title(_ptitle(k, pname)); ax.set_xlabel(r'true tail shape $\xi$'); light_grid(ax)
axes[0][0].set_ylabel('extra expected cost of the\nexponential-tail plan (%)')
axes[0][0].legend(loc='upper left', title=r'dotted: fitted $\hat\xi$', title_fontsize=7)
plt.tight_layout(); plt.savefig(f'{APP_DIR}/saved/evt_when_it_matters.png'); plt.show()

synthA_df.to_csv(f'{APP_DIR}/saved/synthetic_tail_stock_level.csv', index=False)
synthB_df.to_csv(f'{APP_DIR}/saved/synthetic_tail_split.csv', index=False)
try:
    with open(f'{APP_DIR}/saved/{RESULTS_JSON}') as f:
        _fz = json.load(f)
    _fz['synthetic_tail_experiment'] = {'stock_level': synthA_df.to_dict('records'),
                                        'split_heterogeneous_xi': synthB_df.to_dict('records'),
                                        'settings': {'xi': SYNTH_XI, 's_mult': SYNTH_S_MULT, 'B_mult_A': SYNTH_B_MULT,
                                                     'lambda_A': SYNTH_LAMBDA, 'Ebar': EBAR_T.tolist()}}
    with open(f'{APP_DIR}/saved/{RESULTS_JSON}', 'w') as f:
        json.dump(_fz, f, indent=1, default=_j)
    print(f"Appended 'synthetic_tail_experiment' to {RESULTS_JSON}")
except Exception as e:
    print(f"Could not append to {RESULTS_JSON}:", e)


## Step 10i: Demand source -- ex-ante hazard atlas vs ex-post 2022 impact

The baseline demand uses only information available before the decision: the flood-prone share of each district in the NRSC/ISRO Flood Hazard Atlas (1998-2015). This step compares it with the frequently flooded zones only and with the documented 2022 affected population (ex-post), and scores the ex-ante plan under the 2022 demand against the plan that hindsight would have chosen.

In [ ]:
# Step 10i: demand source (Section 7.4; Table 17) and evaluation of the proactive plan against the
# documented 2022 impact (Section 8).

DEMAND_SOURCES = [
    ('atlas_all_classes (ex-ante, baseline)', 'hazard_atlas', None),
    ('atlas_moderate_plus (ex-ante)', 'hazard_atlas', ['Moderate', 'High', 'Very High']),
    ('affected_2022 (ex-post)', 'affected_population', None),
]
dsrc_rows, dsrc_plans = [], {}
for L in [l for l in [LEAD_DAYS, 1] if l in scenario_A_results_by_lead]:
    pL = scenario_A_results_by_lead[L]['p_real']
    for label, src, cls in DEMAND_SOURCES:
        frac, _ = at_risk_from_source(src, cls)
        d_s = d_for_fraction(np.array([frac[n] for n in names]))
        m_s = make_model(pL, BASE_XI, BASE_SIG, d=d_s, lead=L)
        x = min_shortage_alloc(m_s); xm = mean_demand_alloc(m_s); X = x.sum(axis=1)
        f2, f2_0 = m_s.f2(x), m_s.f2(np.zeros_like(x))
        dsrc_plans[(L, label)] = x
        dsrc_rows.append({'L': L, 'source': label,
                          **{f'a_{n}': frac[n] for n in names}, **{f'd_{n}': d_s[i] for i, n in enumerate(names)},
                          **{f'u_{n}': m_s.u_prime[i] for i, n in enumerate(names)},
                          **{f'X_{n}': X[i] for i, n in enumerate(names)},
                          'coverage_%': 100 * X.sum() / sum(d_s), 'n_tail_active': int((X > m_s.u_prime + 1e-6).sum()),
                          'f1': m_s.f1(x), 'f2': f2, 'f2_no_stock': f2_0, 'f2_cut_%': 100 * (1 - f2 / f2_0),
                          'value_vs_MEAN_%': 100 * (m_s.f2(xm) / f2 - 1),
                          'lambda*_min': float(m_s.stocking_threshold_lambda().min())})
dsrc_df = pd.DataFrame(dsrc_rows)

retro_rows = []
frac22, _ = at_risk_from_source('affected_population')
d22 = d_for_fraction(np.array([frac22[n] for n in names]))
for L in sorted({k[0] for k in dsrc_plans}):
    pL = scenario_A_results_by_lead[L]['p_real']
    m22 = make_model(pL, BASE_XI, BASE_SIG, d=d22, lead=L)
    x_ante = dsrc_plans[(L, DEMAND_SOURCES[0][0])]
    x_hind = dsrc_plans[(L, DEMAND_SOURCES[2][0])]
    f_ante, f_hind, f_none = m22.f2(x_ante), m22.f2(x_hind), m22.f2(np.zeros_like(x_ante))
    retro_rows.append({'L': L, **{f'X_exante_{n}': x_ante.sum(1)[i] for i, n in enumerate(names)},
                       **{f'X_hindsight_{n}': x_hind.sum(1)[i] for i, n in enumerate(names)},
                       'f2_2022_no_stock': f_none, 'f2_2022_exante_plan': f_ante, 'f2_2022_hindsight_plan': f_hind,
                       'cut_exante_vs_none_%': 100 * (1 - f_ante / f_none),
                       'cut_hindsight_vs_none_%': 100 * (1 - f_hind / f_none),
                       'regret_exante_vs_hindsight_%': 100 * (f_ante / f_hind - 1)})
retro_df = pd.DataFrame(retro_rows)

print("=" * 110 + "\nSTEP 10i: DEMAND SOURCE (ex-ante hazard atlas vs ex-post 2022 impact)\n" + "=" * 110)
_c = (['L', 'source'] + [f'a_{n}' for n in names] + [f'd_{n}' for n in names] + [f'X_{n}' for n in names]
      + ['coverage_%', 'n_tail_active', 'f1', 'f2', 'f2_no_stock', 'f2_cut_%', 'value_vs_MEAN_%', 'lambda*_min'])
with pd.option_context('display.width', 320, 'display.max_columns', 40, 'display.float_format', '{:,.3f}'.format):
    print(dsrc_df[_c].to_string(index=False))
    print("\nRetrospective: the EX-ANTE plan scored under the documented 2022 demand "
          "(hindsight plan = optimal if 2022 demand had been known)")
    print(retro_df.to_string(index=False))
dsrc_df.to_csv(f'{APP_DIR}/saved/demand_source_comparison.csv', index=False)
retro_df.to_csv(f'{APP_DIR}/saved/retrospective_2022.csv', index=False)
try:
    with open(f'{APP_DIR}/saved/{RESULTS_JSON}') as f:
        _fz = json.load(f)
    _fz['demand_source_comparison'] = dsrc_df.to_dict('records')
    _fz['retrospective_2022'] = retro_df.to_dict('records')
    with open(f'{APP_DIR}/saved/{RESULTS_JSON}', 'w') as f:
        json.dump(_fz, f, indent=1, default=_j)
    print(f"Appended 'demand_source_comparison' and 'retrospective_2022' to {RESULTS_JSON}")
except Exception as e:
    print(f"Could not append to {RESULTS_JSON}:", e)


## Step 10j: Collect the final outputs for upload

Copies the result files, figures and district series into `OUT_DIR/final_output`, in four subfolders, ready to be uploaded to the repository. Files larger than 25 MB (the GitHub web-upload limit) are listed but not copied.

In [ ]:
# Step 10j: copy the files needed for the repository into OUT_DIR/final_output.
import glob, shutil

FINAL_DIR = os.path.join(APP_DIR, 'final_output')
MAX_MB = 25

groups = {
    'results':       glob.glob(f'{APP_DIR}/saved/*.json') + glob.glob(f'{APP_DIR}/saved/*.csv'),
    'figures':       glob.glob(f'{APP_DIR}/*.png') + glob.glob(f'{APP_DIR}/saved/*.png'),
    'district_data': glob.glob(f'{APP_DIR}/data/*.csv') + glob.glob(f'{APP_DIR}/data/*.json'),
    'solved_plans':  glob.glob(f'{APP_DIR}/saved/decision_results_*.pkl'),
}

copied, skipped = [], []
for sub, files in groups.items():
    os.makedirs(os.path.join(FINAL_DIR, sub), exist_ok=True)
    for path in sorted(set(files)):
        size_mb = os.path.getsize(path) / 1e6
        if size_mb > MAX_MB:
            skipped.append((path, size_mb))
            continue
        shutil.copy2(path, os.path.join(FINAL_DIR, sub, os.path.basename(path)))
        copied.append((sub, os.path.basename(path), size_mb))

print(f"Copied {len(copied)} files to {FINAL_DIR}")
for sub in groups:
    names_ = [f for s, f, _ in copied if s == sub]
    print(f"  {sub:<14} {len(names_):>3} files")
if skipped:
    print("Not copied (larger than {} MB):".format(MAX_MB))
    for path, size_mb in skipped:
        print(f"  {os.path.basename(path)}  ({size_mb:.1f} MB)")

## Step 11: Build the dashboard (Home + 6 pages)

Pages: Recent Climate Data (recent extracted district data), Geospatial Map, Flash Flood Monitor, What-If Simulator, Decision Support and Scenario Comparison. The pages read the files saved in `saved/` by Steps 10-10d (`decision_results_*.pkl`, `whatif_inputs.pkl`) and import the model from `decision_model.py` (Step 9).

In [ ]:
# Step 11: write the dashboard pages (Streamlit).

import os
os.makedirs(f'{APP_DIR}/pages', exist_ok=True)

with open(f'{APP_DIR}/Home.py', 'w') as f:
    f.write('''
import streamlit as st
st.set_page_config(page_title="Flood-Risk Decision Support -- Barak Valley", layout="wide")
st.title("Flood-Risk Decision Support (digital-twin-enabled) -- Barak Valley")
st.markdown("Scope: **Cachar, Karimganj, Hailakandi** (true Barak Valley -- Dima Hasao excluded, "
            "it's a Hill district). Use the sidebar: Recent Climate Data, Geospatial Map, "
            "Flash Flood Monitor, What-If Simulator, and **Decision Support** (live model results).")
''')

with open(f'{APP_DIR}/pages/1_Recent_Climate_Data.py', 'w') as f:
    f.write("""
import streamlit as st
import pandas as pd
import os

st.title("Recent Climate Data")
st.info("This lean pipeline deliberately has no trained Rainfall/MaxTemp/MinTemp "
        "forecasting model -- those never fed the decision model, so they were "
        "dropped for speed. This page shows real recent EXTRACTED data per "
        "district, not a forecast.")

DISTRICT_NAMES = ['Cachar', 'Karimganj', 'Hailakandi']
choice = st.selectbox("District", DISTRICT_NAMES)
path = f"data/district_{choice.replace(' ','_')}.csv"
if os.path.exists(path):
    df = pd.read_csv(path, parse_dates=['Date']).tail(30)
    st.line_chart(df.set_index('Date')[['Rainfall','MaxTemp','MinTemp']])
    st.caption(f"Most recent 30 real days on record for {choice}.")
else:
    st.warning(f"Run the extraction step first to populate {path}")
""")

with open(f'{APP_DIR}/pages/2_Geospatial_Map.py', 'w') as f:
    f.write("""
import streamlit as st, folium, pickle, glob
import numpy as np, pandas as pd
from streamlit_folium import st_folium

st.title("Geospatial Decision Map -- Barak Valley")
st.caption("Districts (circles), godowns (squares) and the recommended stock movements (lines) for the selected "
           "scenario. Circle size = planned stock; colour = forecast flood probability. Locations are approximate "
           "district/town centres.")

DISTRICT_XY = {"Cachar": (24.76, 92.86), "Karimganj": (24.62, 92.40), "Hailakandi": (24.55, 92.58)}
WAREHOUSE_XY = [("WH1 Silchar (Cachar)", (24.83, 92.78)), ("WH2 Karimganj", (24.87, 92.36))]

files = sorted(glob.glob('saved/decision_results_*.pkl'))
if not files:
    st.warning("Run Step 10 first."); st.stop()
labels = {}
for fp in files:
    with open(fp, 'rb') as f:
        labels[pickle.load(f).get('scenario_label', fp)] = fp
choice = st.selectbox("Scenario", list(labels.keys()))
with open(labels[choice], 'rb') as f:
    saved = pickle.load(f)
names = saved['districts']; mp = saved['model_params']
pf = sorted([p for p in saved['pareto_frontier'] if p.get('x') is not None], key=lambda p: p['f1'])
which = st.radio("Plan shown", ["Minimum expected shortage (λ = 0)", "Choose a frontier point"], horizontal=True)
if which.startswith("Minimum"):
    pt = min(pf, key=lambda p: (round(p['f2'], 0), p['f1']))
else:
    k = st.slider("Frontier point (left = cheapest, right = least shortage)", 0, len(pf) - 1, len(pf) - 1)
    pt = pf[k]
x = np.asarray(pt['x']); X = x.sum(axis=1)
p = np.asarray(mp['p'], dtype=float); d = np.asarray(mp['d'], dtype=float)
xi = float(np.atleast_1d(mp['xi'])[0]); sig = np.asarray(mp['sigma'], dtype=float); u = float(mp['u'])
Ebar = sig / (1 - xi); kk = d / (u + Ebar); uprime = kk * u

m = folium.Map(location=[24.70, 92.62], zoom_start=9, tiles="OpenStreetMap")
pmax = max(p.max(), 1e-9)
for i, n in enumerate(names):
    lat, lon = DISTRICT_XY.get(n, (24.7, 92.6))
    shade = int(255 * (1 - min(p[i] / pmax, 1) * 0.85))
    colour = f"#ff{shade:02x}{shade:02x}"
    folium.CircleMarker([lat, lon], radius=8 + 30 * np.sqrt(X[i] / max(X.max(), 1e-9)),
                        color="darkred", weight=2, fill=True, fill_color=colour, fill_opacity=0.85,
                        popup=folium.Popup(f"<b>{n}</b><br>p = {p[i]:.3f}<br>demand d = {d[i]:.1f} k households"
                                           f"<br>certain demand u' = {uprime[i]:.1f} k<br>planned stock X = {X[i]:.1f} k kits"
                                           f"<br>coverage = {100 * X[i] / d[i]:.0f}%", max_width=260)).add_to(m)
    folium.map.Marker([lat - 0.06, lon], icon=folium.DivIcon(html=f"<div style='font-size:12px;font-weight:bold'>{n}<br>{X[i]:.1f}k / {d[i]:.1f}k</div>")).add_to(m)
for j, (wn, (wlat, wlon)) in enumerate(WAREHOUSE_XY):
    folium.RegularPolygonMarker([wlat, wlon], number_of_sides=4, radius=10, color="black", fill=True,
                                fill_color="#333333", popup=f"{wn}<br>load = {x[:, j].sum():.1f} k kits").add_to(m)
    for i, n in enumerate(names):
        if x[i, j] > 0.5:
            folium.PolyLine([[wlat, wlon], list(DISTRICT_XY.get(n, (24.7, 92.6)))], weight=1 + 6 * x[i, j] / max(x.max(), 1e-9),
                            color="#1f4e79", opacity=0.8, tooltip=f"{wn} -> {n}: {x[i, j]:.1f} k kits").add_to(m)
st_folium(m, width=950, height=600)

st.subheader("Plan shown on the map")
st.dataframe(pd.DataFrame({'District': names, 'p (flood within L days)': p.round(4), 'demand d (k)': d.round(1),
                           "certain demand u' (k)": uprime.round(1), 'from WH1 (k)': x[:, 0].round(1),
                           'from WH2 (k)': x[:, 1].round(1), 'stock X (k)': X.round(1),
                           'coverage %': (100 * X / d).round(0)}), use_container_width=True)
st.caption(f"Preparation cost f₁ = Rs {pt['f1'] / 1e7:.2f} crore; expected shortage cost f₂ = Rs {pt['f2'] / 1e7:.2f} crore.")
""")

with open(f'{APP_DIR}/pages/3_Flash_Flood_Monitor.py', 'w') as f:
    f.write("""
import streamlit as st
import pickle
import pandas as pd

st.title("Flash Flood Monitor")
try:
    import glob
    _files = sorted(glob.glob('saved/decision_results_scenario_A_lead*.pkl'))
    if not _files:
        raise FileNotFoundError
    _pick = st.selectbox("Scenario", _files)
    with open(_pick, 'rb') as f:
        saved = pickle.load(f)
    names = saved['districts']
    p_vals = saved['model_params']['p']
    df_risk = pd.DataFrame({'District': names, 'P(alert day within L days), calibrated': p_vals})
    st.bar_chart(df_risk.set_index('District'))
    st.dataframe(df_risk.style.format({'P(alert day within L days), calibrated': '{:.2%}'}))
    st.caption("Calibrated probability of a flash-flood ALERT day (two consecutive days >= 64.5 mm) "
               "within the lead window -- not a probability of an observed flood.")
except FileNotFoundError:
    st.warning("Run Step 10 (solve) first to populate saved/decision_results_scenario_A_lead*.pkl")
""")

with open(f'{APP_DIR}/pages/4_What_If_Simulator.py', 'w') as f:
    f.write("""
import streamlit as st
import pandas as pd
import numpy as np
import pickle
from decision_model import PrepositioningModel, solve_exact, solve_slsqp, apply_calibration, _HAS_CVXPY

st.title("What-If Simulator")
st.write("Scale one district's rainfall inputs and see BOTH the resulting flood-probability "
         "shift (from its trained classifier) AND the re-optimized pre-positioning allocation "
         "(a live re-run of the real solver, not the static Decision Support snapshot).")

try:
    with open('saved/whatif_inputs.pkl', 'rb') as f:
        wi = pickle.load(f)
except FileNotFoundError:
    st.warning("Run Step 9 and Step 10 first to populate saved/whatif_inputs.pkl and decision_model.py")
    st.stop()

names = wi['districts']
feat_flood = wi['feat_flood']

choice = st.selectbox("District to scale rainfall for", names)
scale_pct = st.slider("Rainfall scaling (%)", 50, 200, 100, step=10)
lam = st.select_slider("Trade-off weight λ",
                        options=[round(x, 2) for x in np.linspace(0, 1, 21)], value=0.5)
st.write(f"Scenario: **{choice}**'s rainfall at **{scale_pct}%** of its recorded baseline "
         f"(baseline date: {wi['baseline_date']}), evaluated at λ = {lam}.")

# ---- 1. Flood probability under scaled rainfall (real classifier, real features) ----
baseline_row = wi['baseline_rows'][choice]
rain_cols = [c for c in feat_flood if c.startswith('Rainfall')]
scaled_row = dict(baseline_row)
for c in rain_cols:
    scaled_row[c] = baseline_row[c] * (scale_pct / 100.0)
clf = wi['classifiers'][choice]
_cal, _meth = wi.get('calibrator'), wi.get('calibration_method', 'platt')
p_base = float(apply_calibration(_cal, _meth, clf.predict_proba(pd.DataFrame([baseline_row])[feat_flood])[:, 1])[0])
p_scaled = float(apply_calibration(_cal, _meth, clf.predict_proba(pd.DataFrame([scaled_row])[feat_flood])[:, 1])[0])

col1, col2 = st.columns(2)
col1.metric(f"{choice} baseline flood probability", f"{p_base:.1%}")
col2.metric(f"At {scale_pct}% rainfall", f"{p_scaled:.1%}", delta=f"{(p_scaled - p_base):+.1%}")
st.caption(f"Rainfall-derived features scaled: {', '.join(rain_cols)}. Other districts' "
           "probabilities, and DayOfYear/Month/Year/temperature features for this one, are "
           "held at their recorded baseline.")

# ---- 2. Re-run the REAL pre-positioning optimizer with this district's p_i updated ----
p_scenario = list(wi['p_baseline'])
p_scenario[names.index(choice)] = float(p_scaled)
model = PrepositioningModel(p=p_scenario, d=wi['d'], xi=wi['xi'], sigma=wi['sigma'], u=wi['u'],
                             h=wi['h'], c=wi['c'], s=wi['s'], o=wi['o'], K=wi['K'], B=wi['B'], pi=wi['pi'],
                             min_service_frac=wi.get('min_service_frac', 0.0), disallowed=wi['disallowed'])

with st.spinner("Re-optimizing allocation for this scenario..."):
    if _HAS_CVXPY:
        x = solve_exact(model, lam=float(lam))['x']
    else:
        x = solve_slsqp(model, float(lam))['x']

st.subheader(f"Re-optimized allocation at λ = {lam}")
alloc_df = pd.DataFrame({
    'District': names,
    'pᵢ (this scenario)': [round(v, 4) for v in p_scenario],
    'Warehouse 1': x[:, 0].round(1),
    'Warehouse 2': x[:, 1].round(1),
    'Total': x.sum(axis=1).round(1),
})
st.dataframe(alloc_df, use_container_width=True)
st.caption("Solved exactly with the convex (conic) solver; SLSQP is used only if cvxpy is unavailable. "
           "When the budget binds and warehouse capacity does not, splits of a district's stock between "
           "warehouses that differ only in transport cost are practically equivalent.")
""")

with open(f'{APP_DIR}/pages/5_Decision_Support.py', 'w') as f:
    f.write("""
import streamlit as st
import pandas as pd
import numpy as np
import pickle
import glob
import plotly.graph_objects as go

st.title("Pre-Positioning Decision Support -- Barak Valley")
st.caption("Pick a scenario, then a point on its Pareto frontier. Left end = cheapest plan (no stock), "
           "right end = least expected shortage (budget fully used).")

scenario_files = sorted(glob.glob('saved/decision_results_*.pkl'))
if not scenario_files:
    st.warning("Run Step 10 (solve) first to populate saved/decision_results_*.pkl")
    st.stop()
scenario_labels = {}
for fpath in scenario_files:
    with open(fpath, 'rb') as sf:
        scenario_labels[pickle.load(sf).get('scenario_label', fpath)] = fpath
chosen_label = st.selectbox("Scenario", list(scenario_labels.keys()))
with open(scenario_labels[chosen_label], 'rb') as f:
    saved = pickle.load(f)

names = saved['districts']
pf = sorted([pt for pt in saved['pareto_frontier'] if pt.get('x') is not None], key=lambda pt: pt['f1'])
f1c = [pt['f1'] / 1e7 for pt in pf]; f2c = [pt['f2'] / 1e7 for pt in pf]

k = st.slider("Frontier point (0 = cheapest ... right = least shortage)", 0, len(pf) - 1, len(pf) - 1)
chosen = pf[k]

st.subheader("Pareto frontier: preparation cost vs expected shortage cost")
fig = go.Figure()
fig.add_trace(go.Scatter(x=f1c, y=f2c, mode='markers+lines', marker=dict(size=7, color='#1f4e79'),
                         text=[f"point {i}, implied λ = {pt['lambda']:.3f}" for i, pt in enumerate(pf)],
                         hovertemplate="f₁ = Rs %{x:.3f} cr<br>f₂ = Rs %{y:.3f} cr<br>%{text}", name='frontier'))
fig.add_trace(go.Scatter(x=[f1c[k]], y=[f2c[k]], mode='markers', marker=dict(size=16, color='crimson', symbol='star'),
                         name='selected'))
fig.update_layout(xaxis_title="f₁: expected preparation cost (Rs crore)",
                  yaxis_title="f₂: expected shortage cost (Rs crore)", height=480)
st.plotly_chart(fig, use_container_width=True)

st.subheader("Recommended allocation (thousand kits)")
x = np.asarray(chosen['x'])
st.dataframe(pd.DataFrame({'District': names, 'Warehouse 1': x[:, 0].round(1), 'Warehouse 2': x[:, 1].round(1),
                           'Total': x.sum(axis=1).round(1)}), use_container_width=True)
c1, c2, c3 = st.columns(3)
c1.metric("Preparation cost f₁", f"Rs {chosen['f1'] / 1e7:.2f} cr")
c2.metric("Expected shortage cost f₂", f"Rs {chosen['f2'] / 1e7:.2f} cr")
c3.metric("Implied λ", f"{chosen['lambda']:.3f}")
""")

with open(f'{APP_DIR}/pages/6_Scenario_Comparison.py', 'w') as f:
    f.write("""
import streamlit as st
import pandas as pd
import numpy as np
import pickle
import glob
import plotly.graph_objects as go

st.title("Scenario Comparison -- Barak Valley")
st.caption("Every saved scenario at once: each LEAD_DAYS run of Scenario A (proactive) plus "
           "Scenario B (reactive). Same generic model, same code path -- only the SOURCE of "
           "p_i differs between them (see Step 10's own comment block for the full explanation).")

scenario_files = sorted(glob.glob('saved/decision_results_*.pkl'))
if not scenario_files:
    st.warning("Run Step 10 (and Step 10b, for more than one LEAD_DAYS) first to populate "
               "saved/decision_results_*.pkl")
    st.stop()

scenarios = {}   # label -> saved dict
for fpath in scenario_files:
    with open(fpath, 'rb') as sf:
        _saved = pickle.load(sf)
    scenarios[_saved.get('scenario_label', fpath)] = _saved

if len(scenarios) < 2:
    st.info("Only one scenario saved so far -- this page is most useful once Step 10b has run "
             "more than one LEAD_DAYS, or Scenario B has also been solved. Showing what's "
             "available below regardless.")

names = next(iter(scenarios.values()))['districts']
COLORS = ['#2E5C8A', '#C97A2B', '#3A7D63', '#8A5A9E', '#B23A48', '#5C7A99']

# ---- Overlaid Pareto frontiers of every saved scenario ----
st.subheader("Pareto frontiers, overlaid")
fig = go.Figure()
for i, (label, saved) in enumerate(scenarios.items()):
    pf = saved['pareto_frontier']
    f1_vals = [pt['f1'] / 1e7 for pt in pf]      # Rs crore, as in the paper
    f2_vals = [pt['f2'] / 1e7 for pt in pf]
    color = COLORS[i % len(COLORS)]
    if len(pf) > 1:
        fig.add_trace(go.Scatter(x=f1_vals, y=f2_vals, mode='markers+lines', name=label,
                                  line=dict(color=color),
                                  marker=dict(size=7, color=color),
                                  hovertemplate="f₁ = Rs %{x:.3f} cr<br>f₂ = Rs %{y:.3f} cr<extra>" + label + "</extra>"))
    else:
        fig.add_trace(go.Scatter(x=f1_vals, y=f2_vals, mode='markers', name=label,
                                  marker=dict(size=16, symbol='star', color=color),
                                  hovertemplate="f₁ = Rs %{x:.3f} cr<br>f₂ = Rs %{y:.3f} cr<extra>" + label + "</extra>"))
fig.update_layout(xaxis_title="f₁: expected preparation cost (Rs crore)",
                   yaxis_title="f₂: expected shortage cost (Rs crore)",
                   height=500, legend=dict(orientation="h", yanchor="bottom", y=1.02))
st.plotly_chart(fig, use_container_width=True)
st.caption("Each line is the exact Pareto frontier of one scenario (Scenario A for each lead time; "
           "Scenario B = reactive plan once the alert is confirmed, p = 1).")

# ---- Comparison point: nearest-to-selected-lambda for every scenario ----
st.subheader("Allocation at a comparable point")
share = st.slider("Share of the budget spent (%) -- the same share is applied to every scenario", 0, 100, 100, step=5)

rows = []
alloc_by_scenario = {}
for label, saved in scenarios.items():
    pf = [pt for pt in saved['pareto_frontier'] if pt.get('x') is not None]
    kits = [float(np.asarray(pt['x']).sum()) for pt in pf]
    target = share / 100.0 * max(kits)
    chosen = min(pf, key=lambda pt: (abs(float(np.asarray(pt['x']).sum()) - target), pt['f2']))
    x = np.array(chosen['x'])
    alloc_by_scenario[label] = x
    rows.append({'Scenario': label, 'implied λ': round(chosen['lambda'], 3),
                 'f₁ (Rs crore)': round(chosen['f1'] / 1e7, 2), 'f₂ (Rs crore)': round(chosen['f2'] / 1e7, 2),
                 'Total kits (all districts, x1000)': round(float(x.sum()), 1)})
st.dataframe(pd.DataFrame(rows), use_container_width=True, hide_index=True)

# ---- Grouped bar: total kits allocated per district, one bar-group per scenario ----
st.subheader("Allocation per district, by scenario")
bar_fig = go.Figure()
for i, (label, x) in enumerate(alloc_by_scenario.items()):
    per_district_total = x.sum(axis=1)
    bar_fig.add_trace(go.Bar(name=label, x=names, y=per_district_total, marker_color=COLORS[i % len(COLORS)]))
bar_fig.update_layout(barmode='group', yaxis_title="Total kits allocated (x1000)", height=450,
                       legend=dict(orientation="h", yanchor="bottom", y=1.02))
st.plotly_chart(bar_fig, use_container_width=True)
st.caption("Each scenario's own pᵢ (occurrence probability or confirmed same-day flag) is what "
           "moves these bars -- same model, same cost parameters, different real-world question "
           "(see the Pareto frontier caption above).")
""")

print("Dashboard pages written: Home, Recent Climate Data, Geospatial Map, "
      "Flash Flood Monitor, What-If Simulator, Decision Support, Scenario Comparison.")


## Step 12: Launch the dashboard

The ngrok token is no longer stored in the notebook. Add it as a Colab secret named `NGROK_AUTH_TOKEN` (key icon in the left sidebar), or type it when prompted.

In [ ]:
# Step 12: launch the dashboard. Store the ngrok token as a Colab secret named NGROK_AUTH_TOKEN, or type it
# when prompted. Never write the token in the notebook.

import os, getpass
from pyngrok import ngrok
NGROK_TOKEN = os.environ.get('NGROK_AUTH_TOKEN')
if not NGROK_TOKEN:
    try:
        from google.colab import userdata
        NGROK_TOKEN = userdata.get('NGROK_AUTH_TOKEN')
    except Exception:
        NGROK_TOKEN = None
if not NGROK_TOKEN:
    NGROK_TOKEN = getpass.getpass('ngrok auth token (input hidden): ')
ngrok.set_auth_token(NGROK_TOKEN)

import subprocess, time
os.chdir(APP_DIR)
proc = subprocess.Popen(['streamlit', 'run', 'Home.py', '--server.port=8501'])
time.sleep(5)
public_url = ngrok.connect(8501)
print(f"Dashboard live at: {public_url}")


## Step 13: Stop the dashboard (run when you're done)

In [ ]:
# ngrok.kill()
# proc.terminate()
# print("Server stopped.")
